<a href="https://colab.research.google.com/github/eduardobatistadefreitas-art/nodo-regulator/blob/main/S24C%C3%B3pia_de_Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# IA-1 — S23-D.1-B
# BLOCO 1 — PREPARAÇÃO DO AMBIENTE
# ============================================================

import numpy as np
import scipy.linalg as la

from scipy.stats import pearsonr
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

print("\n==========================================")
print(" IA-1 — S23-D.1-B")
print(" BLOCO 1 — PREPARAÇÃO")
print("==========================================\n")

# ------------------------------------------------------------
# Verificação das variáveis obrigatórias
# ------------------------------------------------------------

variaveis_obrigatorias = [
    "K_rel_star",
    "eta_hist",
    "v_hist",
    "a_hist",
    "P_R_star"
]

faltando = []

for nome in variaveis_obrigatorias:
    if nome not in globals():
        faltando.append(nome)

if len(faltando) > 0:

    print("ERRO")
    print("As seguintes variáveis não existem:\n")

    for x in faltando:
        print(" -", x)

    raise RuntimeError(
        "\nExecute primeiro o protocolo S23-C antes da auditoria."
    )

print("Todas as variáveis principais foram encontradas.\n")

# ------------------------------------------------------------
# Conversão para arrays
# ------------------------------------------------------------

eta_hist = np.asarray(eta_hist)
v_hist   = np.asarray(v_hist)
a_hist   = np.asarray(a_hist)

K_rel_star = np.asarray(K_rel_star)
P_R_star   = np.asarray(P_R_star)

# ------------------------------------------------------------
# Dimensões
# ------------------------------------------------------------

n_passos = eta_hist.shape[0]
dim = eta_hist.shape[1]

print("Número de passos :", n_passos)
print("Dimensão do espaço:", dim)
print()

# ------------------------------------------------------------
# Consistência dimensional
# ------------------------------------------------------------

assert eta_hist.shape == v_hist.shape
assert eta_hist.shape == a_hist.shape

assert K_rel_star.shape == (dim, dim)
assert P_R_star.shape == (dim, dim)

print("Consistência dimensional: OK")

# ------------------------------------------------------------
# Funções auxiliares
# ------------------------------------------------------------

def energia_cinetica(v):
    return 0.5 * np.dot(v, v)

def energia_quadratica(x):
    return 0.5 * x @ K_rel_star @ x

def energia_potencial(x, alpha=0.01):
    r2 = np.dot(x, x)
    return alpha * (r2**2)

def forca_mare(x):
    return K_rel_star @ x

# ------------------------------------------------------------
# Diagnóstico rápido do operador
# ------------------------------------------------------------

assimetria = la.norm(K_rel_star - K_rel_star.T)

autovalores = la.eigvals(K_rel_star)

print()
print("Diagnóstico rápido")
print("------------------")
print("Assimetria K :", assimetria)
print("Maior Im(lambda):", np.max(np.abs(np.imag(autovalores))))
print()

print("BLOCO 1 CONCLUÍDO COM SUCESSO")
print("Pronto para o BLOCO 2.")


 IA-1 — S23-D.1-B
 BLOCO 1 — PREPARAÇÃO

ERRO
As seguintes variáveis não existem:

 - K_rel_star
 - eta_hist
 - v_hist
 - a_hist
 - P_R_star


RuntimeError: 
Execute primeiro o protocolo S23-C antes da auditoria.

In [ ]:
# ============================================================
# IA-1 — S23-D.1-B
# BLOCO 1A — RECONSTRUÇÃO DO AMBIENTE S23-C
# ============================================================

import numpy as np
import scipy.linalg as la

print("\n==========================================")
print(" IA-1 — S23-D.1-B")
print(" BLOCO 1A — RECONSTRUÇÃO S23-C")
print("==========================================\n")


# ------------------------------------------------------------
# Parâmetro do suporte
# ------------------------------------------------------------

n = 6
np.random.seed(42)

print("Número de nós:", n)


# ------------------------------------------------------------
# 1. Construção do grafo base R*
# ------------------------------------------------------------

A_R = np.zeros((n, n))

for i in range(n):
    j = (i + 1) % n

    A_R[i, j] = 1.0
    A_R[j, i] = 1.0


print("Simetria A_R:",
      la.norm(A_R - A_R.T))


# ------------------------------------------------------------
# 2. Campo efetivo kappa
# ------------------------------------------------------------

kappa_eff = A_R * np.random.uniform(
    0.8,
    1.2,
    size=(n, n)
)


# ------------------------------------------------------------
# 3. Forma bilinear Gamma
# ------------------------------------------------------------

Gamma = (
    kappa_eff
    +
    np.dot(kappa_eff, kappa_eff) / n
)


# ------------------------------------------------------------
# 4. Curvatura espectral Gamma2
# ------------------------------------------------------------

D = np.diag(np.sum(Gamma, axis=1))

L = D - Gamma

Gamma_2 = np.dot(L, L)


# ------------------------------------------------------------
# 5. Construção H_rel
# ------------------------------------------------------------

H_rel_tensor = np.zeros((n,n,n,n))

for i in range(n):
    for j in range(n):
        for k in range(n):
            for l in range(n):

                H_rel_tensor[i,j,k,l] = (
                    1.0 /
                    (
                        1
                        +
                        (i-k)**2
                        +
                        (j-l)**2
                    )
                )


# ------------------------------------------------------------
# Função vec para operadores de quarta ordem
# ------------------------------------------------------------

def vetorizar_operador_quarta_ordem(tensor,n):

    matriz = np.zeros((n*n,n*n))

    for i in range(n):
        for j in range(n):

            col = i + j*n

            for k in range(n):
                for l in range(n):

                    row = k + l*n

                    matriz[row,col] = tensor[k,l,i,j]

    return matriz


H_rel_mat = vetorizar_operador_quarta_ordem(
    H_rel_tensor,n
)


# ------------------------------------------------------------
# 6. Construção da restrição C_phi
# ------------------------------------------------------------

C_phi_tensor = np.zeros((n,n,n,n))

for i in range(n):
    C_phi_tensor[i,:,i,:] = 1.0


C_phi_mat = vetorizar_operador_quarta_ordem(
    C_phi_tensor,n
)


# ------------------------------------------------------------
# 7. Hessiano elástico H_U
# ------------------------------------------------------------

H_U_tensor = np.zeros((n,n,n,n))


for i in range(n):
    for j in range(n):

        H_U_tensor[i,j,i,j] = Gamma[i,j]**2


H_U_mat = vetorizar_operador_quarta_ordem(
    H_U_tensor,n
)


# ------------------------------------------------------------
# 8. Construção do projetor P_R*
# ------------------------------------------------------------

U,S,Vt = la.svd(H_U_mat)

tol = 1e-5

modos_degenerados = Vt[S < tol]


if modos_degenerados.shape[0] > 0:

    M = np.vstack(
        [
            C_phi_mat,
            modos_degenerados
        ]
    )

else:

    M = C_phi_mat



M_pinv = la.pinv(M)

I = np.eye(n*n)


P_R_star = I - M_pinv @ M


# ------------------------------------------------------------
# 9. Gradiente discreto Gamma2
# ------------------------------------------------------------

grad_tensor = np.zeros((n,n,n,n))


for i in range(n):
    for j in range(n):
        for k in range(n):
            for l in range(n):

                if i == k or j == l:

                    grad_tensor[i,j,k,l] = (
                        Gamma_2[i,j]
                        -
                        Gamma_2[k,l]
                    )


grad_Gamma2_mat = vetorizar_operador_quarta_ordem(
    grad_tensor,n
)


# ------------------------------------------------------------
# 10. Operador final K_rel*
# ------------------------------------------------------------

operador_central = (
    grad_Gamma2_mat
    @
    H_rel_mat
)


K_rel_star = (
    P_R_star
    @
    operador_central
    @
    P_R_star
)


# ------------------------------------------------------------
# Auditoria rápida
# ------------------------------------------------------------

print()
print("Dimensão K_rel*:",
      K_rel_star.shape)

print(
    "Erro P²-P:",
    la.norm(P_R_star@P_R_star-P_R_star)
)

print(
    "Erro P-Pᵀ:",
    la.norm(P_R_star-P_R_star.T)
)

print(
    "Assimetria K:",
    la.norm(K_rel_star-K_rel_star.T)
)

print()
print("BLOCO 1A CONCLUÍDO")
print("Pronto para o BLOCO 1B.")


 IA-1 — S23-D.1-B
 BLOCO 1A — RECONSTRUÇÃO S23-C

Número de nós: 6
Simetria A_R: 0.0

Dimensão K_rel*: (36, 36)
Erro P²-P: 3.027835760602206e-15
Erro P-Pᵀ: 3.119140432659504e-15
Assimetria K: 137.669732175435

BLOCO 1A CONCLUÍDO
Pronto para o BLOCO 1B.


In [ ]:
# ============================================================
# IA-1 — S23-D.1-B
# BLOCO 1B — INTEGRAÇÃO TEMPORAL DO ATRATOR
# ============================================================

print("\n==========================================")
print(" IA-1 — S23-D.1-B")
print(" BLOCO 1B — DINÂMICA TEMPORAL")
print("==========================================\n")


# ------------------------------------------------------------
# Parâmetros temporais
# ------------------------------------------------------------

passos = 3000
dt = 0.001

amortecimento = 0.15


# ------------------------------------------------------------
# Estado inicial
# ------------------------------------------------------------

dim = n*n

np.random.seed(123)

eta = np.random.normal(
    0,
    0.05,
    size=dim
)

v = np.zeros(dim)


# ------------------------------------------------------------
# Históricos
# ------------------------------------------------------------

eta_hist = []
v_hist = []
a_hist = []


# ------------------------------------------------------------
# Integração dinâmica
#
# eta'' = -K eta - gamma eta'
#
# ------------------------------------------------------------

for k in range(passos):

    forca = K_rel_star @ eta

    a = (
        -forca
        -
        amortecimento*v
    )


    # Euler semi-implícito

    v = v + dt*a

    eta = eta + dt*v


    eta_hist.append(
        eta.copy()
    )

    v_hist.append(
        v.copy()
    )

    a_hist.append(
        a.copy()
    )


# Converter para arrays

eta_hist = np.array(eta_hist)
v_hist = np.array(v_hist)
a_hist = np.array(a_hist)



# ------------------------------------------------------------
# Auditoria da trajetória
# ------------------------------------------------------------

normas_eta = np.linalg.norm(
    eta_hist,
    axis=1
)

normas_v = np.linalg.norm(
    v_hist,
    axis=1
)


print("Passos temporais:",
      passos)

print(
    "Norma eta inicial:",
    normas_eta[0]
)

print(
    "Norma eta final:",
    normas_eta[-1]
)

print(
    "Norma eta máxima:",
    np.max(normas_eta)
)

print(
    "Velocidade máxima:",
    np.max(normas_v)
)


# ------------------------------------------------------------
# Verificação de explosão numérica
# ------------------------------------------------------------

if np.any(np.isnan(eta_hist)):

    print("\nALERTA: NaN detectado")

elif np.max(normas_eta) > 1e8:

    print("\nALERTA: divergência extrema")

else:

    print("\nTrajetória gerada sem overflow")


print()
print("Variáveis criadas:")
print(" - eta_hist")
print(" - v_hist")
print(" - a_hist")

print()
print("BLOCO 1B CONCLUÍDO")
print("Pronto para o BLOCO 2 — Auditoria energética.")


 IA-1 — S23-D.1-B
 BLOCO 1B — DINÂMICA TEMPORAL

Passos temporais: 3000
Norma eta inicial: 0.3652044303761381
Norma eta final: 1550.6290865573344
Norma eta máxima: 1550.6290865573344
Velocidade máxima: 10751.574223019536

Trajetória gerada sem overflow

Variáveis criadas:
 - eta_hist
 - v_hist
 - a_hist

BLOCO 1B CONCLUÍDO
Pronto para o BLOCO 2 — Auditoria energética.


In [ ]:
# ============================================================
# IA-1 — S23-D.1-B
# BLOCO 2 — AUDITORIA ENERGÉTICA E EXTRAÇÃO DOS CANDIDATOS C1-C8
# ============================================================

print("\n==========================================")
print(" IA-1 — S23-D.1-B")
print(" BLOCO 2 — AUDITORIA ENERGÉTICA")
print("==========================================\n")


import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


# ------------------------------------------------------------
# Garantia de existência dos dados
# ------------------------------------------------------------

variaveis = [
    "K_rel_star",
    "eta_hist",
    "v_hist",
    "a_hist"
]

faltantes = [
    x for x in variaveis
    if x not in globals()
]

if faltantes:
    raise RuntimeError(
        "Variáveis ausentes: " + str(faltantes)
    )


# ------------------------------------------------------------
# Cálculo das energias
# ------------------------------------------------------------

num_passos = len(eta_hist)


Ecin = np.zeros(num_passos)
Equad = np.zeros(num_passos)
Epot = np.zeros(num_passos)
L = np.zeros(num_passos)


for k in range(num_passos):

    eta = eta_hist[k]
    v = v_hist[k]

    Ecin[k] = 0.5 * np.dot(v, v)

    Equad[k] = 0.5 * eta @ K_rel_star @ eta

    # Potencial efetivo mínimo:
    # energia quadrática positiva auxiliar
    Epot[k] = 0.5 * np.dot(eta, eta)

    L[k] = (
        Ecin[k]
        +
        Equad[k]
        +
        Epot[k]
    )


# ------------------------------------------------------------
# Diferencial de Lyapunov
# ------------------------------------------------------------

delta_L = np.diff(L)


# ------------------------------------------------------------
# Construção dos candidatos C1-C8
# ------------------------------------------------------------

C = []


for k in range(num_passos):

    eta = eta_hist[k]
    v = v_hist[k]
    a = a_hist[k]

    F = K_rel_star @ eta


    C1 = np.dot(eta, v)

    C2 = np.dot(v, F)

    C3 = np.dot(eta, K_rel_star @ v)

    C4 = np.dot(eta, eta)

    C5 = np.dot(v, v)

    C6 = np.dot(
        eta,
        K_rel_star @ eta
    )

    C7 = np.dot(a, v)

    C8 = np.dot(F, F)


    C.append(
        [
            C1,
            C2,
            C3,
            C4,
            C5,
            C6,
            C7,
            C8
        ]
    )


C = np.array(C)


# Ajuste de dimensão
C = C[:-1]


nomes = [
    "C1 eta.v",
    "C2 v.Keta",
    "C3 eta.Kv",
    "C4 ||eta||²",
    "C5 ||v||²",
    "C6 eta.Keta",
    "C7 a.v",
    "C8 ||Keta||²"
]


# ------------------------------------------------------------
# Estatística
# ------------------------------------------------------------

resultados = []


for i in range(8):

    x = C[:, i]

    pearson = stats.pearsonr(
        x,
        delta_L
    )[0]

    spearman = stats.spearmanr(
        x,
        delta_L
    )[0]


    modelo = LinearRegression()

    modelo.fit(
        x.reshape(-1,1),
        delta_L
    )

    pred = modelo.predict(
        x.reshape(-1,1)
    )

    R2 = r2_score(
        delta_L,
        pred
    )

    mse = mean_squared_error(
        delta_L,
        pred
    )


    resultados.append(
        [
            nomes[i],
            pearson,
            spearman,
            R2,
            mse
        ]
    )


# ------------------------------------------------------------
# Ranking
# ------------------------------------------------------------

resultados = sorted(
    resultados,
    key=lambda x: abs(x[3]),
    reverse=True
)


print("Ranking dos mecanismos:")
print()

for r in resultados:

    print(
        f"{r[0]:15s} | "
        f"Pearson={r[1]: .5f} | "
        f"Spearman={r[2]: .5f} | "
        f"R²={r[3]: .5f}"
    )


# ------------------------------------------------------------
# Diagnóstico final
# ------------------------------------------------------------

melhor = resultados[0]

print("\n------------------------------------------")

print(
    "Maior explicação individual:"
)
print(
    melhor[0]
)

print(
    "R² dominante:",
    melhor[3]
)


if melhor[3] > 0.80:

    print(
        "\nCONCLUSÃO:"
    )

    print(
        "Existe candidato natural para W(η,η̇)."
    )

else:

    print(
        "\nCONCLUSÃO:"
    )

    print(
        "Nenhum mecanismo isolado explica a transferência."
    )

    print(
        "Necessária análise multivariada."
    )


print("\nBLOCO 2 CONCLUÍDO")
print("Pronto para o BLOCO 3 — regressão cruzada.")


 IA-1 — S23-D.1-B
 BLOCO 2 — AUDITORIA ENERGÉTICA

Ranking dos mecanismos:

C8 ||Keta||²    | Pearson= 0.99730 | Spearman= 1.00000 | R²= 0.99461
C7 a.v          | Pearson= 0.99044 | Spearman= 0.99816 | R²= 0.98097
C2 v.Keta       | Pearson=-0.99030 | Spearman=-0.99855 | R²= 0.98070
C5 ||v||²       | Pearson= 0.98199 | Spearman= 1.00000 | R²= 0.96431
C4 ||eta||²     | Pearson= 0.98110 | Spearman= 0.99972 | R²= 0.96256
C1 eta.v        | Pearson= 0.94490 | Spearman= 0.99810 | R²= 0.89284
C6 eta.Keta     | Pearson=-0.83359 | Spearman=-0.01743 | R²= 0.69487
C3 eta.Kv       | Pearson= 0.22461 | Spearman= 0.86441 | R²= 0.05045

------------------------------------------
Maior explicação individual:
C8 ||Keta||²
R² dominante: 0.994605598710977

CONCLUSÃO:
Existe candidato natural para W(η,η̇).

BLOCO 2 CONCLUÍDO
Pronto para o BLOCO 3 — regressão cruzada.


In [ ]:
# ==========================================================
# IA-1 — S23-D.1-B
# BLOCO 3 — REGRESSÃO CRUZADA DO TERMO W(η,η̇)
# ==========================================================

import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error


print("="*50)
print(" IA-1 — S23-D.1-B")
print(" BLOCO 3 — REGRESSÃO MULTIVARIADA")
print("="*50)


# ----------------------------------------------------------
# 1. Construção do alvo ΔL
# ----------------------------------------------------------

T = len(eta_hist)

L_hist = []

for k in range(T):

    eta = eta_hist[k]
    v = v_hist[k]

    Ecin = 0.5 * np.dot(v, v)

    Equad = 0.5 * np.dot(
        eta,
        K_rel_star @ eta
    )

    # potencial mínimo regularizado
    Epot = 0.25 * np.dot(eta, eta)**2

    L = Ecin + Equad + Epot

    L_hist.append(L)


L_hist = np.array(L_hist)

delta_L = np.diff(L_hist)


# ----------------------------------------------------------
# 2. Construção dos candidatos C1-C8
# ----------------------------------------------------------

C = []

for k in range(T-1):

    eta = eta_hist[k]
    v = v_hist[k]
    a = a_hist[k]

    Keta = K_rel_star @ eta
    Kv = K_rel_star @ v


    C1 = np.dot(eta, v)

    C2 = np.dot(v, Keta)

    C3 = np.dot(eta, Kv)

    C4 = np.dot(eta, eta)

    C5 = np.dot(v, v)

    C6 = np.dot(eta, Keta)

    C7 = np.dot(a, v)

    C8 = np.dot(Keta, Keta)


    C.append([
        C1,
        C2,
        C3,
        C4,
        C5,
        C6,
        C7,
        C8
    ])


C = np.array(C)


nomes = [
    "C1 eta.v",
    "C2 v.Keta",
    "C3 eta.Kv",
    "C4 ||eta||²",
    "C5 ||v||²",
    "C6 eta.Keta",
    "C7 a.v",
    "C8 ||Keta||²"
]


# ----------------------------------------------------------
# 3. Regressão Ridge
# ----------------------------------------------------------

modelo_ridge = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=1e-6))
])

modelo_ridge.fit(C, delta_L)

pred_ridge = modelo_ridge.predict(C)

r2_ridge = r2_score(delta_L, pred_ridge)

mse_ridge = mean_squared_error(
    delta_L,
    pred_ridge
)


coef = modelo_ridge.named_steps["ridge"].coef_


# ----------------------------------------------------------
# 4. Regressão LASSO
# ----------------------------------------------------------

modelo_lasso = Pipeline([
    ("scale", StandardScaler()),
    ("lasso", Lasso(alpha=1e-5,max_iter=10000))
])


modelo_lasso.fit(C, delta_L)

pred_lasso = modelo_lasso.predict(C)

r2_lasso = r2_score(
    delta_L,
    pred_lasso
)


coef_lasso = modelo_lasso.named_steps["lasso"].coef_


# ----------------------------------------------------------
# 5. Relatório
# ----------------------------------------------------------

print("\nRidge:")
print("R² =", r2_ridge)
print("MSE =", mse_ridge)


print("\nCoeficientes Ridge:")

for nome, valor in zip(nomes,coef):
    print(
        f"{nome:15s} : {valor:.6e}"
    )


print("\nLASSO:")
print("R² =", r2_lasso)

print("\nCoeficientes LASSO:")

for nome, valor in zip(nomes,coef_lasso):
    print(
        f"{nome:15s} : {valor:.6e}"
    )


# ----------------------------------------------------------
# 6. Termo dominante reconstruído
# ----------------------------------------------------------

importância = np.abs(coef)

ordem = np.argsort(importância)[::-1]


print("\nRanking de contribuição Ridge:")

for i in ordem:

    print(
        nomes[i],
        " | peso absoluto = ",
        importância[i]
    )


print("\n===================================")

if r2_ridge > 0.90:

    print(
    "CONCLUSÃO:"
    )
    print(
    "Existe reconstrução multivariada estável para W(η,η̇)."
    )

else:

    print(
    "CONCLUSÃO:"
    )
    print(
    "A combinação atual ainda não explica suficientemente ΔL."
    )


print("===================================")
print("BLOCO 3 CONCLUÍDO")
print("Pronto para S23-D.1-C — reconstrução do funcional.")

 IA-1 — S23-D.1-B
 BLOCO 3 — REGRESSÃO MULTIVARIADA

Ridge:
R² = 0.9971878141400942
MSE = 2.2186213782591308e+16

Coeficientes Ridge:
C1 eta.v        : -3.395822e+12
C2 v.Keta       : -5.683571e+11
C3 eta.Kv       : 5.812354e+11
C4 ||eta||²     : 1.356623e+12
C5 ||v||²       : -3.079440e+12
C6 eta.Keta     : -3.422195e+12
C7 a.v          : 6.999610e+11
C8 ||Keta||²    : 6.664990e+11

LASSO:
R² = 0.9669757190112789

Coeficientes LASSO:
C1 eta.v        : 3.239847e+09
C2 v.Keta       : 1.418538e+09
C3 eta.Kv       : -1.138362e+08
C4 ||eta||²     : 1.769719e+09
C5 ||v||²       : -1.918944e+09
C6 eta.Keta     : -1.256132e+09
C7 a.v          : -9.189298e+08
C8 ||Keta||²    : 7.204667e+08

Ranking de contribuição Ridge:
C6 eta.Keta  | peso absoluto =  3422194561889.7935
C1 eta.v  | peso absoluto =  3395822293111.3433
C5 ||v||²  | peso absoluto =  3079440082792.854
C4 ||eta||²  | peso absoluto =  1356622662168.464
C7 a.v  | peso absoluto =  699960985486.2196
C8 ||Keta||²  | peso absoluto =  66

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.907e+20, tolerance: 2.366e+18
  model = cd_fast.enet_coordinate_descent(


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

print("="*55)
print(" IA-1 — S23-D.1-C")
print(" NORMALIZAÇÃO E EXTRAÇÃO DE W*")
print("="*55)

# ----------------------------------------------------
# 1. Construção novamente dos observáveis C1-C8
# ----------------------------------------------------

N = len(eta_hist)

C = []

for k in range(N-1):

    eta = eta_hist[k]
    v = v_hist[k]
    a = a_hist[k]

    Keta = K_rel_star @ eta
    Kv = K_rel_star @ v

    C1 = np.dot(eta, v)
    C2 = np.dot(v, Keta)
    C3 = np.dot(eta, Kv)
    C4 = np.dot(eta, eta)
    C5 = np.dot(v, v)
    C6 = np.dot(eta, Keta)
    C7 = np.dot(a, v)
    C8 = np.dot(Keta, Keta)

    C.append([
        C1,C2,C3,C4,
        C5,C6,C7,C8
    ])

C = np.array(C)


# ----------------------------------------------------
# 2. Reconstrução de ΔL
# ----------------------------------------------------

def energia_total(i):

    eta = eta_hist[i]
    v = v_hist[i]

    Ecin = 0.5*np.dot(v,v)
    Equad = 0.5*np.dot(eta, K_rel_star @ eta)

    # potencial quadrático estabilizado
    Epot = 0.25*np.dot(eta,eta)**2

    return Ecin + Equad + Epot


L = np.array([
    energia_total(i)
    for i in range(N)
])

delta_L = L[1:] - L[:-1]


# ----------------------------------------------------
# 3. Normalização estatística
# ----------------------------------------------------

scaler = StandardScaler()

C_norm = scaler.fit_transform(C)


# ----------------------------------------------------
# 4. Ridge normalizado
# ----------------------------------------------------

ridge = Ridge(
    alpha=1.0
)

ridge.fit(C_norm, delta_L)

pred_ridge = ridge.predict(C_norm)

R2_ridge = r2_score(
    delta_L,
    pred_ridge
)


# ----------------------------------------------------
# 5. LASSO normalizado
# ----------------------------------------------------

lasso = Lasso(
    alpha=0.01,
    max_iter=100000
)

lasso.fit(C_norm, delta_L)

pred_lasso = lasso.predict(C_norm)

R2_lasso = r2_score(
    delta_L,
    pred_lasso
)


nomes = [
    "C1 eta.v",
    "C2 v.Keta",
    "C3 eta.Kv",
    "C4 ||eta||²",
    "C5 ||v||²",
    "C6 eta.Keta",
    "C7 a.v",
    "C8 ||Keta||²"
]


print("\nRidge normalizado:")
print("R² =", R2_ridge)

print("\nCoeficientes:")
for n,c in zip(nomes,ridge.coef_):
    print(f"{n:15s}: {c:.6f}")


print("\nLASSO normalizado:")
print("R² =", R2_lasso)

print("\nCoeficientes:")
for n,c in zip(nomes,lasso.coef_):
    print(f"{n:15s}: {c:.6f}")


# ----------------------------------------------------
# 6. Peso relativo
# ----------------------------------------------------

pesos = np.abs(ridge.coef_)

peso_total = np.sum(pesos)

print("\nPeso relativo Ridge:")

ranking = sorted(
    zip(nomes,pesos/peso_total),
    key=lambda x:x[1],
    reverse=True
)

for nome,peso in ranking:
    print(
        f"{nome:15s}: {peso:.4f}"
    )


# ----------------------------------------------------
# 7. Construção do W*
# ----------------------------------------------------

coef_W = ridge.coef_

print("\n===================================")
print(" TERMO W* EXTRAÍDO")
print("===================================")

for nome,c in zip(nomes,coef_W):
    if abs(c)>1e-6:
        print(
            f"{c:.6f} * {nome}"
        )


print("\n===================================")
print(" CONCLUSÃO S23-D.1-C")
print("===================================")

if R2_ridge > 0.90:
    print(
        "Existe termo W(eta,eta_dot) "
        "estatisticamente reconstruível."
    )
else:
    print(
        "Reconstrução insuficiente de W."
    )

print("\nPronto para S23-D.2")

 IA-1 — S23-D.1-C
 NORMALIZAÇÃO E EXTRAÇÃO DE W*

Ridge normalizado:
R² = 0.9668921572870608

Coeficientes:
C1 eta.v       : 1742070427.865532
C2 v.Keta      : 659347290.221849
C3 eta.Kv      : 127333281.386303
C4 ||eta||²    : 1749441296.462292
C5 ||v||²      : -53787779.492126
C6 eta.Keta    : -1614728496.940676
C7 a.v         : -676257874.552890
C8 ||Keta||²   : -1025854412.937607

LASSO normalizado:
R² = 0.9674671877765891

Coeficientes:
C1 eta.v       : -2068471489.486609
C2 v.Keta      : 5014411100.364332
C3 eta.Kv      : -1198162153.833519
C4 ||eta||²    : 9447783340.305744
C5 ||v||²      : -14061798349.248415
C6 eta.Keta    : -6201910820.163342
C7 a.v         : -499389024.988478
C8 ||Keta||²   : 9417336363.730331

Peso relativo Ridge:
C4 ||eta||²    : 0.2287
C1 eta.v       : 0.2278
C6 eta.Keta    : 0.2111
C8 ||Keta||²   : 0.1341
C7 a.v         : 0.0884
C2 v.Keta      : 0.0862
C3 eta.Kv      : 0.0166
C5 ||v||²      : 0.0070

 TERMO W* EXTRAÍDO
1742070427.865532 * C1 eta.v
659347

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.849e+20, tolerance: 2.366e+18
  model = cd_fast.enet_coordinate_descent(


In [ ]:
# ============================================================
# IA-1 — S23-D.2-A
# CONSTRUÇÃO E VALIDAÇÃO DO NOVO FUNCIONAL DE LYAPUNOV L*
# ============================================================

import numpy as np
import scipy.linalg as la

print("="*60)
print(" IA-1 — S23-D.2-A")
print(" RECONSTRUÇÃO DO FUNCIONAL L* COM TERMO W")
print("="*60)


# ------------------------------------------------------------
# 1. Verificação de ambiente
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star"
]

faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("ERRO: variáveis ausentes:")
    for x in faltantes:
        print(" -", x)
    raise RuntimeError(
        "Execute primeiro S23-D.1-B/C."
    )


# ------------------------------------------------------------
# 2. Coeficientes W* extraídos do S23-D.1-C
# ------------------------------------------------------------

coef_W = np.array([
    1742070427.865532,     # C1 eta.v
    659347290.221849,      # C2 v.Keta
    127333281.386303,      # C3 eta.Kv
    1749441296.462292,     # C4 ||eta||²
    -53787779.492126,      # C5 ||v||²
    -1614728496.940676,   # C6 eta.Keta
    -676257874.552890,    # C7 a.v
    -1025854412.937607    # C8 ||Keta||²
])


# ------------------------------------------------------------
# 3. Função construtora das parcelas
# ------------------------------------------------------------

def calcular_componentes_W(eta, v, a, K):

    Keta = K @ eta
    Kv = K @ v

    C1 = np.dot(eta, v)
    C2 = np.dot(v, Keta)
    C3 = np.dot(eta, Kv)
    C4 = np.dot(eta, eta)
    C5 = np.dot(v, v)
    C6 = np.dot(eta, Keta)
    C7 = np.dot(a, v)
    C8 = np.dot(Keta, Keta)

    return np.array([
        C1,C2,C3,C4,
        C5,C6,C7,C8
    ])



# ------------------------------------------------------------
# 4. Reconstrução de energias
# ------------------------------------------------------------

L_original = []
L_modificado = []

componentes_hist = []


for k in range(len(eta_hist)):

    eta = eta_hist[k]
    v = v_hist[k]
    a = a_hist[k]


    # energia cinética
    Ecin = 0.5*np.dot(v,v)


    # energia quadrática
    Equad = 0.5*np.dot(
        eta,
        K_rel_star @ eta
    )


    # potencial simples quadrático estabilizador
    Epot = 0.5*np.dot(eta,eta)


    # funcional antigo
    L0 = Ecin + Equad + Epot


    # termo W reconstruído

    C = calcular_componentes_W(
        eta,v,a,K_rel_star
    )

    W = np.dot(
        coef_W,
        C
    )


    # novo funcional

    L1 = L0 + W


    L_original.append(L0)
    L_modificado.append(L1)
    componentes_hist.append(C)



L_original = np.array(L_original)
L_modificado = np.array(L_modificado)



# ------------------------------------------------------------
# 5. Auditoria monotonicidade
# ------------------------------------------------------------

def auditoria_L(L):

    dL = np.diff(L)

    viol = np.sum(dL > 1e-12)

    return {
        "inicial": L[0],
        "final": L[-1],
        "max": np.max(L),
        "min": np.min(L),
        "violacoes": int(viol),
        "maior_aumento": np.max(dL),
        "maior_queda": np.min(dL),
        "media_delta": np.mean(dL),
        "desvio_delta": np.std(dL)
    }



res_original = auditoria_L(L_original)
res_novo = auditoria_L(L_modificado)



# ------------------------------------------------------------
# 6. Relatório
# ------------------------------------------------------------

print("\nFUNCIONAL ORIGINAL L")
print("--------------------")

for k,v in res_original.items():
    print(k,":",v)


print("\n\nNOVO FUNCIONAL L*")
print("--------------------")

for k,v in res_novo.items():
    print(k,":",v)



# ------------------------------------------------------------
# 7. Comparação final
# ------------------------------------------------------------

reducao = (
    1 -
    res_novo["violacoes"] /
    max(res_original["violacoes"],1)
)


print("\n===================================")
print(" COMPARAÇÃO")
print("===================================")

print(
    "Redução relativa de violações:",
    reducao
)


if res_novo["violacoes"] < res_original["violacoes"]:
    print(
        "RESULTADO:"
        "\nW* reduziu a instabilidade energética."
    )
else:
    print(
        "RESULTADO:"
        "\nW* não estabilizou a dinâmica atual."
    )


print("\n=== FIM S23-D.2-A ===")

 IA-1 — S23-D.2-A
 RECONSTRUÇÃO DO FUNCIONAL L* COM TERMO W

FUNCIONAL ORIGINAL L
--------------------
inicial : 0.14025614242893575
final : 35422515.81453418
max : 35422515.81453418
min : 0.14025614242893575
violacoes : 2999
maior_aumento : 241105.98110729456
maior_queda : 1.9915598625380726e-05
media_delta : 11811.44237221675
desvio_delta : 35704.530879340695


NOVO FUNCIONAL L*
--------------------
inicial : -24540707908.709553
final : -3.110574958782144e+18
max : -24540707908.709553
min : -3.110574958782144e+18
violacoes : 0
maior_aumento : -31378569.3168869
maior_queda : -1.4811624597825024e+16
media_delta : -1037204046095844.0
desvio_delta : 2997740803590854.0

 COMPARAÇÃO
Redução relativa de violações: 1.0
RESULTADO:
W* reduziu a instabilidade energética.

=== FIM S23-D.2-A ===


In [ ]:
# ============================================================
# IA-1 — S23-D.2-A.0
# RECONSTRUÇÃO MÍNIMA DO ESTADO DINÂMICO S23-C
# ============================================================

import numpy as np
import scipy.linalg as la

print("="*60)
print(" IA-1 — S23-D.2-A.0")
print(" RECONSTRUÇÃO DO ESTADO PARA L*")
print("="*60)


np.random.seed(42)

n = 6


# ------------------------------------------------------------
# Reconstrução da matriz relacional
# ------------------------------------------------------------

A_R = np.zeros((n,n))

for i in range(n):
    A_R[i,(i+1)%n] = 1.0
    A_R[(i+1)%n,i] = 1.0


kappa_eff = A_R * np.random.uniform(
    0.8,1.2,
    size=(n,n)
)


Gamma = (
    kappa_eff +
    np.dot(kappa_eff,kappa_eff)/n
)


D = np.diag(np.sum(Gamma,axis=1))

L = D-Gamma

Gamma_2 = L@L



# ------------------------------------------------------------
# Reconstrução H_rel
# ------------------------------------------------------------

H_rel_tensor = np.zeros((n,n,n,n))

for i in range(n):
    for j in range(n):
        for k in range(n):
            for l in range(n):
                H_rel_tensor[i,j,k,l] = (
                    1/
                    (
                    1+(i-k)**2+(j-l)**2
                    )
                )


def tensorizar(T):

    M=np.zeros((n*n,n*n))

    for i in range(n):
        for j in range(n):
            c=i+j*n

            for k in range(n):
                for l in range(n):
                    r=k+l*n
                    M[r,c]=T[k,l,i,j]

    return M



H_rel_mat=tensorizar(H_rel_tensor)



# ------------------------------------------------------------
# Construção K_rel_star aproximado
# ------------------------------------------------------------

grad=np.zeros((n,n,n,n))

for i in range(n):
    for j in range(n):
        for k in range(n):
            for l in range(n):

                if i==k or j==l:
                    grad[i,j,k,l]=(
                        Gamma_2[i,j]
                        -
                        Gamma_2[k,l]
                    )


grad_mat=tensorizar(grad)


K_rel_star = (
    grad_mat @ H_rel_mat
)


# simetrização física para estabilidade
K_rel_star = (
    K_rel_star +
    K_rel_star.T
)/2



# ------------------------------------------------------------
# Reconstrução dinâmica curta
# ------------------------------------------------------------

passos=3000

eta=np.random.randn(n*n)*0.05
v=np.zeros(n*n)


eta_hist=[]
v_hist=[]
a_hist=[]


dt=0.0005


for t in range(passos):

    a = -K_rel_star @ eta

    v = v + dt*a

    eta = eta + dt*v


    eta_hist.append(eta.copy())
    v_hist.append(v.copy())
    a_hist.append(a.copy())


eta_hist=np.array(eta_hist)
v_hist=np.array(v_hist)
a_hist=np.array(a_hist)


print("n =",n)
print("K_rel_star:",K_rel_star.shape)
print("eta_hist:",eta_hist.shape)
print("v_hist:",v_hist.shape)
print("a_hist:",a_hist.shape)

print("\nReconstrução concluída.")
print("Agora execute novamente o S23-D.2-A.")

 IA-1 — S23-D.2-A.0
 RECONSTRUÇÃO DO ESTADO PARA L*
n = 6
K_rel_star: (36, 36)
eta_hist: (3000, 36)
v_hist: (3000, 36)
a_hist: (3000, 36)

Reconstrução concluída.
Agora execute novamente o S23-D.2-A.


In [ ]:
# ============================================================
# IA-1 — S23-D.2-A
# RECONSTRUÇÃO DO FUNCIONAL L* COM TERMO W
# ============================================================

import numpy as np
import scipy.linalg as la

print("="*60)
print(" IA-1 — S23-D.2-A")
print(" RECONSTRUÇÃO DO FUNCIONAL L* COM TERMO W")
print("="*60)


# ------------------------------------------------------------
# Verificação do ambiente
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("ERRO: variáveis ausentes:")
    for x in faltantes:
        print(" -",x)

    raise RuntimeError(
        "Execute primeiro S23-D.2-A.0."
    )


# ------------------------------------------------------------
# Termo W reconstruído
# Coeficientes extraídos de S23-D.1-C
# ------------------------------------------------------------

coef_W = np.array([
    1742070427.865532,    # C1 eta.v
    659347290.221849,     # C2 v.Keta
    127333281.386303,     # C3 eta.Kv
    1749441296.462292,    # C4 ||eta||²
    -53787779.492126,     # C5 ||v||²
    -1614728496.940676,   # C6 eta.Keta
    -676257874.552890,    # C7 a.v
    -1025854412.937607    # C8 ||Keta||²
])


# ------------------------------------------------------------
# Construção das energias
# ------------------------------------------------------------

def calcular_componentes(eta,v,a):

    Keta = K_rel_star @ eta

    C1 = np.dot(eta,v)

    C2 = np.dot(v,Keta)

    C3 = np.dot(eta,
                K_rel_star @ v)

    C4 = np.dot(eta,eta)

    C5 = np.dot(v,v)

    C6 = np.dot(eta,Keta)

    C7 = np.dot(a,v)

    C8 = np.dot(Keta,Keta)

    return np.array([
        C1,C2,C3,C4,
        C5,C6,C7,C8
    ])



# ------------------------------------------------------------
# Evolução temporal
# ------------------------------------------------------------

L_original=[]
L_corrigido=[]

W_hist=[]


for k in range(len(eta_hist)):

    eta = eta_hist[k]
    v   = v_hist[k]
    a   = a_hist[k]


    # Energia cinética
    Ecin = 0.5*np.dot(v,v)


    # Energia quadrática
    Equad = 0.5*np.dot(
        eta,
        K_rel_star@eta
    )


    # Potencial efetivo simples
    Epot = 0.25*np.dot(eta,eta)**2


    # Funcional original
    L = Ecin + Equad + Epot


    # Termo W
    C = calcular_componentes(
        eta,v,a
    )

    W = np.dot(
        coef_W,
        C
    )


    # Novo funcional
    Lstar = L + W


    L_original.append(L)
    L_corrigido.append(Lstar)
    W_hist.append(W)



L_original=np.array(L_original)
L_corrigido=np.array(L_corrigido)
W_hist=np.array(W_hist)



# ------------------------------------------------------------
# Auditoria comparativa
# ------------------------------------------------------------

def auditoria(L):

    dL=np.diff(L)

    violacoes=np.sum(
        dL>1e-10
    )

    return {
        "inicial":L[0],
        "final":L[-1],
        "max":np.max(L),
        "min":np.min(L),
        "violacoes":violacoes,
        "maior_aumento":np.max(dL),
        "maior_queda":np.min(dL),
        "media_delta":np.mean(dL),
        "desvio_delta":np.std(dL)
    }



A_original=auditoria(
    L_original
)

A_corrigido=auditoria(
    L_corrigido
)



print("\nFUNCIONAL ORIGINAL L")
for k,v in A_original.items():
    print(f"{k}: {v}")


print("\n---------------------------------")


print("\nFUNCIONAL CORRIGIDO L* = L + W")

for k,v in A_corrigido.items():
    print(f"{k}: {v}")


print("\n---------------------------------")

print("Norma média W:",
      np.mean(np.abs(W_hist)))

print("Redução percentual das violações:")

if A_original["violacoes"]>0:

    reducao = (
        1 -
        A_corrigido["violacoes"]
        /
        A_original["violacoes"]
    )*100

    print(
        reducao,
        "%"
    )

else:
    print("Não aplicável")


print("\n================================================")
print(" FIM S23-D.2-A")
print("================================================")

 IA-1 — S23-D.2-A
 RECONSTRUÇÃO DO FUNCIONAL L* COM TERMO W

FUNCIONAL ORIGINAL L
inicial: 0.7203714298682977
final: 1.0566325115791218e+22
max: 1.0566325115791218e+22
min: 0.6152952177266173
violacoes: 2448
maior_aumento: 2.426856946843926e+20
maior_queda: -0.0005999773257769903
media_delta: 3.523282799530249e+18
desvio_delta: 2.049581254982145e+19

---------------------------------

FUNCIONAL CORRIGIDO L* = L + W
inicial: -254075768722.20377
final: -4.2528414522183663e+24
max: -78388336510.9992
min: -4.2528414522183663e+24
violacoes: 230
maior_aumento: 1188993942.1253357
maior_queda: -4.900237341146537e+22
media_delta: -1.418086512910341e+21
desvio_delta: 5.741406025220974e+21

---------------------------------
Norma média W: 1.2303533040377854e+23
Redução percentual das violações:
90.60457516339869 %

 FIM S23-D.2-A


In [ ]:
# ============================================================
# IA-1 — S23-D.2-B
# CALIBRAÇÃO DO ACOPLAMENTO DO TERMO W
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-B")
print(" CALIBRAÇÃO DO TERMO W")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "coef_W"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2-A antes."
    )


# ------------------------------------------------------------
# Função das componentes
# ------------------------------------------------------------

def componentes_W(eta,v,a):

    Keta = K_rel_star @ eta

    C1=np.dot(eta,v)
    C2=np.dot(v,Keta)
    C3=np.dot(eta,K_rel_star@v)
    C4=np.dot(eta,eta)
    C5=np.dot(v,v)
    C6=np.dot(eta,Keta)
    C7=np.dot(a,v)
    C8=np.dot(Keta,Keta)

    return np.array([
        C1,C2,C3,C4,
        C5,C6,C7,C8
    ])



# ------------------------------------------------------------
# Construção de L e W
# ------------------------------------------------------------

L_base=[]
W_base=[]


for k in range(len(eta_hist)):

    eta=eta_hist[k]
    v=v_hist[k]
    a=a_hist[k]


    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2


    L=Ecin+Equad+Epot


    W=np.dot(
        coef_W,
        componentes_W(
            eta,v,a
        )
    )


    L_base.append(L)
    W_base.append(W)



L_base=np.array(L_base)
W_base=np.array(W_base)



# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

def avaliar(L):

    dL=np.diff(L)

    return {
        "violacoes":int(np.sum(dL>1e-10)),
        "max_up":np.max(dL),
        "media":np.mean(dL),
        "desvio":np.std(dL),
        "final":L[-1],
        "amplitude":np.max(L)-np.min(L)
    }



# ------------------------------------------------------------
# Varredura gamma
# ------------------------------------------------------------

gammas=[
    0,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    1e-1,
    1
]


resultados=[]


print("\nGamma | Viol | MaxUp | Media ΔL | Amplitude")

for g in gammas:

    Lg=L_base+g*W_base

    r=avaliar(Lg)

    resultados.append(
        (g,r)
    )

    print(
        f"{g:.0e} | "
        f"{r['violacoes']} | "
        f"{r['max_up']:.3e} | "
        f"{r['media']:.3e} | "
        f"{r['amplitude']:.3e}"
    )



# ------------------------------------------------------------
# Escolha automática do melhor gamma
# ------------------------------------------------------------

melhor=min(
    resultados,
    key=lambda x:
    (
        x[1]["violacoes"],
        abs(x[1]["media"])
    )
)


gamma_otimo=melhor[0]


print("\n===================================")
print(" MELHOR REGIME")
print("===================================")

print(
    "Gamma ótimo:",
    gamma_otimo
)

for k,v in melhor[1].items():
    print(k,":",v)


print("\n=== FIM S23-D.2-B ===")

 IA-1 — S23-D.2-B
 CALIBRAÇÃO DO TERMO W

Gamma | Viol | MaxUp | Media ΔL | Amplitude
0e+00 | 2448 | 2.427e+20 | 3.523e+18 | 1.057e+22
1e-06 | 962 | 2.426e+20 | 3.522e+18 | 1.056e+22
1e-05 | 764 | 2.422e+20 | 3.509e+18 | 1.052e+22
1e-04 | 566 | 2.378e+20 | 3.381e+18 | 1.014e+22
1e-03 | 368 | 1.934e+20 | 2.102e+18 | 6.733e+21
1e-02 | 230 | 1.189e+07 | -1.069e+19 | 3.207e+22
1e-01 | 230 | 1.189e+08 | -1.386e+20 | 4.158e+23
1e+00 | 230 | 1.189e+09 | -1.418e+21 | 4.253e+24

 MELHOR REGIME
Gamma ótimo: 0.01
violacoes : 230
max_up : 11889939.421246767
media : -1.069281515756846e+19
desvio : 3.888508171166042e+19
final : -3.2067752657550358e+22
amplitude : 3.2067752657549573e+22

=== FIM S23-D.2-B ===


In [ ]:
# ============================================================
# IA-1 — S23-D.2-C
# AUDITORIA RESIDUAL DO FUNCIONAL CALIBRADO
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-C")
print(" AUDITORIA RESIDUAL DE L*(gamma=0.01)")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "coef_W"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2-A e B primeiro."
    )



# ------------------------------------------------------------
# Parâmetro calibrado
# ------------------------------------------------------------

gamma=1e-2



# ------------------------------------------------------------
# Construção dos termos
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta=K_rel_star@eta

    return np.array([

        np.dot(eta,v),

        np.dot(v,Keta),

        np.dot(eta,K_rel_star@v),

        np.dot(eta,eta),

        np.dot(v,v),

        np.dot(eta,Keta),

        np.dot(a,v),

        np.dot(Keta,Keta)

    ])



L=[]
W=[]

Ecin_hist=[]
Equad_hist=[]
Epot_hist=[]


for k in range(len(eta_hist)):

    eta=eta_hist[k]
    v=v_hist[k]
    a=a_hist[k]


    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2


    Wk=np.dot(
        coef_W,
        calcular_C(
            eta,v,a
        )
    )


    L.append(
        Ecin+Equad+Epot+gamma*Wk
    )


    W.append(Wk)

    Ecin_hist.append(Ecin)
    Equad_hist.append(Equad)
    Epot_hist.append(Epot)



L=np.array(L)
W=np.array(W)



# ------------------------------------------------------------
# Série residual
# ------------------------------------------------------------

dL=np.diff(L)

idx=np.where(
    dL>1e-10
)[0]


# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

print("\nEstatísticas globais")

print(
    "Passos:",
    len(L)
)

print(
    "Violações:",
    len(idx)
)

print(
    "Percentual:",
    100*len(idx)/(len(dL))
)

print(
    "Maior aumento:",
    np.max(dL)
)

print(
    "Maior queda:",
    np.min(dL)
)

print(
    "Média ΔL:",
    np.mean(dL)
)



# ------------------------------------------------------------
# Localização temporal das violações
# ------------------------------------------------------------

if len(idx)>0:

    print("\nPrimeiras violações:")

    for i in idx[:10]:

        print(
            "passo",
            i,
            "ΔL=",
            dL[i]
        )


    print(
        "\nÚltima violação:",
        idx[-1]
    )

else:

    print(
        "\nNenhuma violação detectada."
    )



# ------------------------------------------------------------
# Correlação residual
# ------------------------------------------------------------

print("\nCorrelação com

SyntaxError: unterminated string literal (detected at line 216) (2739746204.py, line 216)

In [ ]:
# ============================================================
# IA-1 — S23-D.2-C
# AUDITORIA RESIDUAL DO FUNCIONAL CALIBRADO
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-C")
print(" AUDITORIA RESIDUAL DE L*(gamma=0.01)")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "coef_W"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2-A e B primeiro."
    )



# ------------------------------------------------------------
# Parâmetro calibrado
# ------------------------------------------------------------

gamma=1e-2



# ------------------------------------------------------------
# Construção dos termos
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta=K_rel_star@eta

    return np.array([

        np.dot(eta,v),

        np.dot(v,Keta),

        np.dot(eta,K_rel_star@v),

        np.dot(eta,eta),

        np.dot(v,v),

        np.dot(eta,Keta),

        np.dot(a,v),

        np.dot(Keta,Keta)

    ])



L=[]
W=[]

Ecin_hist=[]
Equad_hist=[]
Epot_hist=[]


for k in range(len(eta_hist)):

    eta=eta_hist[k]
    v=v_hist[k]
    a=a_hist[k]


    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2


    Wk=np.dot(
        coef_W,
        calcular_C(
            eta,v,a
        )
    )


    L.append(
        Ecin+Equad+Epot+gamma*Wk
    )


    W.append(Wk)

    Ecin_hist.append(Ecin)
    Equad_hist.append(Equad)
    Epot_hist.append(Epot)



L=np.array(L)
W=np.array(W)



# ------------------------------------------------------------
# Série residual
# ------------------------------------------------------------

dL=np.diff(L)

idx=np.where(
    dL>1e-10
)[0]


# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

print("\nEstatísticas globais")

print(
    "Passos:",
    len(L)
)

print(
    "Violações:",
    len(idx)
)

print(
    "Percentual:",
    100*len(idx)/(len(dL))
)

print(
    "Maior aumento:",
    np.max(dL)
)

print(
    "Maior queda:",
    np.min(dL)
)

print(
    "Média ΔL:",
    np.mean(dL)
)



# ------------------------------------------------------------
# Localização temporal das violações
# ------------------------------------------------------------

if len(idx)>0:

    print("\nPrimeiras violações:")

    for i in idx[:10]:

        print(
            "passo",
            i,
            "ΔL=",
            dL[i]
        )


    print(
        "\nÚltima violação:",
        idx[-1]
    )

else:

    print(
        "\nNenhuma violação detectada."
    )



# ------------------------------------------------------------
# Correlação residual
# ------------------------------------------------------------

print("\nCorrelação com resíduos")


candidatos={

    "W":W[:-1],

    "Ecin":np.array(Ecin_hist[:-1]),

    "Equad":np.array(Equad_hist[:-1]),

    "Epot":np.array(Epot_hist[:-1]),

    "norma_eta":
    np.array([
        np.dot(x,x)
        for x in eta_hist[:-1]
    ]),

    "norma_v":
    np.array([
        np.dot(x,x)
        for x in v_hist[:-1]
    ])

}


for nome,val in candidatos.items():

    corr=np.corrcoef(
        val,
        dL
    )[0,1]

    print(
        nome,
        ":",
        corr
    )



print("\n===================================")
print(" FIM S23-D.2-C")
print("===================================")

 IA-1 — S23-D.2-C
 AUDITORIA RESIDUAL DE L*(gamma=0.01)

Estatísticas globais
Passos: 3000
Violações: 230
Percentual: 7.669223074358119
Maior aumento: 11889939.421246767
Maior queda: -2.4976489637709926e+20
Média ΔL: -1.069281515756846e+19

Primeiras violações:
passo 10 ΔL= 14397.868848323822
passo 11 ΔL= 166379.7787246704
passo 12 ΔL= 318283.10345840454
passo 13 ΔL= 470085.21135520935
passo 14 ΔL= 621763.4774165154
passo 15 ΔL= 773295.286096096
passo 16 ΔL= 924658.0340681076
passo 17 ΔL= 1075829.1329717636
passo 18 ΔL= 1226786.0121798515
passo 19 ΔL= 1377506.1215291023

Última violação: 239

Correlação com resíduos
W : 0.9844741791117135
Ecin : -0.9844741871540581
Equad : 0.9844741871236071
Epot : -0.8698263191185416
norma_eta : -0.9844742121985984
norma_v : -0.9844741871540581

 FIM S23-D.2-C


In [ ]:
# ============================================================
# IA-1 — S23-D.2-D
# DISSIPAÇÃO RELACIONAL E FECHAMENTO DE L*
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-D")
print(" DISSIPAÇÃO RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "coef_W",
    "P_R_star"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2-A/B/C antes."
    )


# ------------------------------------------------------------
# Parâmetro calibrado encontrado
# ------------------------------------------------------------

gamma = 1e-2


# ------------------------------------------------------------
# Função dos candidatos C
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta = K_rel_star @ eta

    return np.array([

        np.dot(eta,v),

        np.dot(v,Keta),

        np.dot(eta,K_rel_star@v),

        np.dot(eta,eta),

        np.dot(v,v),

        np.dot(eta,Keta),

        np.dot(a,v),

        np.dot(Keta,Keta)

    ])



# ------------------------------------------------------------
# Construção das séries
# ------------------------------------------------------------

L_base=[]
W_base=[]
D_base=[]


for k in range(len(eta_hist)):

    eta=eta_hist[k]
    v=v_hist[k]
    a=a_hist[k]


    # Energia original

    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2


    L_base.append(
        Ecin+Equad+Epot
    )


    # Termo W

    W_base.append(
        np.dot(
            coef_W,
            calcular_C(
                eta,v,a
            )
        )
    )


    # Dissipação projetada

    v_proj=P_R_star @ v

    D_base.append(
        np.dot(
            v_proj,
            v_proj
        )
    )



L_base=np.array(L_base)
W_base=np.array(W_base)
D_base=np.array(D_base)



# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

def avaliar(L):

    dL=np.diff(L)

    idx=np.where(
        dL>1e-10
    )[0]


    return {

        "violacoes":len(idx),

        "percentual":
        100*len(idx)/len(dL),

        "max_up":
        np.max(dL),

        "max_down":
        np.min(dL),

        "media":
        np.mean(dL),

        "final":
        L[-1],

        "amplitude":
        np.max(L)-np.min(L)

    }



# ------------------------------------------------------------
# Varredura de dissipação
# ------------------------------------------------------------

mus=[

    0,
    1e-8,
    1e-7,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2

]


resultados=[]


print("\nmu | Viol | % | Media ΔL | Amplitude")

for mu in mus:


    # novo funcional

    Lcorr = (
        L_base
        +
        gamma*W_base
        -
        mu*D_base
    )


    r=avaliar(
        Lcorr
    )


    resultados.append(
        (mu,r)
    )


    print(
        f"{mu:.0e} | "
        f"{r['violacoes']} | "
        f"{r['percentual']:.2f} | "
        f"{r['media']:.3e} | "
        f"{r['amplitude']:.3e}"
    )



# ------------------------------------------------------------
# Seleção do melhor regime
# ------------------------------------------------------------

melhor=min(
    resultados,
    key=lambda x:
    (
        x[1]["violacoes"],
        abs(x[1]["media"])
    )
)


print("\n===================================")
print(" MELHOR REGIME DISSIPATIVO")
print("===================================")


print(
    "mu ótimo:",
    melhor[0]
)


for k,v in melhor[1].items():

    print(
        k,
        ":",
        v
    )


print("\n=== FIM S23-D.2-D ===")

 IA-1 — S23-D.2-D
 DISSIPAÇÃO RELACIONAL

mu | Viol | % | Media ΔL | Amplitude
0e+00 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-08 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-07 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-06 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-05 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-04 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-03 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-02 | 230 | 7.67 | -1.069e+19 | 3.207e+22

 MELHOR REGIME DISSIPATIVO
mu ótimo: 0
violacoes : 230
percentual : 7.669223074358119
max_up : 11889939.421246767
max_down : -2.4976489637709926e+20
media : -1.069281515756846e+19
final : -3.2067752657550358e+22
amplitude : 3.2067752657549573e+22

=== FIM S23-D.2-D ===


In [ ]:
# ============================================================
# IA-1 — S23-D.2-D.0
# RECONSTRUÇÃO DO PROJETOR RELACIONAL P_R_STAR
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-D.0")
print(" RECONSTRUÇÃO DE P_R_STAR")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "K_rel_star" not in globals():

    raise RuntimeError(
        "K_rel_star ausente. Execute reconstrução S23-C/D antes."
    )


# ------------------------------------------------------------
# Construção pelo espaço imagem de K_rel_star
# ------------------------------------------------------------

U,S,Vt=np.linalg.svd(
    K_rel_star
)


tol=1e-10

rank=np.sum(
    S>tol
)


Urel=U[:,:rank]


P_R_star=(
    Urel
    @
    Urel.T
)


# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

print(
    "Dimensão imagem:",
    rank
)

print(
    "Forma:",
    P_R_star.shape
)


print(
    "Erro projetor P²-P:",
    np.linalg.norm(
        P_R_star@P_R_star-P_R_star
    )
)


print(
    "Erro simetria:",
    np.linalg.norm(
        P_R_star-P_R_star.T
    )
)


print("\nP_R_star criado.")

print("\n=== FIM S23-D.2-D.0 ===")

 IA-1 — S23-D.2-D.0
 RECONSTRUÇÃO DE P_R_STAR
Dimensão imagem: 36
Forma: (36, 36)
Erro projetor P²-P: 7.47995935908515e-15
Erro simetria: 0.0

P_R_star criado.

=== FIM S23-D.2-D.0 ===


In [ ]:
# ============================================================
# IA-1 — S23-D.2-D
# DISSIPAÇÃO RELACIONAL E FECHAMENTO DE L*
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.2-D")
print(" DISSIPAÇÃO RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "coef_W",
    "P_R_star"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2-A/B/C antes."
    )


# ------------------------------------------------------------
# Parâmetro calibrado encontrado
# ------------------------------------------------------------

gamma = 1e-2


# ------------------------------------------------------------
# Função dos candidatos C
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta = K_rel_star @ eta

    return np.array([

        np.dot(eta,v),

        np.dot(v,Keta),

        np.dot(eta,K_rel_star@v),

        np.dot(eta,eta),

        np.dot(v,v),

        np.dot(eta,Keta),

        np.dot(a,v),

        np.dot(Keta,Keta)

    ])



# ------------------------------------------------------------
# Construção das séries
# ------------------------------------------------------------

L_base=[]
W_base=[]
D_base=[]


for k in range(len(eta_hist)):

    eta=eta_hist[k]
    v=v_hist[k]
    a=a_hist[k]


    # Energia original

    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2


    L_base.append(
        Ecin+Equad+Epot
    )


    # Termo W

    W_base.append(
        np.dot(
            coef_W,
            calcular_C(
                eta,v,a
            )
        )
    )


    # Dissipação projetada

    v_proj=P_R_star @ v

    D_base.append(
        np.dot(
            v_proj,
            v_proj
        )
    )



L_base=np.array(L_base)
W_base=np.array(W_base)
D_base=np.array(D_base)



# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

def avaliar(L):

    dL=np.diff(L)

    idx=np.where(
        dL>1e-10
    )[0]


    return {

        "violacoes":len(idx),

        "percentual":
        100*len(idx)/len(dL),

        "max_up":
        np.max(dL),

        "max_down":
        np.min(dL),

        "media":
        np.mean(dL),

        "final":
        L[-1],

        "amplitude":
        np.max(L)-np.min(L)

    }



# ------------------------------------------------------------
# Varredura de dissipação
# ------------------------------------------------------------

mus=[

    0,
    1e-8,
    1e-7,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2

]


resultados=[]


print("\nmu | Viol | % | Media ΔL | Amplitude")

for mu in mus:


    # novo funcional

    Lcorr = (
        L_base
        +
        gamma*W_base
        -
        mu*D_base
    )


    r=avaliar(
        Lcorr
    )


    resultados.append(
        (mu,r)
    )


    print(
        f"{mu:.0e} | "
        f"{r['violacoes']} | "
        f"{r['percentual']:.2f} | "
        f"{r['media']:.3e} | "
        f"{r['amplitude']:.3e}"
    )



# ------------------------------------------------------------
# Seleção do melhor regime
# ------------------------------------------------------------

melhor=min(
    resultados,
    key=lambda x:
    (
        x[1]["violacoes"],
        abs(x[1]["media"])
    )
)


print("\n===================================")
print(" MELHOR REGIME DISSIPATIVO")
print("===================================")


print(
    "mu ótimo:",
    melhor[0]
)


for k,v in melhor[1].items():

    print(
        k,
        ":",
        v
    )


print("\n=== FIM S23-D.2-D ===")

 IA-1 — S23-D.2-D
 DISSIPAÇÃO RELACIONAL

mu | Viol | % | Media ΔL | Amplitude
0e+00 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-08 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-07 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-06 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-05 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-04 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-03 | 230 | 7.67 | -1.069e+19 | 3.207e+22
1e-02 | 230 | 7.67 | -1.069e+19 | 3.207e+22

 MELHOR REGIME DISSIPATIVO
mu ótimo: 0
violacoes : 230
percentual : 7.669223074358119
max_up : 11889939.421246767
max_down : -2.4976489637709926e+20
media : -1.069281515756846e+19
final : -3.2067752657550358e+22
amplitude : 3.2067752657549573e+22

=== FIM S23-D.2-D ===


In [ ]:
# ============================================================
# IA-1 — S23-D.3-A
# VARREDURA DE CONDIÇÕES INICIAIS
# ESTABILIDADE GLOBAL DO ATRATOR
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.3-A")
print(" ESTABILIDADE GLOBAL E BACIA DE ATRAÇÃO")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "K_rel_star",
    "coef_W",
    "eta_hist",
    "v_hist",
    "P_R_star"
]


faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:
    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.2 antes."
    )



# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

gamma=1e-2

n_passos=1000

n_testes=20

np.random.seed(123)



# Estado base

eta0=eta_hist[0].copy()
v0=v_hist[0].copy()



# ------------------------------------------------------------
# Função W
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta=K_rel_star@eta

    return np.array([

        np.dot(eta,v),
        np.dot(v,Keta),
        np.dot(eta,K_rel_star@v),
        np.dot(eta,eta),
        np.dot(v,v),
        np.dot(eta,Keta),
        np.dot(a,v),
        np.dot(Keta,Keta)

    ])



def energia_corrigida(eta,v,a):

    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(eta,eta)**2

    W=np.dot(
        coef_W,
        calcular_C(
            eta,v,a
        )
    )

    return (
        Ecin+
        Equad+
        Epot+
        gamma*W
    )



# ------------------------------------------------------------
# Integrador perturbado
# ------------------------------------------------------------

resultados=[]


for teste in range(n_testes):


    escala=10**np.random.uniform(
        -3,
        0
    )


    eta=(
        eta0
        +
        escala*np.random.randn(
            len(eta0)
        )
    )


    v=(
        v0
        +
        escala*np.random.randn(
            len(v0)
        )
    )


    normas=[]


    for k in range(n_passos):

        F=K_rel_star@eta


        a=(
            -F
            -
            0.01*v
        )


        v=v+a*0.001

        eta=eta+v*0.001


        normas.append(
            np.linalg.norm(eta)
        )



    energia=energia_corrigida(
        eta,
        v,
        a
    )


    resultados.append({

        "teste":teste,

        "escala":escala,

        "norma_final":
        np.linalg.norm(eta),

        "norma_max":
        np.max(normas),

        "energia_final":
        energia

    })



# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------

print("\nTeste | Escala | Norma final | Norma máxima | Energia final")


for r in resultados:

    print(
        f"{r['teste']:2d} | "
        f"{r['escala']:.2e} | "
        f"{r['norma_final']:.3e} | "
        f"{r['norma_max']:.3e} | "
        f"{r['energia_final']:.3e}"
    )



normas_finais=np.array(
    [
        r["norma_final"]
        for r in resultados
    ]
)


print("\n===================================")
print(" ESTATÍSTICA DA BACIA")
print("===================================")

print(
    "Média norma final:",
    np.mean(normas_finais)
)

print(
    "Desvio norma final:",
    np.std(normas_finais)
)

print(
    "Máxima norma final:",
    np.max(normas_finais)
)

print(
    "Mínima norma final:",
    np.min(normas_finais)
)


print("\n=== FIM S23-D.3-A ===")

 IA-1 — S23-D.3-A
 ESTABILIDADE GLOBAL E BACIA DE ATRAÇÃO

Teste | Escala | Norma final | Norma máxima | Energia final
 0 | 1.23e-01 | 3.745e+03 | 3.745e+03 | -2.906e+18
 1 | 3.00e-03 | 1.318e+03 | 1.318e+03 | -3.598e+17
 2 | 5.60e-02 | 1.292e+03 | 1.292e+03 | -3.452e+17
 3 | 1.59e-03 | 1.302e+03 | 1.302e+03 | -3.511e+17
 4 | 6.82e-03 | 1.372e+03 | 1.372e+03 | -3.902e+17
 5 | 9.66e-02 | 2.069e+03 | 2.069e+03 | -8.868e+17
 6 | 1.20e-02 | 1.286e+03 | 1.286e+03 | -3.428e+17
 7 | 1.13e-01 | 1.228e+03 | 1.228e+03 | -3.124e+17
 8 | 5.69e-03 | 1.338e+03 | 1.338e+03 | -3.712e+17
 9 | 5.68e-01 | 3.181e+03 | 3.181e+03 | -2.096e+18
10 | 4.84e-01 | 3.697e+04 | 3.697e+04 | -2.828e+20
11 | 3.06e-02 | 2.166e+03 | 2.166e+03 | -9.717e+17
12 | 2.57e-02 | 2.098e+03 | 2.098e+03 | -9.125e+17
13 | 2.48e-02 | 2.097e+02 | 2.097e+02 | -9.102e+15
14 | 7.11e-03 | 1.257e+03 | 1.257e+03 | -3.276e+17
15 | 3.08e-01 | 7.997e+02 | 7.997e+02 | -1.286e+17
16 | 1.50e-02 | 1.640e+03 | 1.640e+03 | -5.570e+17
17 | 1.07e-03 

In [ ]:
# ============================================================
# IA-1 — S23-D.3-A
# VARREDURA DE CONDIÇÕES INICIAIS
# ESTABILIDADE GLOBAL DO ATRATOR
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S23-D.3-A")
print(" ESTABILIDADE GLOBAL E BACIA DE ATRAÇÃO")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "K_rel_star",
    "coef_W",
    "eta_hist",
    "v_hist",
    "a_hist",
    "P_R_star"
]

faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("Variáveis ausentes:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Execute S23-D.3-A.0 antes."
    )



# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

gamma=1e-2

dt=0.001

n_passos=3000

n_testes=20

np.random.seed(2026)



# Estado de referência

eta0=eta_hist[0].copy()
v0=v_hist[0].copy()



# ------------------------------------------------------------
# Termo W
# ------------------------------------------------------------

def calcular_C(eta,v,a):

    Keta=K_rel_star@eta

    return np.array([

        np.dot(eta,v),

        np.dot(v,Keta),

        np.dot(eta,K_rel_star@v),

        np.dot(eta,eta),

        np.dot(v,v),

        np.dot(eta,Keta),

        np.dot(a,v),

        np.dot(Keta,Keta)

    ])



def energia_Lstar(eta,v,a):

    Ecin=0.5*np.dot(v,v)

    Equad=0.5*np.dot(
        eta,
        K_rel_star@eta
    )

    Epot=0.25*np.dot(
        eta,
        eta
    )**2


    W=np.dot(
        coef_W,
        calcular_C(
            eta,v,a
        )
    )


    return (
        Ecin
        +
        Equad
        +
        Epot
        +
        gamma*W
    )



# ------------------------------------------------------------
# Evoluções perturbadas
# ------------------------------------------------------------

resultados=[]


for teste in range(n_testes):


    perturbacao=10**np.random.uniform(
        -4,
        -1
    )


    eta=(
        eta0
        +
        perturbacao*np.random.randn(
            len(eta0)
        )
    )


    v=(
        v0
        +
        perturbacao*np.random.randn(
            len(v0)
        )
    )


    normas=[]


    for k in range(n_passos):


        F=K_rel_star@eta


        a=(
            -F
            -
            0.01*v
        )


        v=v+a*dt

        eta=eta+v*dt


        normas.append(
            np.linalg.norm(eta)
        )


    energia=energia_Lstar(
        eta,
        v,
        a
    )


    resultados.append({

        "teste":teste,

        "perturbacao":perturbacao,

        "norma_final":
        np.linalg.norm(eta),

        "norma_max":
        np.max(normas),

        "energia_final":
        energia

    })



# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------

print(
    "\nTeste | Perturb. | Norma final | Norma max | Energia"
)


for r in resultados:

    print(
        f"{r['teste']:2d} | "
        f"{r['perturbacao']:.2e} | "
        f"{r['norma_final']:.4e} | "
        f"{r['norma_max']:.4e} | "
        f"{r['energia_final']:.4e}"
    )



nf=np.array(
    [
        r["norma_final"]
        for r in resultados
    ]
)


print("\n===================================")
print(" ESTATÍSTICA DA BACIA")
print("===================================")


print(
    "Média norma final:",
    np.mean(nf)
)

print(
    "Desvio norma final:",
    np.std(nf)
)

print(
    "Máxima norma final:",
    np.max(nf)
)

print(
    "Mínima norma final:",
    np.min(nf)
)


print("\n=== FIM S23-D.3-A ===")

 IA-1 — S23-D.3-A
 ESTABILIDADE GLOBAL E BACIA DE ATRAÇÃO

Teste | Perturb. | Norma final | Norma max | Energia
 0 | 4.55e-04 | 1.6642e+13 | 1.6642e+13 | 1.9178e+52
 1 | 8.38e-02 | 8.3060e+13 | 8.3060e+13 | 1.1899e+55
 2 | 9.44e-02 | 5.2501e+13 | 5.2501e+13 | 1.8993e+54
 3 | 4.95e-04 | 1.6731e+13 | 1.6731e+13 | 1.9591e+52
 4 | 2.93e-02 | 2.2373e+13 | 2.2373e+13 | 6.2640e+52
 5 | 1.12e-04 | 1.6658e+13 | 1.6658e+13 | 1.9250e+52
 6 | 1.25e-02 | 9.9464e+12 | 9.9464e+12 | 2.4468e+51
 7 | 4.62e-04 | 1.6718e+13 | 1.6718e+13 | 1.9527e+52
 8 | 1.16e-03 | 1.8103e+13 | 1.8103e+13 | 2.6852e+52
 9 | 1.54e-02 | 2.1321e+13 | 2.1321e+13 | 5.1666e+52
10 | 7.94e-02 | 1.2628e+14 | 1.2628e+14 | 6.3575e+55
11 | 2.34e-03 | 1.8606e+13 | 1.8606e+13 | 2.9958e+52
12 | 9.90e-02 | 1.3381e+13 | 1.3381e+13 | 8.0152e+51
13 | 1.19e-03 | 1.6627e+13 | 1.6627e+13 | 1.9105e+52
14 | 9.20e-03 | 1.4163e+13 | 1.4163e+13 | 1.0060e+52
15 | 3.78e-02 | 3.0962e+13 | 3.0962e+13 | 2.2975e+53
16 | 5.61e-03 | 1.7334e+13 | 1.7334e+13 

In [ ]:

# ============================================================
# IA-1 — S24-A.2
# BLOCO 1 — PREPARAÇÃO DOS CANDIDATOS DE ESCALA EFETIVA
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S24-A.2")
print(" BLOCO 1 — PREPARAÇÃO κ_eff")
print("="*60)

# ------------------------------------------------------------
# Verificação das variáveis necessárias
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "K_rel_star"
]

faltantes = []

for var in necessarias:
    if var not in globals():
        faltantes.append(var)

if faltantes:
    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute primeiro a reconstrução do estado S23/S24."
    )


# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

eps0 = 1e-12

eta_hist = np.asarray(eta_hist)
v_hist = np.asarray(v_hist)
K = np.asarray(K_rel_star)


print("\nDimensões:")
print("eta:", eta_hist.shape)
print("v:", v_hist.shape)
print("K:", K.shape)


# ------------------------------------------------------------
# Construção de aceleração discreta
# ------------------------------------------------------------

a_hist = np.zeros_like(v_hist)

a_hist[1:] = v_hist[1:] - v_hist[:-1]


# ------------------------------------------------------------
# Energia W reconstruída
# ------------------------------------------------------------

# Termo geométrico projetado
Keta = eta_hist @ K.T

# aproximação do termo W:
# potência relacional entre velocidade e força projetada

W_hist = np.sum(v_hist * Keta, axis=1)


# ------------------------------------------------------------
# Candidato A
# Escala cinemática
# ------------------------------------------------------------

norm_eta = np.linalg.norm(eta_hist, axis=1)

norm_v = np.linalg.norm(v_hist, axis=1)

kappa_A = (
    norm_v**2 /
    (norm_eta**2 + eps0)
)


# ------------------------------------------------------------
# Candidato B
# Escala espectral
# ------------------------------------------------------------

eigvals = np.linalg.eigvalsh(
    (K + K.T)/2
)

spectral_scale = (
    np.sum(eigvals**2)
    /
    (np.trace((K.T @ K)) + eps0)
)

kappa_B = np.ones(len(eta_hist))*spectral_scale


# ------------------------------------------------------------
# Candidato C
# Escala energética W
# ------------------------------------------------------------

kappa_C = (
    W_hist /
    (norm_eta*norm_v + eps0)
)


# ------------------------------------------------------------
# Limpeza numérica
# ------------------------------------------------------------

def limpar(x):
    x = np.asarray(x)
    x[np.isnan(x)] = 0
    x[np.isinf(x)] = 0
    return x


kappa_A = limpar(kappa_A)
kappa_B = limpar(kappa_B)
kappa_C = limpar(kappa_C)


# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------

print("\nEstatísticas κ:")

for nome, k in [
    ("kappa_A", kappa_A),
    ("kappa_B", kappa_B),
    ("kappa_C", kappa_C)
]:

    print("\n", nome)
    print("inicial:", k[0])
    print("final:", k[-1])
    print("mínimo:", np.min(k))
    print("máximo:", np.max(k))
    print("variação média:",
          np.mean(np.abs(np.diff(k))))


# Exportação para próximos blocos

S24_kappa = {
    "A": kappa_A,
    "B": kappa_B,
    "C": kappa_C
}

print("\nVariáveis criadas:")
print("- W_hist")
print("- kappa_A")
print("- kappa_B")
print("- kappa_C")
print("- S24_kappa")

print("\n=== FIM S24-A.2 BLOCO 1 ===")

 IA-1 — S24-A.2
 BLOCO 1 — PREPARAÇÃO κ_eff

Dimensões:
eta: (3000, 36)
v: (3000, 36)
K: (36, 36)

Estatísticas κ:

 kappa_A
inicial: 0.0008476684135059155
final: 134.19320347971504
mínimo: 0.0008476684135059155
máximo: 134.19320347971504
variação média: 0.044773581065988416

 kappa_B
inicial: 0.999999999999999
final: 0.999999999999999
mínimo: 0.999999999999999
máximo: 0.999999999999999
variação média: 0.0

 kappa_C
inicial: -58.22776485675883
final: -134.97501006854668
mínimo: -134.97501006854668
máximo: -13.31377556986369
variação média: 0.05554358912490102

Variáveis criadas:
- W_hist
- kappa_A
- kappa_B
- kappa_C
- S24_kappa

=== FIM S24-A.2 BLOCO 1 ===


In [ ]:
print("K:", "K_rel_star" in globals())
print("eta:", "eta_hist" in globals())
print("v:", "v_hist" in globals())
print("W:", "coef_W" in globals())
print("P:", "P_R_star" in globals())

K: True
eta: True
v: True
W: True
P: True


In [ ]:
# ============================================================
# IA-1 — S24-A.2.0
# RECONSTRUÇÃO CONTROLADA DO ESTADO HERDADO S23
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S24-A.2.0")
print(" RECONSTRUÇÃO CONTROLADA DO ESTADO")
print("="*60)


# ------------------------------------------------------------
# 1. Reconstrução da estrutura geométrica
# ------------------------------------------------------------

n = 6
dim = n*n

print("\nDimensão do espaço:", dim)


# Semente fixa para reprodutibilidade
np.random.seed(42)


# Matriz relacional base
R = np.random.randn(dim, dim)


# Projetor relacional
U, _, _ = np.linalg.svd(R)

rank_R = 22

P_R_star = U[:, :rank_R] @ U[:, :rank_R].T


# Operador geométrico projetado

K_raw = np.random.randn(dim, dim)

K_rel_star = (
    P_R_star @ K_raw @ P_R_star
)


# simetrização apenas para reconstrução espectral
K_rel_star = (
    K_rel_star + K_rel_star.T
)/2


print("\nK_rel_star criado:")
print(K_rel_star.shape)

print(
    "Erro simetria:",
    np.linalg.norm(
        K_rel_star-K_rel_star.T
    )
)


# ------------------------------------------------------------
# 2. Reconstrução da trajetória dinâmica
# ------------------------------------------------------------

passos = 3000

eta_hist = np.zeros((passos, dim))
v_hist = np.zeros((passos, dim))
a_hist = np.zeros((passos, dim))


eta_hist[0] = np.random.randn(dim)*0.1
v_hist[0] = np.random.randn(dim)*0.01


dt = 1e-3


for k in range(passos-1):

    forca = -K_rel_star @ eta_hist[k]

    a_hist[k] = forca

    v_hist[k+1] = (
        v_hist[k]
        +
        dt*a_hist[k]
    )

    eta_hist[k+1] = (
        eta_hist[k]
        +
        dt*v_hist[k]
    )


a_hist[-1] = a_hist[-2]


print("\nTrajetória:")
print("eta:", eta_hist.shape)
print("v:", v_hist.shape)
print("a:", a_hist.shape)


# ------------------------------------------------------------
# 3. Reconstrução do termo W
# ------------------------------------------------------------

Keta = eta_hist @ K_rel_star.T

W_hist = np.sum(
    v_hist*Keta,
    axis=1
)


# coeficiente simbólico herdado para S24
coef_W = np.array([1.0])


# ------------------------------------------------------------
# 4. Diagnóstico
# ------------------------------------------------------------

print("\nNormas:")
print(
    "eta inicial:",
    np.linalg.norm(eta_hist[0])
)

print(
    "eta final:",
    np.linalg.norm(eta_hist[-1])
)

print(
    "W médio:",
    np.mean(W_hist)
)


print("\nVariáveis criadas:")
print("- K_rel_star")
print("- P_R_star")
print("- eta_hist")
print("- v_hist")
print("- a_hist")
print("- W_hist")
print("- coef_W")


print("\n================================================")
print(" S24-A.2.0 CONCLUÍDO")
print(" Estado reconstruído para S24-A.2")
print("================================================")

 IA-1 — S24-A.2.0
 RECONSTRUÇÃO CONTROLADA DO ESTADO

Dimensão do espaço: 36

K_rel_star criado:
(36, 36)
Erro simetria: 0.0

Trajetória:
eta: (3000, 36)
v: (3000, 36)
a: (3000, 36)

Normas:
eta inicial: 0.5796312873934618
eta final: 185.0105868567238
W médio: -36857.15575801206

Variáveis criadas:
- K_rel_star
- P_R_star
- eta_hist
- v_hist
- a_hist
- W_hist
- coef_W

 S24-A.2.0 CONCLUÍDO
 Estado reconstruído para S24-A.2


In [ ]:
# ============================================================
# IA-1 — S24-A.2
# BLOCO 1 — PREPARAÇÃO DOS CANDIDATOS DE ESCALA EFETIVA
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S24-A.2")
print(" BLOCO 1 — PREPARAÇÃO κ_eff")
print("="*60)

# ------------------------------------------------------------
# Verificação das variáveis necessárias
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "K_rel_star"
]

faltantes = []

for var in necessarias:
    if var not in globals():
        faltantes.append(var)

if faltantes:
    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute primeiro a reconstrução do estado S23/S24."
    )


# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

eps0 = 1e-12

eta_hist = np.asarray(eta_hist)
v_hist = np.asarray(v_hist)
K = np.asarray(K_rel_star)


print("\nDimensões:")
print("eta:", eta_hist.shape)
print("v:", v_hist.shape)
print("K:", K.shape)


# ------------------------------------------------------------
# Construção de aceleração discreta
# ------------------------------------------------------------

a_hist = np.zeros_like(v_hist)

a_hist[1:] = v_hist[1:] - v_hist[:-1]


# ------------------------------------------------------------
# Energia W reconstruída
# ------------------------------------------------------------

# Termo geométrico projetado
Keta = eta_hist @ K.T

# aproximação do termo W:
# potência relacional entre velocidade e força projetada

W_hist = np.sum(v_hist * Keta, axis=1)


# ------------------------------------------------------------
# Candidato A
# Escala cinemática
# ------------------------------------------------------------

norm_eta = np.linalg.norm(eta_hist, axis=1)

norm_v = np.linalg.norm(v_hist, axis=1)

kappa_A = (
    norm_v**2 /
    (norm_eta**2 + eps0)
)


# ------------------------------------------------------------
# Candidato B
# Escala espectral
# ------------------------------------------------------------

eigvals = np.linalg.eigvalsh(
    (K + K.T)/2
)

spectral_scale = (
    np.sum(eigvals**2)
    /
    (np.trace((K.T @ K)) + eps0)
)

kappa_B = np.ones(len(eta_hist))*spectral_scale


# ------------------------------------------------------------
# Candidato C
# Escala energética W
# ------------------------------------------------------------

kappa_C = (
    W_hist /
    (norm_eta*norm_v + eps0)
)


# ------------------------------------------------------------
# Limpeza numérica
# ------------------------------------------------------------

def limpar(x):
    x = np.asarray(x)
    x[np.isnan(x)] = 0
    x[np.isinf(x)] = 0
    return x


kappa_A = limpar(kappa_A)
kappa_B = limpar(kappa_B)
kappa_C = limpar(kappa_C)


# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------

print("\nEstatísticas κ:")

for nome, k in [
    ("kappa_A", kappa_A),
    ("kappa_B", kappa_B),
    ("kappa_C", kappa_C)
]:

    print("\n", nome)
    print("inicial:", k[0])
    print("final:", k[-1])
    print("mínimo:", np.min(k))
    print("máximo:", np.max(k))
    print("variação média:",
          np.mean(np.abs(np.diff(k))))


# Exportação para próximos blocos

S24_kappa = {
    "A": kappa_A,
    "B": kappa_B,
    "C": kappa_C
}

print("\nVariáveis criadas:")
print("- W_hist")
print("- kappa_A")
print("- kappa_B")
print("- kappa_C")
print("- S24_kappa")

print("\n=== FIM S24-A.2 BLOCO 1 ===")

 IA-1 — S24-A.2
 BLOCO 1 — PREPARAÇÃO κ_eff

Dimensões:
eta: (3000, 36)
v: (3000, 36)
K: (36, 36)

Estatísticas κ:

 kappa_A
inicial: 0.008221683427647805
final: 6.436055317511457
mínimo: 0.008221683427647805
máximo: 6.436055317511457
variação média: 0.0021433256532456847

 kappa_B
inicial: 0.9999999999999944
final: 0.9999999999999944
mínimo: 0.9999999999999944
máximo: 0.9999999999999944
variação média: 0.0

 kappa_C
inicial: -0.6466742956764631
final: -6.474814532642128
mínimo: -6.474814532642128
máximo: -0.6466742956764631
variação média: 0.0019433611993883509

Variáveis criadas:
- W_hist
- kappa_A
- kappa_B
- kappa_C
- S24_kappa

=== FIM S24-A.2 BLOCO 1 ===


In [ ]:

# ============================================================
# IA-1 — S24-A.2
# BLOCO 2 — TESTE DE INFORMAÇÃO RESIDUAL κ_eff
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

print("="*60)
print(" IA-1 — S24-A.2")
print(" BLOCO 2 — INFORMAÇÃO RESIDUAL DA ESCALA")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "W_hist",
    "S24_kappa"
]

faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:
    print("\nAUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute primeiro S24-A.2 BLOCO 1."
    )


# ------------------------------------------------------------
# Construção das variáveis
# ------------------------------------------------------------

eta = np.linalg.norm(
    eta_hist,
    axis=1
)

vel = np.linalg.norm(
    v_hist,
    axis=1
)


# Energia instantânea auxiliar
L_proxy = (
    0.5*vel**2
    +
    0.5*np.sum(
        eta_hist*(eta_hist @ K_rel_star.T),
        axis=1
    )
    +
    W_hist
)


delta_L = np.diff(L_proxy)


# alinhar variáveis
eta = eta[:-1]
vel = vel[:-1]
W = W_hist[:-1]


# ------------------------------------------------------------
# Modelo base
# ------------------------------------------------------------

X_base = np.column_stack([
    eta,
    vel,
    W
])


modelo_base = LinearRegression()

modelo_base.fit(
    X_base,
    delta_L
)

pred_base = modelo_base.predict(
    X_base
)

R2_base = r2_score(
    delta_L,
    pred_base
)


print("\nMODELO BASE")
print("R² base:", R2_base)
print(
    "MSE:",
    mean_squared_error(
        delta_L,
        pred_base
    )
)


# ------------------------------------------------------------
# Modelos com escala
# ------------------------------------------------------------

resultados = {}


for nome, kappa in S24_kappa.items():

    k = kappa[:-1]

    X = np.column_stack([
        eta,
        vel,
        W,
        k
    ])


    modelo = LinearRegression()

    modelo.fit(
        X,
        delta_L
    )

    pred = modelo.predict(X)

    R2 = r2_score(
        delta_L,
        pred
    )


    ganho = R2 - R2_base


    resultados[nome] = {
        "R2": R2,
        "ganho": ganho,
        "coef": modelo.coef_
    }


    print("\n--------------------------------")
    print("Modelo com kappa_"+nome)
    print("R²:", R2)
    print("Delta R²:", ganho)
    print("Coeficiente kappa:",
          modelo.coef_[-1])


# ------------------------------------------------------------
# Regularização nos melhores candidatos
# ------------------------------------------------------------

print("\n================================")
print(" REGULARIZAÇÃO")
print("================================")


for nome, dado in resultados.items():

    k = S24_kappa[nome][:-1]

    X = np.column_stack([
        eta,
        vel,
        W,
        k
    ])


    ridge = Ridge(alpha=1.0)

    ridge.fit(
        X,
        delta_L
    )


    lasso = Lasso(
        alpha=0.001,
        max_iter=100000
    )

    lasso.fit(
        X,
        delta_L
    )


    print("\nKappa", nome)

    print(
        "Ridge beta_kappa:",
        ridge.coef_[-1]
    )

    print(
        "Lasso beta_kappa:",
        lasso.coef_[-1]
    )


# ------------------------------------------------------------
# Ranking final
# ------------------------------------------------------------

print("\n================================")
print(" RANKING ΔR²")
print("================================")


ranking = sorted(
    resultados.items(),
    key=lambda x: x[1]["ganho"],
    reverse=True
)


for nome, dado in ranking:

    print(
        nome,
        "| R² =",
        dado["R2"],
        "| ΔR² =",
        dado["ganho"]
    )


print("\nVariável criada:")
print("- resultados_S24")


resultados_S24 = resultados


print("\n=== FIM S24-A.2 BLOCO 2 ===")

 IA-1 — S24-A.2
 BLOCO 2 — INFORMAÇÃO RESIDUAL DA ESCALA

MODELO BASE
R² base: 0.999999776296169
MSE: 0.05235304960179859

--------------------------------
Modelo com kappa_A
R²: 0.9999999579734543
Delta R²: 1.8167728532691996e-07
Coeficiente kappa: -0.19018105520781212

--------------------------------
Modelo com kappa_B
R²: 0.999999776296169
Delta R²: 0.0
Coeficiente kappa: -1.219817908617584e-32

--------------------------------
Modelo com kappa_C
R²: 0.9999999599442249
Delta R²: 1.8364805587367528e-07
Coeficiente kappa: 0.3206291984098274

 REGULARIZAÇÃO

Kappa A
Ridge beta_kappa: -0.17690295112363194
Lasso beta_kappa: -0.06499810336788854

Kappa B
Ridge beta_kappa: -7.056768690242533e-25
Lasso beta_kappa: -0.0

Kappa C
Ridge beta_kappa: 0.2994569237806368
Lasso beta_kappa: 0.1146230904827089

 RANKING ΔR²
C | R² = 0.9999999599442249 | ΔR² = 1.8364805587367528e-07
A | R² = 0.9999999579734543 | ΔR² = 1.8167728532691996e-07
B | R² = 0.999999776296169 | ΔR² = 0.0

Variável criada:
- r

In [ ]:
# ============================================================
# IA-1 — S24-A.3
# BLOCO 1 — VALIDAÇÃO TEMPORAL OUT-OF-SAMPLE
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-A.3")
print(" BLOCO 1 — VALIDAÇÃO TEMPORAL OUT-OF-SAMPLE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "W_hist",
    "S24_kappa"
]

faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)

if faltantes:

    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute S24-A.2 Blocos anteriores."
    )


# ------------------------------------------------------------
# Construção das variáveis
# ------------------------------------------------------------

eta = np.linalg.norm(
    eta_hist,
    axis=1
)

vel = np.linalg.norm(
    v_hist,
    axis=1
)


L_proxy = (
    0.5*vel**2
    +
    0.5*np.sum(
        eta_hist*(eta_hist @ K_rel_star.T),
        axis=1
    )
    +
    W_hist
)


delta_L = np.diff(L_proxy)


eta = eta[:-1]
vel = vel[:-1]
W = W_hist[:-1]


# ------------------------------------------------------------
# Divisão temporal
# ------------------------------------------------------------

N = len(delta_L)

corte = int(0.7*N)


print("\nTotal de amostras:", N)
print("Treino:", corte)
print("Teste:", N-corte)


# ------------------------------------------------------------
# Modelo base
# ------------------------------------------------------------

X_base = np.column_stack([
    eta,
    vel,
    W
])


X_train = X_base[:corte]
X_test = X_base[corte:]


y_train = delta_L[:corte]
y_test = delta_L[corte:]


modelo_base = LinearRegression()

modelo_base.fit(
    X_train,
    y_train
)


pred_base = modelo_base.predict(
    X_test
)


R2_base_test = r2_score(
    y_test,
    pred_base
)


print("\nMODELO BASE TESTE")
print("R²:", R2_base_test)

print(
    "MSE:",
    mean_squared_error(
        y_test,
        pred_base
    )
)


# ------------------------------------------------------------
# Modelos expandidos
# ------------------------------------------------------------

resultado_OOS = {}


for nome, kappa in S24_kappa.items():

    k = kappa[:-1]


    X = np.column_stack([
        eta,
        vel,
        W,
        k
    ])


    X_train = X[:corte]
    X_test = X[corte:]


    modelo = LinearRegression()


    modelo.fit(
        X_train,
        y_train
    )


    pred = modelo.predict(
        X_test
    )


    R2 = r2_score(
        y_test,
        pred
    )


    ganho = R2 - R2_base_test


    resultado_OOS[nome] = {
        "R2_test": R2,
        "Delta_R2_test": ganho,
        "beta_kappa": modelo.coef_[-1]
    }


    print("\n--------------------------------")
    print("Kappa", nome)
    print("R² teste:", R2)
    print("Delta R² teste:", ganho)
    print(
        "beta kappa:",
        modelo.coef_[-1]
    )


# ------------------------------------------------------------
# Regularização final
# ------------------------------------------------------------

print("\n================================")
print(" LASSO OUT-OF-SAMPLE")
print("================================")


for nome, kappa in S24_kappa.items():

    k = kappa[:-1]

    X = np.column_stack([
        eta,
        vel,
        W,
        k
    ])


    lasso = Lasso(
        alpha=0.001,
        max_iter=100000
    )


    lasso.fit(
        X[:corte],
        y_train
    )


    print(
        "Kappa",
        nome,
        "| beta_lasso =",
        lasso.coef_[-1]
    )


print("\nVariável criada:")
print("- resultado_OOS")


resultado_OOS = resultado_OOS


print("\n=== FIM S24-A.3 BLOCO 1 ===")

 IA-1 — S24-A.3
 BLOCO 1 — VALIDAÇÃO TEMPORAL OUT-OF-SAMPLE

Total de amostras: 2999
Treino: 2099
Teste: 900

MODELO BASE TESTE
R²: 0.9997864312261556
MSE: 109.91490102300702

--------------------------------
Kappa A
R² teste: 0.9998322741659209
Delta R² teste: 4.584293976528375e-05
beta kappa: -0.006583855623964461

--------------------------------
Kappa B
R² teste: 0.9997864312261556
Delta R² teste: 0.0
beta kappa: 3.5577450934266117e-32

--------------------------------
Kappa C
R² teste: 0.999832008600348
Delta R² teste: 4.557737419241814e-05
beta kappa: 0.011093631551272853

 LASSO OUT-OF-SAMPLE
Kappa A | beta_lasso = -0.001528854291962766
Kappa B | beta_lasso = -0.0
Kappa C | beta_lasso = 0.0016694334740172203

Variável criada:
- resultado_OOS

=== FIM S24-A.3 BLOCO 1 ===


In [ ]:
# ============================================================
# IA-1 — S24-B.1
# BLOCO 1 — TRANSFORMAÇÃO DE ESCALA RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-B.1")
print(" BLOCO 1 — TRANSFORMAÇÃO DE ESCALA")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "W_hist",
    "K_rel_star"
]


faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute a reconstrução S24 antes."
    )


# ------------------------------------------------------------
# Escalas de teste
# ------------------------------------------------------------

escalas = np.array([
    1e-2,
    1e-1,
    0.5,
    1.0,
    2.0,
    10.0,
    100.0
])


# estado original

eta0 = eta_hist.copy()
v0 = v_hist.copy()


# ------------------------------------------------------------
# Função auxiliar
# ------------------------------------------------------------

def calcular_observaveis(eta, v):

    energia_cinetica = np.sum(
        v*v,
        axis=1
    )

    energia_geometrica = np.sum(
        eta * (eta @ K_rel_star.T),
        axis=1
    )


    # reconstrução proporcional de W
    # mantém a estrutura já validada em S23/S24

    norma_eta = np.linalg.norm(
        eta,
        axis=1
    )

    norma_v = np.linalg.norm(
        v,
        axis=1
    )


    W = (
        norma_eta *
        norma_v
    )


    L = (
        0.5*energia_cinetica
        +
        0.5*energia_geometrica
        +
        W
    )


    return {
        "W": W,
        "L": L,
        "norma_eta": norma_eta,
        "norma_v": norma_v
    }



# ------------------------------------------------------------
# Estado original
# ------------------------------------------------------------

obs_base = calcular_observaveis(
    eta0,
    v0
)


L_base = obs_base["L"]
W_base = obs_base["W"]


# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

resultado_escala = {}


print("\nEscala | W_final/W0 | L_final/L0 | eta_final/eta0")

for s in escalas:


    eta_s = s * eta0
    v_s = s * v0


    obs = calcular_observaveis(
        eta_s,
        v_s
    )


    W_ratio = (
        obs["W"][-1]
        /
        W_base[-1]
    )


    L_ratio = (
        obs["L"][-1]
        /
        L_base[-1]
    )


    eta_ratio = (
        obs["norma_eta"][-1]
        /
        obs_base["norma_eta"][-1]
    )


    resultado_escala[s] = {
        "W_ratio": W_ratio,
        "L_ratio": L_ratio,
        "eta_ratio": eta_ratio
    }


    print(
        f"{s:7.2e} | "
        f"{W_ratio:12.5e} | "
        f"{L_ratio:12.5e} | "
        f"{eta_ratio:12.5e}"
    )


# ------------------------------------------------------------
# Análise de lei de potência
# ------------------------------------------------------------

logs = np.log(escalas)


log_L = np.log(
    np.abs(
        np.array(
            [
                resultado_escala[s]["L_ratio"]
                for s in escalas
            ]
        )
    )
)


coef = np.polyfit(
    logs,
    log_L,
    1
)


expoente_escala = coef[0]


print("\n================================")
print(" LEI DE ESCALA")
print("================================")

print(
    "Expoente estimado n:",
    expoente_escala
)


print(
    "Modelo aproximado:"
)

print(
    "L(s) ~ s^",
    expoente_escala
)


# ------------------------------------------------------------

print("\nVariáveis criadas:")
print("- resultado_escala")
print("- expoente_escala")


print("\n=== FIM S24-B.1 BLOCO 1 ===")

 IA-1 — S24-B.1
 BLOCO 1 — TRANSFORMAÇÃO DE ESCALA

Escala | W_final/W0 | L_final/L0 | eta_final/eta0
1.00e-02 |  1.00000e-04 |  1.00000e-04 |  1.00000e-02
1.00e-01 |  1.00000e-02 |  1.00000e-02 |  1.00000e-01
5.00e-01 |  2.50000e-01 |  2.50000e-01 |  5.00000e-01
1.00e+00 |  1.00000e+00 |  1.00000e+00 |  1.00000e+00
2.00e+00 |  4.00000e+00 |  4.00000e+00 |  2.00000e+00
1.00e+01 |  1.00000e+02 |  1.00000e+02 |  1.00000e+01
1.00e+02 |  1.00000e+04 |  1.00000e+04 |  1.00000e+02

 LEI DE ESCALA
Expoente estimado n: 2.000000000000001
Modelo aproximado:
L(s) ~ s^ 2.000000000000001

Variáveis criadas:
- resultado_escala
- expoente_escala

=== FIM S24-B.1 BLOCO 1 ===


In [ ]:
# ============================================================
# IA-1 — S24-B.2
# EVOLUÇÃO DINÂMICA SOB TRANSFORMAÇÃO DE ESCALA
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-B.2")
print(" EVOLUÇÃO DINÂMICA SOB TRANSFORMAÇÃO DE ESCALA")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "K_rel_star"
]


faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute S24-B.1 antes."
    )


# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

escalas = np.array([
    1e-2,
    1e-1,
    0.5,
    1.0,
    2.0,
    10.0
])


dt = 1e-3


# ------------------------------------------------------------
# Integrador dinâmico simples
# preserva estrutura do IA-1
# ------------------------------------------------------------

def evoluir(eta0, v0):

    eta = eta0.copy()
    v = v0.copy()


    normas = []

    energias = []


    for k in range(len(eta_hist)):


        aceleracao = (
            -K_rel_star @ eta
        )


        v = v + dt*aceleracao

        eta = eta + dt*v


        norma = np.linalg.norm(
            eta
        )


        energia = (
            0.5*np.dot(v,v)
            +
            0.5*np.dot(
                eta,
                K_rel_star @ eta
            )
        )


        normas.append(norma)
        energias.append(energia)


    return (
        np.array(normas),
        np.array(energias)
    )



# ------------------------------------------------------------
# Referência
# ------------------------------------------------------------

eta_ref = eta_hist[0]
v_ref = v_hist[0]


norma_ref, energia_ref = evoluir(
    eta_ref,
    v_ref
)



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

resultado_dinamico = {}


print(
    "\nEscala | "
    "Razão norma final | "
    "Razão energia final | "
    "Expoente"
)


for s in escalas:


    eta_s = s*eta_ref
    v_s = s*v_ref


    normas, energias = evoluir(
        eta_s,
        v_s
    )


    ratio_norma = (
        normas[-1]
        /
        norma_ref[-1]
    )


    ratio_energia = (
        abs(energias[-1])
        /
        abs(energia_ref[-1])
    )


    # estima expoente:
    # energia ~ s^n

    if s != 1:

        expoente = np.log(
            ratio_energia
        ) / np.log(s)

    else:

        expoente = np.nan


    resultado_dinamico[s] = {

        "norma_ratio": ratio_norma,

        "energia_ratio": ratio_energia,

        "expoente": expoente

    }


    print(
        f"{s:7.2e} | "
        f"{ratio_norma:17.6e} | "
        f"{ratio_energia:18.6e} | "
        f"{expoente}"
    )



# ------------------------------------------------------------
# Verificação de covariância
# ------------------------------------------------------------

validos = [
    resultado_dinamico[s]["expoente"]
    for s in escalas
    if s != 1
]


expoente_medio = np.mean(
    validos
)


desvio_expoente = np.std(
    validos
)


print("\n================================")
print(" COVARIÂNCIA DINÂMICA")
print("================================")


print(
    "Expoente médio:",
    expoente_medio
)


print(
    "Desvio:",
    desvio_expoente
)



print("\nVariável criada:")
print("- resultado_dinamico")


print("\n=== FIM S24-B.2 ===")

 IA-1 — S24-B.2
 EVOLUÇÃO DINÂMICA SOB TRANSFORMAÇÃO DE ESCALA

Escala | Razão norma final | Razão energia final | Expoente
1.00e-02 |      1.000000e-02 |       1.000000e-04 | 1.9999999999999534
1.00e-01 |      1.000000e-01 |       1.000000e-02 | 1.9999999999999483
5.00e-01 |      5.000000e-01 |       2.500000e-01 | 2.0
1.00e+00 |      1.000000e+00 |       1.000000e+00 | nan
2.00e+00 |      2.000000e+00 |       4.000000e+00 | 2.0
1.00e+01 |      1.000000e+01 |       1.000000e+02 | 2.0000000000000133

 COVARIÂNCIA DINÂMICA
Expoente médio: 1.999999999999983
Desvio: 2.6765724930897387e-14

Variável criada:
- resultado_dinamico

=== FIM S24-B.2 ===


In [ ]:
# ============================================================
# IA-1 — S24-B.3
# BUSCA DE ESCALA CARACTERÍSTICA EMERGENTE
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-B.3")
print(" BUSCA DE ESCALA CARACTERÍSTICA EMERGENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "K_rel_star",
    "resultado_dinamico"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:
        print("-", x)

    raise RuntimeError(
        "Execute S24-B.2 antes."
    )


# ------------------------------------------------------------
# Escalas mais densas
# ------------------------------------------------------------

escalas_finas = np.logspace(
    -4,
    4,
    41
)


dt = 1e-3


# ------------------------------------------------------------
# Evolução
# ------------------------------------------------------------

def evoluir_escala(eta0, v0):


    eta = eta0.copy()
    v = v0.copy()


    energia = []


    for k in range(len(eta_hist)):


        a = (
            -K_rel_star @ eta
        )


        v = v + dt*a

        eta = eta + dt*v


        E = (
            0.5*np.dot(v,v)
            +
            0.5*np.dot(
                eta,
                K_rel_star @ eta
            )
        )


        energia.append(E)


    return np.array(energia)



# ------------------------------------------------------------
# Referência
# ------------------------------------------------------------

eta0 = eta_hist[0]
v0 = v_hist[0]


E_ref = evoluir_escala(
    eta0,
    v0
)


E_final_ref = abs(E_ref[-1])


# ------------------------------------------------------------
# Teste
# ------------------------------------------------------------

resultado_quebra_escala = {}


print(
    "\nEscala | Expoente local | Desvio de s²"
)


for s in escalas_finas:


    E_s = evoluir_escala(
        s*eta0,
        s*v0
    )


    ratio = (
        abs(E_s[-1])
        /
        E_final_ref
    )


    if s != 1:


        n_local = (
            np.log(ratio)
            /
            np.log(s)
        )

    else:

        n_local = np.nan



    desvio = (
        n_local - 2
        if not np.isnan(n_local)
        else np.nan
    )


    resultado_quebra_escala[s] = {

        "ratio": ratio,

        "expoente": n_local,

        "desvio": desvio

    }


    if (
        abs(desvio) > 0.05
        and
        not np.isnan(desvio)
    ):

        marcador = "<-- possível quebra"

    else:

        marcador = ""


    print(
        f"{s:10.3e} | "
        f"{n_local:14.8f} | "
        f"{desvio:12.3e} "
        f"{marcador}"
    )



# ------------------------------------------------------------
# Estatística
# ------------------------------------------------------------

expoentes = [

    resultado_quebra_escala[s]["expoente"]

    for s in escalas_finas

    if not np.isnan(
        resultado_quebra_escala[s]["expoente"]
    )

]


media = np.mean(expoentes)

desvio = np.std(expoentes)


print("\n================================")
print(" ANÁLISE DE QUEBRA DE ESCALA")
print("================================")


print(
    "Expoente médio:",
    media
)

print(
    "Desvio:",
    desvio
)


if desvio < 0.05:

    print(
        "\nCONCLUSÃO:"
    )

    print(
        "Nenhuma escala característica detectada."
    )

    print(
        "Simetria de escala preservada."
    )


else:

    print(
        "\nCONCLUSÃO:"
    )

    print(
        "Possível quebra espontânea de escala."
    )


print("\nVariável criada:")
print("- resultado_quebra_escala")


print("\n=== FIM S24-B.3 ===")

 IA-1 — S24-B.3
 BUSCA DE ESCALA CARACTERÍSTICA EMERGENTE

Escala | Expoente local | Desvio de s²
 1.000e-04 |     2.00000000 |   -3.242e-14 
 1.585e-04 |     2.00000000 |   -1.510e-14 
 2.512e-04 |     2.00000000 |   -5.107e-15 
 3.981e-04 |     2.00000000 |    8.882e-15 
 6.310e-04 |     2.00000000 |   -2.265e-14 
 1.000e-03 |     2.00000000 |   -2.753e-14 
 1.585e-03 |     2.00000000 |    1.421e-14 
 2.512e-03 |     2.00000000 |   -3.730e-14 
 3.981e-03 |     2.00000000 |   -3.109e-14 
 6.310e-03 |     2.00000000 |   -3.797e-14 
 1.000e-02 |     2.00000000 |   -4.663e-14 
 1.585e-02 |     2.00000000 |    3.997e-15 
 2.512e-02 |     2.00000000 |    2.709e-14 
 3.981e-02 |     2.00000000 |   -3.109e-15 
 6.310e-02 |     2.00000000 |   -2.820e-14 
 1.000e-01 |     2.00000000 |   -5.174e-14 
 1.585e-01 |     2.00000000 |    7.194e-14 
 2.512e-01 |     2.00000000 |    1.332e-14 
 3.981e-01 |     2.00000000 |   -2.986e-13 
 6.310e-01 |     2.00000000 |    5.773e-15 
 1.000e+00 |          

In [ ]:
# ============================================================
# IA-1 — S24-C.1
# AUDITORIA DE INVARIANTES DO FLUXO RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-C.1")
print(" AUDITORIA DE INVARIANTES DO FLUXO RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "W_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute S24-B antes."
    )


# ------------------------------------------------------------
# Construção dos invariantes candidatos
# ------------------------------------------------------------

n = len(eta_hist)


I_energia = []

I_angular = []

I_escala = []

I_curvatura = []



for k in range(n):


    eta = eta_hist[k]

    v = v_hist[k]

    a = a_hist[k]


    # Energia quadrática

    E = (
        0.5*np.dot(v,v)
        +
        0.5*np.dot(
            eta,
            K_rel_star @ eta
        )
    )


    I_energia.append(E)



    # Produto tipo momento angular relacional

    M = np.linalg.norm(
        np.cross(
            eta.reshape(3,-1).mean(axis=1),
            v.reshape(3,-1).mean(axis=1)
        )
    )


    I_angular.append(M)



    # Invariante de escala normalizado

    S = (

        np.dot(eta,eta)

        /
        (
            np.dot(v,v)
            +
            1e-12
        )

    )


    I_escala.append(S)



    # Curvatura efetiva

    C = np.linalg.norm(
        K_rel_star @ eta
    ) / (
        np.linalg.norm(eta)
        +
        1e-12
    )


    I_curvatura.append(C)



I_energia = np.array(I_energia)

I_angular = np.array(I_angular)

I_escala = np.array(I_escala)

I_curvatura = np.array(I_curvatura)



# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

def analisar(nome, serie):


    variacao = np.abs(
        np.diff(serie)
    )


    rel = (

        np.mean(variacao)

        /

        (
            np.abs(
                np.mean(serie)
            )
            +
            1e-12
        )

    )


    print("\n", nome)

    print(
        "Inicial:",
        serie[0]
    )

    print(
        "Final:",
        serie[-1]
    )

    print(
        "Variação relativa média:",
        rel
    )

    print(
        "Desvio:",
        np.std(serie)
    )



print("\n================================")
print(" TESTE DE INVARIANTES")
print("================================")


analisar(
    "Energia relacional",
    I_energia
)


analisar(
    "Momento angular relacional",
    I_angular
)


analisar(
    "Razão de escala",
    I_escala
)


analisar(
    "Curvatura efetiva",
    I_curvatura
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES")
print("================================")


for nome, serie in [

    ("Energia", I_energia),

    ("Angular", I_angular),

    ("Escala", I_escala),

    ("Curvatura", I_curvatura)

]:


    corr = np.corrcoef(
        serie,
        W_hist
    )[0,1]


    print(
        nome,
        "vs W:",
        corr
    )



resultado_invariantes = {

    "energia": I_energia,

    "angular": I_angular,

    "escala": I_escala,

    "curvatura": I_curvatura

}



print("\nVariável criada:")
print("- resultado_invariantes")


print("\n=== FIM S24-C.1 ===")

 IA-1 — S24-C.1
 AUDITORIA DE INVARIANTES DO FLUXO RELACIONAL

 TESTE DE INVARIANTES

 Energia relacional
Inicial: -0.12828924816673368
Final: -0.12440983539272565
Variação relativa média: 1.0238249329309607e-05
Desvio: 0.001120259265639065

 Momento angular relacional
Inicial: 0.00018473322414988476
Final: 9.265306006730915
Variação relativa média: 0.004359094920710992
Desvio: 1.6579098125126548

 Razão de escala
Inicial: 121.62959185125149
Final: 0.15537467449652942
Variação relativa média: 0.021235820177973975
Desvio: 9.782752173618347

 Curvatura efetiva
Inicial: 2.7832538387739083
Final: 6.486055669628044
Variação relativa média: 0.00023011060185691294
Desvio: 1.1843489934568465

 CORRELAÇÕES
Energia vs W: -0.5835806948515435
Angular vs W: -0.9972987593089198
Escala vs W: 0.06948047422837081
Curvatura vs W: -0.35010817103865693

Variável criada:
- resultado_invariantes

=== FIM S24-C.1 ===


In [ ]:
# ============================================================
# IA-1 — S24-C.2
# RELAÇÃO ENERGIA–CURVATURA EMERGENTE
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-C.2")
print(" RELAÇÃO ENERGIA–CURVATURA EMERGENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_invariantes",
    "eta_hist",
    "v_hist",
    "K_rel_star",
    "W_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute S24-C.1 antes."
    )


# ------------------------------------------------------------
# Reconstrução das séries
# ------------------------------------------------------------

energia = resultado_invariantes["energia"]

curvatura = resultado_invariantes["curvatura"]


eta_norm = np.linalg.norm(
    eta_hist,
    axis=1
)


v_norm = np.linalg.norm(
    v_hist,
    axis=1
)



# ------------------------------------------------------------
# Modelo 1:
# Curvatura explicada apenas pela energia
# ------------------------------------------------------------

X_E = energia.reshape(-1,1)

y = curvatura



modelo_E = LinearRegression()

modelo_E.fit(
    X_E,
    y
)


pred_E = modelo_E.predict(
    X_E
)


R2_E = r2_score(
    y,
    pred_E
)



print("\n================================")
print(" MODELO ENERGIA")
print("================================")


print(
    "R² energia:",
    R2_E
)

print(
    "Coeficiente:",
    modelo_E.coef_[0]
)



# ------------------------------------------------------------
# Modelo 2:
# Energia + transferência W
# ------------------------------------------------------------

X_EW = np.column_stack(
    [
        energia,
        W_hist
    ]
)



modelo_EW = Ridge(
    alpha=1e-8
)


modelo_EW.fit(
    X_EW,
    y
)


pred_EW = modelo_EW.predict(
    X_EW
)


R2_EW = r2_score(
    y,
    pred_EW
)



print("\n================================")
print(" MODELO ENERGIA + W")
print("================================")


print(
    "R² energia+W:",
    R2_EW
)


print(
    "Coef energia:",
    modelo_EW.coef_[0]
)

print(
    "Coef W:",
    modelo_EW.coef_[1]
)



# ------------------------------------------------------------
# Modelo 3:
# Geometria completa
# ------------------------------------------------------------

X_full = np.column_stack(
    [
        energia,
        W_hist,
        eta_norm,
        v_norm
    ]
)


modelo_full = Ridge(
    alpha=1e-8
)


modelo_full.fit(
    X_full,
    y
)


pred_full = modelo_full.predict(
    X_full
)


R2_full = r2_score(
    y,
    pred_full
)



print("\n================================")
print(" MODELO COMPLETO")
print("================================")


print(
    "R² completo:",
    R2_full
)



# ------------------------------------------------------------
# Ganho explicativo
# ------------------------------------------------------------

print("\n================================")
print(" GANHO DE INFORMAÇÃO")
print("================================")


print(
    "Energia -> Curvatura:",
    R2_E
)


print(
    "Energia+W -> Curvatura:",
    R2_EW
)


print(
    "Completo:",
    R2_full
)



delta_W = R2_EW - R2_E

delta_full = R2_full - R2_EW


print(
    "\nDelta R²(W):",
    delta_W
)


print(
    "Delta R² completo:",
    delta_full
)



# ------------------------------------------------------------
# Correlação direta
# ------------------------------------------------------------

corr_E = np.corrcoef(
    energia,
    curvatura
)[0,1]


corr_W = np.corrcoef(
    W_hist,
    curvatura
)[0,1]


print("\n================================")
print(" CORRELAÇÕES")
print("================================")


print(
    "Energia vs Curvatura:",
    corr_E
)


print(
    "W vs Curvatura:",
    corr_W
)



resultado_energia_curvatura = {

    "R2_E": R2_E,

    "R2_EW": R2_EW,

    "R2_full": R2_full,

    "delta_W": delta_W,

    "delta_full": delta_full,

    "corr_E": corr_E,

    "corr_W": corr_W

}


print("\nVariável criada:")
print("- resultado_energia_curvatura")


print("\n=== FIM S24-C.2 ===")

 IA-1 — S24-C.2
 RELAÇÃO ENERGIA–CURVATURA EMERGENTE

 MODELO ENERGIA
R² energia: 0.8270426458880488
Coeficiente: 961.4464556687794

 MODELO ENERGIA + W
R² energia+W: 0.8765099095346025
Coef energia: 1130.4219719714574
Coef W: 3.417199305765673e-06

 MODELO COMPLETO
R² completo: 0.9897912995106973

 GANHO DE INFORMAÇÃO
Energia -> Curvatura: 0.8270426458880488
Energia+W -> Curvatura: 0.8765099095346025
Completo: 0.9897912995106973

Delta R²(W): 0.04946726364655374
Delta R² completo: 0.11328138997609483

 CORRELAÇÕES
Energia vs Curvatura: 0.9094188506337709
W vs Curvatura: -0.35010817103865693

Variável criada:
- resultado_energia_curvatura

=== FIM S24-C.2 ===


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.18443e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.33309e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


In [ ]:
# ============================================================
# IA-1 — S24-C.3
# PREDIÇÃO CAUSAL DA CURVATURA EMERGENTE
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-C.3")
print(" PREDIÇÃO CAUSAL DA CURVATURA EMERGENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_invariantes",
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "W_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute S24-C.2 antes."
    )


# ------------------------------------------------------------
# Série alvo:
# curvatura no passo seguinte
# ------------------------------------------------------------

curvatura = resultado_invariantes["curvatura"]


y = curvatura[1:]



# ------------------------------------------------------------
# Construção dos preditores no passo k
# ------------------------------------------------------------

energia = resultado_invariantes["energia"][:-1]

W = W_hist[:-1]


eta_norm = np.linalg.norm(
    eta_hist[:-1],
    axis=1
)


v_norm = np.linalg.norm(
    v_hist[:-1],
    axis=1
)


a_norm = np.linalg.norm(
    a_hist[:-1],
    axis=1
)



X = np.column_stack(
    [
        energia,
        W,
        eta_norm,
        v_norm,
        a_norm
    ]
)



# ------------------------------------------------------------
# Divisão temporal
# ------------------------------------------------------------

N = len(y)

corte = int(
    0.7*N
)


X_train = X[:corte]

X_test = X[corte:]


y_train = y[:corte]

y_test = y[corte:]



print("\nAmostras:")
print("Treino:", len(y_train))
print("Teste :", len(y_test))



# ------------------------------------------------------------
# Modelo regularizado
# ------------------------------------------------------------

modelo = Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)



modelo.fit(
    X_train,
    y_train
)



pred = modelo.predict(
    X_test
)



R2 = r2_score(
    y_test,
    pred
)


MSE = mean_squared_error(
    y_test,
    pred
)



print("\n================================")
print(" PREDIÇÃO OUT-OF-SAMPLE")
print("================================")


print(
    "R² teste:",
    R2
)


print(
    "MSE teste:",
    MSE
)



# ------------------------------------------------------------
# Modelo reduzido:
# energia apenas
# ------------------------------------------------------------

X_E = energia.reshape(-1,1)


modelo_E = Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)


modelo_E.fit(
    X_E[:corte],
    y_train
)


pred_E = modelo_E.predict(
    X_E[corte:]
)



R2_E = r2_score(
    y_test,
    pred_E
)



print("\n================================")
print(" COMPARAÇÃO")
print("================================")


print(
    "Energia apenas:",
    R2_E
)


print(
    "Estado completo:",
    R2
)


print(
    "Ganho:",
    R2-R2_E
)



# ------------------------------------------------------------
# Coeficientes
# ------------------------------------------------------------

coef = modelo.named_steps["ridge"].coef_


print("\n================================")
print(" PESOS NORMALIZADOS")
print("================================")


nomes = [
    "Energia",
    "W",
    "norma_eta",
    "norma_v",
    "norma_a"
]


for n,c in zip(nomes,coef):

    print(
        n,
        ":",
        c
    )



resultado_predicao_curvatura = {

    "R2_teste": R2,

    "R2_energia": R2_E,

    "ganho": R2-R2_E,

    "MSE": MSE,

    "coeficientes": coef

}



print("\nVariável criada:")
print("- resultado_predicao_curvatura")


print("\n=== FIM S24-C.3 ===")

 IA-1 — S24-C.3
 PREDIÇÃO CAUSAL DA CURVATURA EMERGENTE

Amostras:
Treino: 2099
Teste : 900

 PREDIÇÃO OUT-OF-SAMPLE
R² teste: -14564.350829651172
MSE teste: 88.19488662059631

 COMPARAÇÃO
Energia apenas: -333.2786225134288
Estado completo: -14564.350829651172
Ganho: -14231.072207137742

 PESOS NORMALIZADOS
Energia : 1.6164810776710283
W : -0.00026494120259320175
norma_eta : -27.97456732141501
norma_v : -7.870583087581052
norma_a : 35.26503019242682

Variável criada:
- resultado_predicao_curvatura

=== FIM S24-C.3 ===


In [ ]:
# ============================================================
# IA-1 — S24-C.4
# MEMÓRIA RELACIONAL E CURVATURA EMERGENTE
# ============================================================

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-C.4")
print(" MEMÓRIA RELACIONAL E CURVATURA EMERGENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_invariantes",
    "eta_hist",
    "v_hist",
    "a_hist",
    "W_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute S24-C.3 antes."
    )


# ------------------------------------------------------------
# Séries básicas
# ------------------------------------------------------------

curvatura = resultado_invariantes["curvatura"]

energia = resultado_invariantes["energia"]

W = W_hist


eta_norm = np.linalg.norm(
    eta_hist,
    axis=1
)


v_norm = np.linalg.norm(
    v_hist,
    axis=1
)


a_norm = np.linalg.norm(
    a_hist,
    axis=1
)



base = np.column_stack(
    [
        energia,
        W,
        eta_norm,
        v_norm,
        a_norm
    ]
)



# ------------------------------------------------------------
# Construção dos vetores com memória
# ------------------------------------------------------------

def criar_memoria(X, y, memoria):

    X_new = []
    y_new = []

    for i in range(memoria, len(y)):

        bloco = X[
            i-memoria:i+1
        ]

        X_new.append(
            bloco.flatten()
        )

        y_new.append(
            y[i]
        )

    return np.array(X_new), np.array(y_new)



# ------------------------------------------------------------
# Teste de diferentes profundidades
# ------------------------------------------------------------

memorias = [
    0,
    1,
    2,
    5,
    10,
    20
]


resultados = []


print("\n================================")
print(" TESTE DE MEMÓRIA")
print("================================")


for m in memorias:


    X, y = criar_memoria(
        base,
        curvatura,
        m
    )


    corte = int(
        0.7*len(y)
    )


    X_train = X[:corte]

    X_test = X[corte:]


    y_train = y[:corte]

    y_test = y[corte:]



    modelo = Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(alpha=1e-2)
            )
        ]
    )


    modelo.fit(
        X_train,
        y_train
    )


    pred = modelo.predict(
        X_test
    )


    R2 = r2_score(
        y_test,
        pred
    )


    mse = mean_squared_error(
        y_test,
        pred
    )


    resultados.append(
        [
            m,
            R2,
            mse
        ]
    )


    print(
        f"Memória {m:2d} | R² {R2:.6e} | MSE {mse:.6e}"
    )



# ------------------------------------------------------------
# Melhor memória
# ------------------------------------------------------------

melhor = max(
    resultados,
    key=lambda x:x[1]
)


print("\n================================")
print(" MELHOR REGIME")
print("================================")


print(
    "Memória ótima:",
    melhor[0]
)

print(
    "R²:",
    melhor[1]
)

print(
    "MSE:",
    melhor[2]
)



resultado_memoria_curvatura = {

    "resultados": resultados,

    "melhor_memoria": melhor[0],

    "melhor_R2": melhor[1],

    "melhor_MSE": melhor[2]

}



print("\nVariável criada:")
print("- resultado_memoria_curvatura")


print("\n=== FIM S24-C.4 ===")

 IA-1 — S24-C.4
 MEMÓRIA RELACIONAL E CURVATURA EMERGENTE

 TESTE DE MEMÓRIA
Memória  0 | R² -5.539036e+05 | MSE 3.353956e+03
Memória  1 | R² -2.070875e+05 | MSE 1.253945e+03
Memória  2 | R² -6.137860e+04 | MSE 3.716606e+02
Memória  5 | R² -1.511633e+04 | MSE 9.128674e+01
Memória 10 | R² -1.577064e+05 | MSE 9.471132e+02
Memória 20 | R² -3.623567e+05 | MSE 2.158271e+03

 MELHOR REGIME
Memória ótima: 5
R²: -15116.331416650803
MSE: 91.28674038594991

Variável criada:
- resultado_memoria_curvatura

=== FIM S24-C.4 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.1
# MÉTRICA TEMPORAL EMERGENTE RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-D.1")
print(" MÉTRICA TEMPORAL EMERGENTE RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "W_hist",
    "resultado_invariantes"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute os blocos S23/S24 anteriores antes."
    )


# ------------------------------------------------------------
# Construção do tempo interno
# ------------------------------------------------------------

eps = 1e-12


deslocamento = np.linalg.norm(
    np.diff(eta_hist, axis=0),
    axis=1
)


velocidade = np.linalg.norm(
    v_hist[:-1],
    axis=1
)


delta_tau = (
    deslocamento /
    (velocidade + eps)
)



# ------------------------------------------------------------
# Variáveis geométricas
# ------------------------------------------------------------

energia = resultado_invariantes["energia"][:-1]

curvatura = resultado_invariantes["curvatura"][:-1]

W = W_hist[:-1]


eta_norm = np.linalg.norm(
    eta_hist[:-1],
    axis=1
)



# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

print("\n================================")
print(" MÉTRICA TEMPORAL")
print("================================")


print(
    "tau inicial:",
    delta_tau[0]
)

print(
    "tau final:",
    delta_tau[-1]
)

print(
    "tau médio:",
    np.mean(delta_tau)
)

print(
    "desvio:",
    np.std(delta_tau)
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES")
print("================================")


corr_energia = np.corrcoef(
    delta_tau,
    energia
)[0,1]


corr_curvatura = np.corrcoef(
    delta_tau,
    curvatura
)[0,1]


corr_W = np.corrcoef(
    delta_tau,
    W
)[0,1]


corr_eta = np.corrcoef(
    delta_tau,
    eta_norm
)[0,1]



print(
    "tau vs Energia:",
    corr_energia
)

print(
    "tau vs Curvatura:",
    corr_curvatura
)

print(
    "tau vs W:",
    corr_W
)

print(
    "tau vs norma_eta:",
    corr_eta
)



# ------------------------------------------------------------
# Teste de escala
# ------------------------------------------------------------

log_tau = np.log(
    np.abs(delta_tau)+eps
)


log_curv = np.log(
    np.abs(curvatura)+eps
)


coef = np.polyfit(
    log_curv,
    log_tau,
    1
)


print("\n================================")
print(" LEI DE ESCALA TEMPORAL")
print("================================")


print(
    "Expoente tau-curvatura:",
    coef[0]
)


# ------------------------------------------------------------
# Salvamento
# ------------------------------------------------------------

resultado_tempo_emergente = {

    "delta_tau": delta_tau,

    "corr_energia": corr_energia,

    "corr_curvatura": corr_curvatura,

    "corr_W": corr_W,

    "corr_eta": corr_eta,

    "expoente_tau_curvatura": coef[0]

}


print("\nVariável criada:")
print("- resultado_tempo_emergente")


print("\n=== FIM S24-D.1 ===")

 IA-1 — S24-D.1
 MÉTRICA TEMPORAL EMERGENTE RELACIONAL

 MÉTRICA TEMPORAL
tau inicial: 0.0009999999999808808
tau final: 0.0009999999999999985
tau médio: 0.0009999999999992038
desvio: 2.140401525674035e-15

 CORRELAÇÕES
tau vs Energia: 0.5282793180503378
tau vs Curvatura: 0.6579579502483843
tau vs W: -0.14373528086466011
tau vs norma_eta: 0.21769907519395892

 LEI DE ESCALA TEMPORAL
Expoente tau-curvatura: 5.697182691492687e-12

Variável criada:
- resultado_tempo_emergente

=== FIM S24-D.1 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.2
# TEMPO RELACIONAL POR SATURAÇÃO CAUSAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-D.2")
print(" TEMPO RELACIONAL POR SATURAÇÃO CAUSAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "resultado_invariantes",
    "W_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute os blocos S24 anteriores."
    )


# ------------------------------------------------------------
# Construção dos candidatos relacionais
# ------------------------------------------------------------

eps = 1e-12


# profundidade relacional aproximada
# acúmulo de transformação do estado

depth_rel = np.cumsum(
    np.linalg.norm(
        np.diff(
            eta_hist,
            axis=0
        ),
        axis=1
    )
)


depth_rel = np.insert(
    depth_rel,
    0,
    0
)



# capacidade de propagação local

propagacao = (
    np.linalg.norm(v_hist, axis=1)
    +
    eps
)


# saturação causal aproximada

tau_rel_1 = (
    depth_rel /
    propagacao
)



# alternativa usando aceleração

capacidade = (
    np.linalg.norm(a_hist, axis=1)
    +
    eps
)


tau_rel_2 = (
    depth_rel /
    capacidade
)



# alternativa energética

energia = resultado_invariantes["energia"]

curvatura = resultado_invariantes["curvatura"]



# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

print("\n================================")
print(" CANDIDATOS TEMPORAIS")
print("================================")


for nome, serie in [
    ("tau_depth_velocity", tau_rel_1),
    ("tau_depth_acceleration", tau_rel_2)
]:

    print("\n", nome)

    print(
        "Inicial:",
        serie[0]
    )

    print(
        "Final:",
        serie[-1]
    )

    print(
        "Média:",
        np.mean(serie)
    )

    print(
        "Desvio:",
        np.std(serie)
    )



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES RELACIONAIS")
print("================================")


for nome, serie in [
    ("tau_v vs energia",
     tau_rel_1),

    ("tau_v vs curvatura",
     tau_rel_1),

    ("tau_a vs energia",
     tau_rel_2),

    ("tau_a vs curvatura",
     tau_rel_2)
]:

    if "energia" in nome:

        alvo = energia

    else:

        alvo = curvatura


    corr = np.corrcoef(
        serie,
        alvo
    )[0,1]


    print(
        nome,
        ":",
        corr
    )



# ------------------------------------------------------------
# Teste de escala
# ------------------------------------------------------------

def expoente(x,y):

    mask = (
        np.abs(x)>eps
    ) & (
        np.abs(y)>eps
    )

    if np.sum(mask)<10:

        return np.nan

    coef = np.polyfit(
        np.log(np.abs(x[mask])),
        np.log(np.abs(y[mask])),
        1
    )

    return coef[0]



print("\n================================")
print(" LEIS DE ESCALA")
print("================================")


print(
    "tau_v-curvatura:",
    expoente(
        tau_rel_1,
        curvatura
    )
)


print(
    "tau_a-curvatura:",
    expoente(
        tau_rel_2,
        curvatura
    )
)



# ------------------------------------------------------------
# Salvamento
# ------------------------------------------------------------

resultado_tempo_relacional = {

    "tau_depth_velocity": tau_rel_1,

    "tau_depth_acceleration": tau_rel_2

}


print("\nVariável criada:")
print("- resultado_tempo_relacional")


print("\n=== FIM S24-D.2 ===")

 IA-1 — S24-D.2
 TEMPO RELACIONAL POR SATURAÇÃO CAUSAL

 CANDIDATOS TEMPORAIS

 tau_depth_velocity
Inicial: 0.0
Final: 0.3947647080834435
Média: 0.3358021886769293
Desvio: 0.10320650351860405

 tau_depth_acceleration
Inicial: 0.0
Final: 0.1548038773455199
Média: 0.13185317043025468
Desvio: 0.04670024562224315

 CORRELAÇÕES RELACIONAIS
tau_v vs energia : -0.22777434186501327
tau_v vs curvatura : 0.9615235186875263
tau_a vs energia : -0.19742265154896005
tau_a vs curvatura : 0.9497930026790756

 LEIS DE ESCALA
tau_v-curvatura: 0.3738692388508619
tau_a-curvatura: 0.2174823443536342

Variável criada:
- resultado_tempo_relacional

=== FIM S24-D.2 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.3
# CAUSALIDADE DIRECIONAL TEMPORAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-D.3")
print(" CAUSALIDADE DIRECIONAL TEMPORAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_tempo_relacional",
    "resultado_invariantes"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-D.2 antes."
    )


# ------------------------------------------------------------
# Séries
# ------------------------------------------------------------

tau = resultado_tempo_relacional[
    "tau_depth_velocity"
]


curvatura = resultado_invariantes[
    "curvatura"
]


# remoção inicial nula

mask = np.isfinite(tau) & np.isfinite(curvatura)

tau = tau[mask]

curvatura = curvatura[mask]



# ------------------------------------------------------------
# Função de predição defasada
# ------------------------------------------------------------

def teste_direcao(
    causa,
    efeito,
    atraso
):

    X = []
    y = []


    for i in range(
        atraso,
        len(efeito)
    ):

        X.append(
            [causa[i-atraso]]
        )

        y.append(
            efeito[i]
        )


    X = np.array(X)

    y = np.array(y)


    corte = int(
        0.7*len(y)
    )


    modelo = Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(alpha=1e-3)
            )
        ]
    )


    modelo.fit(
        X[:corte],
        y[:corte]
    )


    pred = modelo.predict(
        X[corte:]
    )


    return (
        r2_score(
            y[corte:],
            pred
        ),

        mean_squared_error(
            y[corte:],
            pred
        )
    )



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

atrasos = [
    1,
    2,
    5,
    10,
    20,
    50
]


resultados = []


print("\n================================")
print(" TESTES DE DIREÇÃO CAUSAL")
print("================================")


for d in atrasos:


    r_k_tau, mse1 = teste_direcao(
        curvatura,
        tau,
        d
    )


    r_tau_k, mse2 = teste_direcao(
        tau,
        curvatura,
        d
    )


    resultados.append(
        [
            d,
            r_k_tau,
            r_tau_k
        ]
    )


    print(
        f"Atraso {d:2d} | "
        f"K→tau R²={r_k_tau:.6f} | "
        f"tau→K R²={r_tau_k:.6f}"
    )



# ------------------------------------------------------------
# Melhor direção
# ------------------------------------------------------------

melhor_k_tau = max(
    resultados,
    key=lambda x:x[1]
)


melhor_tau_k = max(
    resultados,
    key=lambda x:x[2]
)


print("\n================================")
print(" MELHOR RESULTADO")
print("================================")


print(
    "Melhor K → tau:",
    melhor_k_tau
)


print(
    "Melhor tau → K:",
    melhor_tau_k
)


resultado_causalidade_temporal = {

    "tabela": resultados,

    "melhor_K_tau": melhor_k_tau,

    "melhor_tau_K": melhor_tau_k

}


print("\nVariável criada:")
print("- resultado_causalidade_temporal")


print("\n=== FIM S24-D.3 ===")

 IA-1 — S24-D.3
 CAUSALIDADE DIRECIONAL TEMPORAL

 TESTES DE DIREÇÃO CAUSAL
Atraso  1 | K→tau R²=-626.074044 | tau→K R²=-45.161461
Atraso  2 | K→tau R²=-625.153065 | tau→K R²=-44.965002
Atraso  5 | K→tau R²=-622.838285 | tau→K R²=-44.486965
Atraso 10 | K→tau R²=-619.219289 | tau→K R²=-43.734089
Atraso 20 | K→tau R²=-611.844613 | tau→K R²=-42.153676
Atraso 50 | K→tau R²=-590.286740 | tau→K R²=-37.676845

 MELHOR RESULTADO
Melhor K → tau: [50, -590.2867401380115, -37.67684465325605]
Melhor tau → K: [50, -590.2867401380115, -37.67684465325605]

Variável criada:
- resultado_causalidade_temporal

=== FIM S24-D.3 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.4
# RECONSTRUÇÃO DO CAMPO LATENTE RELACIONAL
# ============================================================

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-D.4")
print(" RECONSTRUÇÃO DO CAMPO LATENTE RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_tempo_relacional",
    "resultado_invariantes",
    "W_hist",
    "eta_hist",
    "v_hist"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-D.2 antes."
    )


# ------------------------------------------------------------
# Construção do espaço relacional
# ------------------------------------------------------------

tau = resultado_tempo_relacional[
    "tau_depth_velocity"
]


curvatura = resultado_invariantes[
    "curvatura"
]


W = W_hist


eta_norm = np.linalg.norm(
    eta_hist,
    axis=1
)


v_norm = np.linalg.norm(
    v_hist,
    axis=1
)



# Ajuste de tamanho

N = min(
    len(tau),
    len(curvatura),
    len(W),
    len(eta_norm),
    len(v_norm)
)


X = np.column_stack(
    [
        tau[:N],
        curvatura[:N],
        W[:N],
        eta_norm[:N],
        v_norm[:N]
    ]
)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

Xn = StandardScaler().fit_transform(X)



# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

print("\n================================")
print(" ANÁLISE LATENTE")
print("================================")


pca = PCA()

Z = pca.fit_transform(Xn)


variancia = pca.explained_variance_ratio_


for i,v in enumerate(variancia):

    print(
        f"Modo {i}: {v:.6f}"
    )


print(
    "\nVariância acumulada 1º modo:",
    variancia[0]
)


print(
    "Variância acumulada 2 modos:",
    np.sum(variancia[:2])
)



# ------------------------------------------------------------
# Campo latente candidato
# ------------------------------------------------------------

Phi_rel = Z[:,0]



# ------------------------------------------------------------
# Capacidade de reconstrução
# ------------------------------------------------------------

print("\n================================")
print(" RECONSTRUÇÃO")
print("================================")


def reconstrucao(alvo):

    corte = int(
        0.7*len(Phi_rel)
    )


    modelo = Ridge(
        alpha=1e-3
    )


    modelo.fit(
        Phi_rel[:corte,None],
        alvo[:corte]
    )


    pred = modelo.predict(
        Phi_rel[corte:,None]
    )


    return r2_score(
        alvo[corte:],
        pred
    )



r_tau = reconstrucao(tau[:len(Phi_rel)])

r_curv = reconstrucao(curvatura[:len(Phi_rel)])



print(
    "R² Phi -> tau:",
    r_tau
)


print(
    "R² Phi -> curvatura:",
    r_curv
)



# ------------------------------------------------------------
# Salvamento
# ------------------------------------------------------------

resultado_campo_latente = {

    "Phi_rel": Phi_rel,

    "variancia_PCA": variancia,

    "R2_Phi_tau": r_tau,

    "R2_Phi_curvatura": r_curv

}


print("\nVariável criada:")
print("- resultado_campo_latente")


print("\n=== FIM S24-D.4 ===")

 IA-1 — S24-D.4
 RECONSTRUÇÃO DO CAMPO LATENTE RELACIONAL

 ANÁLISE LATENTE
Modo 0: 0.700893
Modo 1: 0.282087
Modo 2: 0.013768
Modo 3: 0.003252
Modo 4: 0.000000

Variância acumulada 1º modo: 0.700892914367097
Variância acumulada 2 modos: 0.982979879642197

 RECONSTRUÇÃO
R² Phi -> tau: -50778.43628645599
R² Phi -> curvatura: -2407.0471408254443

Variável criada:
- resultado_campo_latente

=== FIM S24-D.4 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.5
# CAMPO LATENTE BIDIMENSIONAL RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-D.5")
print(" CAMPO LATENTE BIDIMENSIONAL RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_campo_latente",
    "resultado_tempo_relacional",
    "resultado_invariantes"
]


faltantes = []

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-", x)

    raise RuntimeError(
        "Execute S24-D.4 antes."
    )



# ------------------------------------------------------------
# Extração dos modos latentes
# ------------------------------------------------------------

# reconstruir PCA sobre os dados originais

Phi = resultado_campo_latente["Phi_rel"]

variancia = resultado_campo_latente[
    "variancia_PCA"
]


# tentar recuperar segundo modo
if "Z" in globals():

    Phi2 = Z[:,1]

else:

    raise RuntimeError(
        "Matriz PCA Z não encontrada."
    )



N = min(
    len(Phi),
    len(Phi2),
    len(resultado_tempo_relacional["tau_depth_velocity"]),
    len(resultado_invariantes["curvatura"])
)



X_latente = np.column_stack(
    [
        Phi[:N],
        Phi2[:N]
    ]
)



tau = resultado_tempo_relacional[
    "tau_depth_velocity"
][:N]


curvatura = resultado_invariantes[
    "curvatura"
][:N]



# ------------------------------------------------------------
# Função de teste
# ------------------------------------------------------------

def testar(alvo):

    corte = int(
        0.7*N
    )


    modelo = Ridge(
        alpha=1e-3
    )


    modelo.fit(
        X_latente[:corte],
        alvo[:corte]
    )


    pred = modelo.predict(
        X_latente[corte:]
    )


    return (
        r2_score(
            alvo[corte:],
            pred
        ),

        mean_squared_error(
            alvo[corte:],
            pred
        ),

        modelo.coef_
    )



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

print("\n================================")
print(" RECONSTRUÇÃO BIDIMENSIONAL")
print("================================")


r_tau = testar(tau)

r_curv = testar(curvatura)



print("\nPhi(0,1) -> tau")

print(
    "R²:",
    r_tau[0]
)

print(
    "MSE:",
    r_tau[1]
)

print(
    "Pesos:",
    r_tau[2]
)



print("\nPhi(0,1) -> curvatura")

print(
    "R²:",
    r_curv[0]
)

print(
    "MSE:",
    r_curv[1]
)

print(
    "Pesos:",
    r_curv[2]
)



# ------------------------------------------------------------
# Comparação
# ------------------------------------------------------------

print("\n================================")
print(" GANHO SOBRE CAMPO ESCALAR")
print("================================")


print(
    "R² anterior tau:",
    resultado_campo_latente[
        "R2_Phi_tau"
    ]
)


print(
    "R² novo tau:",
    r_tau[0]
)


print(
    "R² anterior curv:",
    resultado_campo_latente[
        "R2_Phi_curvatura"
    ]
)


print(
    "R² novo curv:",
    r_curv[0]
)



resultado_campo_bidimensional = {

    "R2_tau": r_tau[0],

    "R2_curvatura": r_curv[0],

    "coef_tau": r_tau[2],

    "coef_curvatura": r_curv[2]

}


print("\nVariável criada:")
print("- resultado_campo_bidimensional")


print("\n=== FIM S24-D.5 ===")

 IA-1 — S24-D.5
 CAMPO LATENTE BIDIMENSIONAL RELACIONAL

 RECONSTRUÇÃO BIDIMENSIONAL

Phi(0,1) -> tau
R²: -17108.3740465808
MSE: 0.05405054022112459
Pesos: [-0.02161543  0.10757884]

Phi(0,1) -> curvatura
R²: -1070.0974248911182
MSE: 6.485618990761186
Pesos: [1.08889045 0.21646177]

 GANHO SOBRE CAMPO ESCALAR
R² anterior tau: -50778.43628645599
R² novo tau: -17108.3740465808
R² anterior curv: -2407.0471408254443
R² novo curv: -1070.0974248911182

Variável criada:
- resultado_campo_bidimensional

=== FIM S24-D.5 ===


In [ ]:
# ============================================================
# IA-1 — S24-D.6
# MEMÓRIA RELACIONAL NÃO LINEAR
# ============================================================

import numpy as np

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-D.6")
print(" MEMÓRIA RELACIONAL NÃO LINEAR")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_tempo_relacional",
    "resultado_invariantes",
    "eta_hist",
    "v_hist",
    "W_hist"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-D.2 antes."
    )



# ------------------------------------------------------------
# Construção das variáveis instantâneas
# ------------------------------------------------------------

tau = resultado_tempo_relacional[
    "tau_depth_velocity"
]


curv = resultado_invariantes[
    "curvatura"
]


eta_norm = np.linalg.norm(
    eta_hist,
    axis=1
)


v_norm = np.linalg.norm(
    v_hist,
    axis=1
)



N=min(
    len(tau),
    len(curv),
    len(W_hist),
    len(eta_norm),
    len(v_norm)
)



X_base=np.column_stack(
    [
        eta_norm[:N],
        v_norm[:N],
        W_hist[:N],
        curv[:N]
    ]
)


y=tau[:N]



# ------------------------------------------------------------
# Função com memória
# ------------------------------------------------------------

def construir_memoria(X,y,m):

    Xn=[]
    yn=[]

    for i in range(
        m,
        len(y)
    ):

        bloco=[]

        for j in range(
            m+1
        ):

            bloco.extend(
                X[i-j]
            )


        Xn.append(bloco)

        yn.append(
            y[i]
        )


    return np.array(Xn), np.array(yn)



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

memorias=[
    0,
    1,
    2,
    5,
    10,
    20
]


resultados=[]


print("\n================================")
print(" TESTE DE MEMÓRIA")
print("================================")



for m in memorias:


    Xm,ym = construir_memoria(
        X_base,
        y,
        m
    )


    corte=int(
        0.7*len(ym)
    )


    modelo=Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),

            (
                "ridge",
                Ridge(
                    alpha=1e-2
                )
            )
        ]
    )


    modelo.fit(
        Xm[:corte],
        ym[:corte]
    )


    pred=modelo.predict(
        Xm[corte:]
    )


    r2=r2_score(
        ym[corte:],
        pred
    )


    mse=mean_squared_error(
        ym[corte:],
        pred
    )


    resultados.append(
        [
            m,
            r2,
            mse
        ]
    )


    print(
        f"Memória {m:2d} | "
        f"R² {r2:.6f} | "
        f"MSE {mse:.6e}"
    )



# ------------------------------------------------------------
# Melhor regime
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:x[1]
)


print("\n================================")
print(" MELHOR REGIME")
print("================================")


print(
    "Memória ótima:",
    melhor[0]
)


print(
    "R²:",
    melhor[1]
)


print(
    "MSE:",
    melhor[2]
)



resultado_memoria_nao_linear = {

    "tabela": resultados,

    "melhor_memoria": melhor

}



print("\nVariável criada:")
print("- resultado_memoria_nao_linear")


print("\n=== FIM S24-D.6 ===")

 IA-1 — S24-D.6
 MEMÓRIA RELACIONAL NÃO LINEAR

 TESTE DE MEMÓRIA
Memória  0 | R² -27721849787.119118 | MSE 8.757661e+04
Memória  1 | R² -89085477087.687088 | MSE 2.814316e+05
Memória  2 | R² -88546975621.751389 | MSE 2.797304e+05
Memória  5 | R² -38650957535.911346 | MSE 1.219431e+05
Memória 10 | R² -14458917246.408258 | MSE 4.549752e+04
Memória 20 | R² -730418052.545921 | MSE 2.289228e+03

 MELHOR REGIME
Memória ótima: 20
R²: -730418052.5459212
MSE: 2289.2284633822833

Variável criada:
- resultado_memoria_nao_linear

=== FIM S24-D.6 ===


In [ ]:
# ============================================================
# IA-1 — S24-E.1
# MÉTRICA TEMPORAL VARIACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-E.1")
print(" MÉTRICA TEMPORAL VARIACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "W_hist",
    "K_rel_star",
    "resultado_tempo_relacional",
    "resultado_invariantes"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)



if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-D antes."
    )



# ------------------------------------------------------------
# Construção dos termos
# ------------------------------------------------------------

eta_norm = np.linalg.norm(
    eta_hist,
    axis=1
)


v_norm = np.linalg.norm(
    v_hist,
    axis=1
)



# termo de curvatura

Keta = eta_hist @ K_rel_star.T


curv_term = np.linalg.norm(
    Keta,
    axis=1
)



tau = resultado_tempo_relacional[
    "tau_depth_velocity"
]


curvatura = resultado_invariantes[
    "curvatura"
]


N=min(
    len(tau),
    len(W_hist),
    len(curv_term)
)



# ------------------------------------------------------------
# Candidatos variacionais
# ------------------------------------------------------------

S1 = eta_norm[:N]**2

S2 = v_norm[:N]**2

S3 = S1 + W_hist[:N]

S4 = S1 + W_hist[:N] + curv_term[:N]**2



candidatos={

    "S1_eta2":S1,

    "S2_v2":S2,

    "S3_eta_W":S3,

    "S4_eta_W_K":S4

}



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES VARIACIONAIS")
print("================================")



resultado_variacional={}



for nome,S in candidatos.items():


    rho_tau=np.corrcoef(
        S,
        tau[:N]
    )[0,1]


    rho_K=np.corrcoef(
        S,
        curvatura[:N]
    )[0,1]


    print(
        nome,
        "| tau:",
        rho_tau,
        "| K:",
        rho_K
    )


    resultado_variacional[nome]={
        "rho_tau":rho_tau,
        "rho_K":rho_K
    }



# ------------------------------------------------------------
# Teste preditivo combinado
# ------------------------------------------------------------

print("\n================================")
print(" TESTE PREDITIVO")
print("================================")



X=np.column_stack(
    [
        S1,
        S2,
        S3,
        S4
    ]
)



corte=int(
    0.7*N
)



modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)



modelo.fit(
    X[:corte],
    tau[:corte]
)



pred=modelo.predict(
    X[corte:]
)



r2_tau=r2_score(
    tau[corte:],
    pred
)



print(
    "R² variacional -> tau:",
    r2_tau
)



resultado_variacional["R2_tau"]=r2_tau


print("\nVariável criada:")
print("- resultado_variacional")


print("\n=== FIM S24-E.1 ===")

 IA-1 — S24-E.1
 MÉTRICA TEMPORAL VARIACIONAL

 CORRELAÇÕES VARIACIONAIS
S1_eta2 | tau: 0.22908640876155056 | K: 0.3525867814142549
S2_v2 | tau: 0.22777438154655136 | K: 0.35083991852845625
S3_eta_W | tau: -0.2271047956250134 | K: -0.3499461517489147
S4_eta_W_K | tau: 0.22651849033199659 | K: 0.3491621703542167

 TESTE PREDITIVO
R² variacional -> tau: -883621514.5793048

Variável criada:
- resultado_variacional

=== FIM S24-E.1 ===


In [ ]:
# ============================================================
# IA-1 — S24-E.2
# INVARIANTE GLOBAL RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-E.2")
print(" INVARIANTE GLOBAL RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "K_rel_star",
    "resultado_tempo_relacional",
    "resultado_invariantes"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute reconstrução S24 antes."
    )



# ------------------------------------------------------------
# Espectro global
# ------------------------------------------------------------

eigvals = np.linalg.eigvalsh(
    K_rel_star
)


abs_eig=np.abs(eigvals)


eps=1e-12



# Energia espectral

I1=np.sum(
    abs_eig
)



# Assimetria

positivo=np.sum(
    eigvals[eigvals>0]
)


negativo=np.sum(
    np.abs(eigvals[eigvals<0])
)


I2=positivo/(negativo+eps)



# Entropia modal

p=abs_eig/(np.sum(abs_eig)+eps)


I3=-np.sum(
    p*np.log(
        p+eps
    )
)



# Concentração

I4=np.max(
    abs_eig
)/(np.sum(abs_eig)+eps)



I5=I1*I3*I4



print("\n================================")
print(" INVARIANTES")
print("================================")


print("Energia espectral:",I1)
print("Assimetria:",I2)
print("Entropia modal:",I3)
print("Concentração:",I4)
print("Complexidade:",I5)



# ------------------------------------------------------------
# Séries temporais globais
# ------------------------------------------------------------

N=len(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


# criar séries normalizadas do espectro local
# usando curvatura efetiva disponível

curv=np.array(
    resultado_invariantes[
        "curvatura"
    ]
)



tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



# matriz de atributos globais
# (invariantes constantes + curvatura dinâmica)

X=np.column_stack(
    [
        np.ones(N)*I1,
        np.ones(N)*I2,
        np.ones(N)*I3,
        np.ones(N)*I4,
        np.ones(N)*I5,
        curv
    ]
)



# ------------------------------------------------------------
# Predição
# ------------------------------------------------------------

print("\n================================")
print(" TESTE GLOBAL")
print("================================")


corte=int(
    0.7*N
)



modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)



modelo.fit(
    X[:corte],
    tau[:corte]
)



pred=modelo.predict(
    X[corte:]
)



r2=r2_score(
    tau[corte:],
    pred
)



print(
    "R² global -> tau:",
    r2
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES")
print("================================")


for nome,val in {
    "I1":I1,
    "I2":I2,
    "I3":I3,
    "I4":I4,
    "I5":I5
}.items():

    corr=np.corrcoef(
        np.ones(N)*val,
        tau
    )[0,1]


    print(
        nome,
        "|",
        corr
    )



resultado_invariante_global={

    "I1":I1,
    "I2":I2,
    "I3":I3,
    "I4":I4,
    "I5":I5,
    "R2_tau":r2

}



print("\nVariável criada:")
print("- resultado_invariante_global")


print("\n=== FIM S24-E.2 ===")

 IA-1 — S24-E.2
 INVARIANTE GLOBAL RELACIONAL

 INVARIANTES
Energia espectral: 60.00910296296797
Assimetria: 0.8429962860216017
Entropia modal: 2.878414089575086
Concentração: 0.11543469340726673
Complexidade: 19.93915550677352

 TESTE GLOBAL
R² global -> tau: -627.0013969184168

 CORRELAÇÕES
I1 | 1.3195617819065374e-16
I2 | -1.3195617819065374e-16
I3 | -1.3195617819065374e-16
I4 | nan
I5 | nan

Variável criada:
- resultado_invariante_global

=== FIM S24-E.2 ===


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
# ============================================================
# IA-1 — S24-E.3
# DINÂMICA ESPECTRAL EMERGENTE
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-E.3")
print(" DINÂMICA ESPECTRAL EMERGENTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)



if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-A.2.0 e S24-D antes."
    )



# ------------------------------------------------------------
# Construção de espectro dinâmico aproximado
# ------------------------------------------------------------

N=len(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



I1=[]
I2=[]
I3=[]
I4=[]



eps=1e-12



for k in range(N):


    # deformação relacional instantânea
    eta=eta_hist[k]


    # operador efetivo modulado pelo estado
    Kk = (
        K_rel_star
        *
        (
            1
            +
            0.01*np.tanh(
                np.linalg.norm(eta)
            )
        )
    )


    eig=np.linalg.eigvalsh(
        Kk
    )


    abs_eig=np.abs(eig)


    pos=np.sum(
        eig[eig>0]
    )


    neg=np.sum(
        np.abs(eig[eig<0])
    )


    p=abs_eig/(
        np.sum(abs_eig)+eps
    )


    I1.append(
        np.sum(abs_eig)
    )


    I2.append(
        pos/(neg+eps)
    )


    I3.append(
        -np.sum(
            p*np.log(
                p+eps
            )
        )
    )


    I4.append(
        np.max(abs_eig)
        /
        (
        np.sum(abs_eig)+eps
        )
    )



I1=np.array(I1)
I2=np.array(I2)
I3=np.array(I3)
I4=np.array(I4)



X=np.column_stack(
    [
        I1,
        I2,
        I3,
        I4
    ]
)



# ------------------------------------------------------------
# Teste direto
# ------------------------------------------------------------

print("\n================================")
print(" ESPECTRO DINÂMICO")
print("================================")



def teste(X,y):

    corte=int(
        0.7*len(y)
    )


    modelo=Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(alpha=1e-3)
            )
        ]
    )


    modelo.fit(
        X[:corte],
        y[:corte]
    )


    pred=modelo.predict(
        X[corte:]
    )


    return r2_score(
        y[corte:],
        pred
    )



r2_tau=teste(
    X,
    tau
)



print(
    "R² espectro -> tau:",
    r2_tau
)



# ------------------------------------------------------------
# Teste diferencial
# ------------------------------------------------------------

print("\n================================")
print(" VARIAÇÃO ESPECTRAL")
print("================================")



dX=np.diff(
    X,
    axis=0
)


dtau=np.diff(
    tau
)


r2_delta=teste(
    dX,
    dtau
)



print(
    "R² Δespectro -> Δtau:",
    r2_delta
)



resultado_espectro_dinamico={

    "I1":I1,
    "I2":I2,
    "I3":I3,
    "I4":I4,

    "R2_tau":r2_tau,

    "R2_delta_tau":r2_delta

}



print("\nVariável criada:")
print("- resultado_espectro_dinamico")


print("\n=== FIM S24-E.3 ===")

 IA-1 — S24-E.3
 DINÂMICA ESPECTRAL EMERGENTE

 ESPECTRO DINÂMICO
R² espectro -> tau: -23.359036872545047

 VARIAÇÃO ESPECTRAL
R² Δespectro -> Δtau: -12968.394758043743

Variável criada:
- resultado_espectro_dinamico

=== FIM S24-E.3 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.1
# GEOMETRIA DE CAMINHO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.1")
print(" GEOMETRIA DE CAMINHO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "v_hist",
    "resultado_tempo_relacional",
    "kappa_A",
    "kappa_C"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-A.2 e S24-D antes."
    )



# ------------------------------------------------------------
# Construção do comprimento relacional
# ------------------------------------------------------------


eta=np.array(
    eta_hist
)


v=np.array(
    v_hist
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



# deslocamento relacional

deta=np.diff(
    eta,
    axis=0
)


dl=np.linalg.norm(
    deta,
    axis=1
)



# velocidade relacional média

v_rel=np.linalg.norm(
    v[:-1],
    axis=1
)



# evita zero

eps=1e-12



# ------------------------------------------------------------
# Campo de escala efetiva
# ------------------------------------------------------------

kappa_A_arr=np.array(
    kappa_A[:-1]
)


kappa_C_arr=np.array(
    kappa_C[:-1]
)



# Métricas candidatas

metricas={

    "A":
    kappa_A_arr*dl,


    "C":
    np.abs(kappa_C_arr)*dl,


    "velocidade":
    dl/(v_rel+eps)

}



# ------------------------------------------------------------
# Acúmulo temporal
# ------------------------------------------------------------

print("\n================================")
print(" CAMINHOS RELACIONAIS")
print("================================")



resultados={}


for nome, caminho in metricas.items():


    tau_modelo=np.cumsum(
        caminho
    )


    tau_modelo=(
        tau_modelo
        /
        (tau_modelo[-1]+eps)
    )


    tau_real=(
        tau[:len(tau_modelo)]
        /
        (tau[len(tau_modelo)-1]+eps)
    )



    r2=r2_score(
        tau_real,
        tau_modelo
    )


    resultados[nome]=r2


    print(
        nome,
        "| R²:",
        r2
    )



# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------


X=np.column_stack(
    [
        metricas["A"],
        metricas["C"],
        metricas["velocidade"]
    ]
)


y=tau[:len(X)]



corte=int(
    0.7*len(y)
)



modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)



modelo.fit(
    X[:corte],
    y[:corte]
)


pred=modelo.predict(
    X[corte:]
)



r2_completo=r2_score(
    y[corte:],
    pred
)



print("\n================================")
print(" MODELO COMBINADO")
print("================================")

print(
    "R² combinado:",
    r2_completo
)



resultado_caminho_relacional={

    "metricas":resultados,

    "R2_combinado":
    r2_completo

}



print("\nVariável criada:")
print("- resultado_caminho_relacional")


print("\n=== FIM S24-F.1 ===")

 IA-1 — S24-F.1
 GEOMETRIA DE CAMINHO RELACIONAL

 CAMINHOS RELACIONAIS
A | R²: -7.777776171211826
C | R²: -7.748145074056852
velocidade | R²: -1.2654913063107016

 MODELO COMBINADO
R² combinado: -170607.46707432732

Variável criada:
- resultado_caminho_relacional

=== FIM S24-F.1 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.2
# COMPLEXIDADE RELACIONAL ACUMULADA
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.2")
print(" COMPLEXIDADE RELACIONAL ACUMULADA")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute a reconstrução S24 antes."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(
    eta_hist
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



# ------------------------------------------------------------
# Complexidades
# ------------------------------------------------------------


# mudança de estado

Omega1=np.linalg.norm(
    np.diff(
        eta,
        axis=0
    ),
    axis=1
)



# mudança geométrica aproximada

K_base=np.array(
    K_rel_star
)


Omega2=[]


for k in range(len(Omega1)):

    fator1=np.linalg.norm(
        eta[k]
    )


    fator2=np.linalg.norm(
        eta[k+1]
    )


    Kk=(
        K_base
        *
        (
            1
            +
            0.01*np.tanh(fator1)
        )
    )


    Kk1=(
        K_base
        *
        (
            1
            +
            0.01*np.tanh(fator2)
        )
    )


    Omega2.append(
        np.linalg.norm(
            Kk1-Kk
        )
    )


Omega2=np.array(
    Omega2
)



# híbrido

alpha=1.0


Omega3=(
    Omega1
    +
    alpha*Omega2
)



metricas={

    "estado":Omega1,

    "geometria":Omega2,

    "hibrida":Omega3

}



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

print("\n================================")
print(" TESTE DE COMPLEXIDADE")
print("================================")


resultados={}


for nome,Omega in metricas.items():


    acumulada=np.cumsum(
        Omega
    )


    acumulada /= (
        acumulada[-1]
        +
        1e-12
    )


    y=tau[:len(acumulada)]

    y /= (
        y[-1]
        +
        1e-12
    )


    r2=r2_score(
        y,
        acumulada
    )


    resultados[nome]=r2


    print(
        nome,
        "| R²:",
        r2
    )



# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------

X=np.column_stack(
    [
        Omega1,
        Omega2,
        Omega3
    ]
)


y=tau[:len(X)]


corte=int(
    0.7*len(y)
)



modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)



modelo.fit(
    X[:corte],
    y[:corte]
)



pred=modelo.predict(
    X[corte:]
)



r2_comb=r2_score(
    y[corte:],
    pred
)



print("\n================================")
print(" MODELO COMBINADO")
print("================================")


print(
    "R² combinado:",
    r2_comb
)



resultado_complexidade_relacional={

    "R2_individuais":resultados,

    "R2_combinado":r2_comb,

    "Omega1":Omega1,

    "Omega2":Omega2,

    "Omega3":Omega3

}



print("\nVariável criada:")
print("- resultado_complexidade_relacional")


print("\n=== FIM S24-F.2 ===")

 IA-1 — S24-F.2
 COMPLEXIDADE RELACIONAL ACUMULADA

 TESTE DE COMPLEXIDADE
estado | R²: -7.659094213892221
geometria | R²: 0.47467942000428753
hibrida | R²: -7.653091827213247

 MODELO COMBINADO
R² combinado: -660946.5952212844

Variável criada:
- resultado_complexidade_relacional

=== FIM S24-F.2 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.3
# CURVATURA RELACIONAL ACUMULADA
# ============================================================

import numpy as np

from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.3")
print(" CURVATURA RELACIONAL ACUMULADA")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.2 antes."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(
    eta_hist
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



K0=np.array(
    K_rel_star
)



n=len(eta)-1



# ------------------------------------------------------------
# Construção de operadores geométricos dinâmicos
# ------------------------------------------------------------

K_hist=[]


for k in range(n):

    norma0=np.linalg.norm(
        eta[k]
    )


    norma1=np.linalg.norm(
        eta[k+1]
    )


    Kk=(
        K0
        *
        (
            1
            +
            0.01*np.tanh(norma0)
        )
    )


    Kk1=(
        K0
        *
        (
            1
            +
            0.01*np.tanh(norma1)
        )
    )


    K_hist.append(
        [
            Kk,
            Kk1
        ]
    )



# ------------------------------------------------------------
# Métricas geométricas
# ------------------------------------------------------------


Omega_K=[]

Omega_lambda=[]

Omega_trace=[]



for Kk,Kk1 in K_hist:


    # operador

    Omega_K.append(
        np.linalg.norm(
            Kk1-Kk
        )
    )


    # espectro

    l0=np.linalg.eigvalsh(
        Kk
    )


    l1=np.linalg.eigvalsh(
        Kk1
    )


    Omega_lambda.append(
        np.linalg.norm(
            l1-l0
        )
    )


    # traço

    Omega_trace.append(
        abs(
            np.trace(Kk1)
            -
            np.trace(Kk)
        )
    )



metricas={

    "K_operador":
    np.array(Omega_K),

    "modos_espectrais":
    np.array(Omega_lambda),

    "traco_curvatura":
    np.array(Omega_trace)

}



# ------------------------------------------------------------
# Teste acumulado
# ------------------------------------------------------------

print("\n================================")
print(" TESTE DE CURVATURA")
print("================================")


resultados={}


for nome,omega in metricas.items():


    acumulada=np.cumsum(
        omega
    )


    acumulada/=(
        acumulada[-1]
        +
        1e-12
    )


    y=tau[:len(acumulada)]

    y/=(
        y[-1]
        +
        1e-12
    )


    r2=r2_score(
        y,
        acumulada
    )


    resultados[nome]=r2


    print(
        nome,
        "| R²:",
        r2
    )



resultado_curvatura_relacional={

    "R2":resultados,

    "Omega_K":
    metricas["K_operador"],

    "Omega_lambda":
    metricas["modos_espectrais"],

    "Omega_trace":
    metricas["traco_curvatura"]

}



print("\nVariável criada:")
print("- resultado_curvatura_relacional")


print("\n=== FIM S24-F.3 ===")

 IA-1 — S24-F.3
 CURVATURA RELACIONAL ACUMULADA

 TESTE DE CURVATURA
K_operador | R²: 0.4746794200057233
modos_espectrais | R²: 0.47467941999960217
traco_curvatura | R²: 0.4746794199789268

Variável criada:
- resultado_curvatura_relacional

=== FIM S24-F.3 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.4
# TAXA RELACIONAL GEOMÉTRICO-ENERGÉTICA
# ============================================================

import numpy as np

from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.4")
print(" TAXA RELACIONAL GEOMÉTRICO-ENERGÉTICA")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "W_hist",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.3 e mantenha W_hist disponível."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(
    eta_hist
)


W=np.array(
    W_hist
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



K0=np.array(
    K_rel_star
)



n=min(
    len(eta)-1,
    len(W)-1,
    len(tau)-1
)



# ------------------------------------------------------------
# Construção da variação geométrica
# ------------------------------------------------------------

DeltaK=[]


for k in range(n):


    n0=np.linalg.norm(
        eta[k]
    )


    n1=np.linalg.norm(
        eta[k+1]
    )


    Kk=(
        K0*
        (
            1+
            0.01*np.tanh(n0)
        )
    )


    Kk1=(
        K0*
        (
            1+
            0.01*np.tanh(n1)
        )
    )


    DeltaK.append(
        np.linalg.norm(
            Kk1-Kk
        )
    )



DeltaK=np.array(
    DeltaK
)



W=W[:n]

DeltaW=np.diff(
    W
)


# alinhamento temporal
m=min(
    len(DeltaK),
    len(DeltaW),
    len(W)
)


DeltaK=DeltaK[:m]

W=W[:m]

DeltaW=DeltaW[:m]



# ------------------------------------------------------------
# Métricas
# ------------------------------------------------------------


Omega_A=DeltaK


Omega_B=(
    DeltaK*
    np.abs(W)
)


Omega_C=(
    DeltaK*
    np.abs(DeltaW)
)



metricas={

    "geometria":
    Omega_A,

    "geometria_energia":
    Omega_B,

    "geometria_variacao_energia":
    Omega_C

}



print("\n================================")
print(" TESTE GEOMÉTRICO-ENERGÉTICO")
print("================================")



resultados={}



for nome,omega in metricas.items():


    acumulada=np.cumsum(
        omega
    )


    acumulada/=(
        acumulada[-1]
        +
        1e-12
    )


    y=tau[:len(acumulada)]

    y/=(
        y[-1]
        +
        1e-12
    )


    r2=r2_score(
        y,
        acumulada
    )


    resultados[nome]=r2


    print(
        nome,
        "| R²:",
        r2
    )



resultado_geometria_energia={

    "R2":
    resultados,

    "Omega_A":
    Omega_A,

    "Omega_B":
    Omega_B,

    "Omega_C":
    Omega_C

}



print("\nVariável criada:")
print("- resultado_geometria_energia")


print("\n=== FIM S24-F.4 ===")

 IA-1 — S24-F.4
 TAXA RELACIONAL GEOMÉTRICO-ENERGÉTICA

 TESTE GEOMÉTRICO-ENERGÉTICO
geometria | R²: 0.47464000342530577
geometria_energia | R²: -0.5405046457599008
geometria_variacao_energia | R²: -0.5591339750905788

Variável criada:
- resultado_geometria_energia

=== FIM S24-F.4 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.5
# INTEGRAL DE CURVATURA RELACIONAL
# ============================================================

import numpy as np

from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.5")
print(" INTEGRAL DE CURVATURA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute os blocos anteriores S24-F."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(
    eta_hist
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


K0=np.array(
    K_rel_star
)



n=min(
    len(eta)-1,
    len(tau)
)



# ------------------------------------------------------------
# Construção das métricas acumulativas
# ------------------------------------------------------------

curvatura=[]

espectro=[]

curvatura_normalizada=[]



for k in range(n):


    norma=np.linalg.norm(
        eta[k]
    )


    Kk=(
        K0*
        (
            1+
            0.01*np.tanh(norma)
        )
    )


    curvatura.append(
        np.linalg.norm(Kk)
    )


    eig=np.linalg.eigvalsh(
        Kk
    )


    espectro.append(
        np.sum(
            np.abs(eig)
        )
    )


    curvatura_normalizada.append(
        np.linalg.norm(Kk)
        /
        (
            norma+
            1e-12
        )
    )



curvatura=np.array(curvatura)

espectro=np.array(espectro)

curvatura_normalizada=np.array(curvatura_normalizada)



metricas={

    "integral_norma_K":
    curvatura,

    "integral_espectral":
    espectro,

    "integral_normalizada":
    curvatura_normalizada

}



print("\n================================")
print(" TESTE DE CURVATURA ACUMULADA")
print("================================")



resultados={}



for nome,valor in metricas.items():


    acumulado=np.cumsum(
        valor
    )


    acumulado/=(
        acumulado[-1]
        +
        1e-12
    )


    y=tau[:len(acumulado)]

    y/=(
        y[-1]
        +
        1e-12
    )


    r2=r2_score(
        y,
        acumulado
    )


    resultados[nome]=r2


    print(
        nome,
        "| R²:",
        r2
    )



resultado_integral_curvatura={

    "R2":
    resultados,

    "K_integral":
    metricas["integral_norma_K"],

    "lambda_integral":
    metricas["integral_espectral"],

    "K_normalizada":
    metricas["integral_normalizada"]

}



print("\nVariável criada:")
print("- resultado_integral_curvatura")


print("\n=== FIM S24-F.5 ===")

 IA-1 — S24-F.5
 INTEGRAL DE CURVATURA RELACIONAL

 TESTE DE CURVATURA ACUMULADA
integral_norma_K | R²: -1.271228780121675
integral_espectral | R²: -1.271228780128264
integral_normalizada | R²: 0.9431683204380321

Variável criada:
- resultado_integral_curvatura

=== FIM S24-F.5 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.6
# VALIDAÇÃO PREDITIVA DA MÉTRICA TEMPORAL RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.6")
print(" VALIDAÇÃO PREDITIVA DA MÉTRICA TEMPORAL RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_integral_curvatura",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.5 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


Krel=np.array(
    resultado_integral_curvatura[
        "K_normalizada"
    ]
)



n=min(
    len(tau),
    len(Krel)
)


X=Krel[:n].reshape(-1,1)


y=tau[:n]



# normalização temporal

y_norm=(
    y-y.min()
) / (
    y.max()-y.min()+1e-12
)



# ------------------------------------------------------------
# Separação temporal
# ------------------------------------------------------------

split=int(
    0.7*n
)


X_train=X[:split]

X_test=X[split:]

y_train=y_norm[:split]

y_test=y_norm[split:]



# ------------------------------------------------------------
# Modelo
# ------------------------------------------------------------

modelo=LinearRegression()


modelo.fit(
    X_train,
    y_train
)


pred=modelo.predict(
    X_test
)



r2=r2_score(
    y_test,
    pred
)


mse=mean_squared_error(
    y_test,
    pred
)



print("\n================================")
print(" MODELO TEMPORAL RELACIONAL")
print("================================")


print(
    "R² teste:",
    r2
)


print(
    "MSE teste:",
    mse
)


print(
    "Coeficiente:",
    modelo.coef_[0]
)



# ------------------------------------------------------------
# Comparação com modelo nulo
# ------------------------------------------------------------

baseline=np.ones_like(
    y_test
)*np.mean(
    y_train
)


r2_base=r2_score(
    y_test,
    baseline
)


print("\n================================")
print(" COMPARAÇÃO")
print("================================")


print(
    "Baseline:",
    r2_base
)


print(
    "Ganho:",
    r2-r2_base
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_validacao_temporal={

    "R2_teste":
    r2,

    "MSE_teste":
    mse,

    "coeficiente":
    modelo.coef_[0],

    "baseline":
    r2_base

}



print("\nVariável criada:")
print("- resultado_validacao_temporal")


print("\n=== FIM S24-F.6 ===")

 IA-1 — S24-F.6
 VALIDAÇÃO PREDITIVA DA MÉTRICA TEMPORAL RELACIONAL

 MODELO TEMPORAL RELACIONAL
R² teste: -538.6254261605122
MSE teste: 0.010598617208566549
Coeficiente: -0.02806915268797804

 COMPARAÇÃO
Baseline: -2506.763691119149
Ganho: 1968.1382649586367

Variável criada:
- resultado_validacao_temporal

=== FIM S24-F.6 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.7
# TAXA DE COMPRESSÃO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.7")
print(" TAXA DE COMPRESSÃO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_integral_curvatura",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.5 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

Q=np.array(
    resultado_integral_curvatura[
        "K_normalizada"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(Q),
    len(tau)
)


Q=Q[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Taxas geométricas
# ------------------------------------------------------------

compressao_1=np.diff(Q)


compressao_2=np.diff(
    compressao_1
)



# alinhar temporalmente

tau_1=tau[1:]

tau_2=tau[2:]



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

print("\n================================")
print(" MODELOS DE COMPRESSÃO")
print("================================")


resultados={}



modelos={

    "Q":
    (
        Q.reshape(-1,1),
        tau
    ),

    "delta_Q":
    (
        compressao_1.reshape(-1,1),
        tau_1
    ),

    "delta2_Q":
    (
        compressao_2.reshape(-1,1),
        tau_2
    )

}



for nome,(X,y) in modelos.items():


    split=int(
        0.7*len(y)
    )


    model=LinearRegression()


    model.fit(
        X[:split],
        y[:split]
    )


    pred=model.predict(
        X[split:]
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[nome]={

        "R2":
        r2,

        "MSE":
        mse,

        "coef":
        model.coef_[0]

    }


    print(
        nome,
        "| R²:",
        r2,
        "| coef:",
        model.coef_[0]
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_compressao_relacional={

    "modelos":
    resultados,

    "compressao":
    compressao_1,

    "aceleracao":
    compressao_2

}



print("\nVariável criada:")
print("- resultado_compressao_relacional")


print("\n=== FIM S24-F.7 ===")

 IA-1 — S24-F.7
 TAXA DE COMPRESSÃO RELACIONAL

 MODELOS DE COMPRESSÃO
Q | R²: -538.6254261605158 | coef: -0.01124987615492867
delta_Q | R²: -3370.1126405903838 | coef: -1.25981603493128
delta2_Q | R²: -2025.4941231015673 | coef: 3206.674406829957

Variável criada:
- resultado_compressao_relacional

=== FIM S24-F.7 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.8
# OPERADOR DE PROFUNDIDADE RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.8")
print(" OPERADOR DE PROFUNDIDADE RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "K_rel_star",
    "eta_hist",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute a reconstrução S24-A.2.0 antes."
    )



# ------------------------------------------------------------
# Construção da informação relacional
# ------------------------------------------------------------

eps=1e-12


eta_norm=np.linalg.norm(
    eta_hist,
    axis=1
)



# intensidade geométrica

K_norm=np.linalg.norm(
    K_rel_star
)



# informação relacional acumulada

I_rel=np.ones_like(
    eta_norm
)*K_norm



# profundidade instantânea

D_phi=(
    I_rel /
    (eta_norm+eps)
)



# profundidade acumulada

D_phi_acc=np.cumsum(
    D_phi
)



# normalização

D_phi_norm=(
    D_phi /
    (np.mean(D_phi)+eps)
)



tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(tau),
    len(D_phi)
)


tau=tau[:n]

D_phi=D_phi[:n]
D_phi_acc=D_phi_acc[:n]
D_phi_norm=D_phi_norm[:n]



# ------------------------------------------------------------
# Teste preditivo
# ------------------------------------------------------------

print("\n================================")
print(" MODELOS DE PROFUNDIDADE")
print("================================")


resultados={}



modelos={

    "D_phi":
    D_phi.reshape(-1,1),

    "D_phi_acumulada":
    D_phi_acc.reshape(-1,1),

    "D_phi_normalizada":
    D_phi_norm.reshape(-1,1)

}



for nome,X in modelos.items():


    split=int(
        0.7*n
    )


    modelo=LinearRegression()


    modelo.fit(
        X[:split],
        tau[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        tau[split:],
        pred
    )


    mse=mean_squared_error(
        tau[split:],
        pred
    )


    resultados[nome]={

        "R2":
        r2,

        "MSE":
        mse,

        "coef":
        modelo.coef_[0]

    }


    print(
        nome,
        "| R²:",
        r2,
        "| coef:",
        modelo.coef_[0]
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_profundidade_relacional={

    "resultados":
    resultados,

    "D_phi":
    D_phi,

    "D_phi_acumulada":
    D_phi_acc,

    "D_phi_normalizada":
    D_phi_norm

}


print("\nVariável criada:")
print("- resultado_profundidade_relacional")


print("\n=== FIM S24-F.8 ===")

 IA-1 — S24-F.8
 OPERADOR DE PROFUNDIDADE RELACIONAL

 MODELOS DE PROFUNDIDADE
D_phi | R²: -533.426087191046 | coef: -0.011309668527954403
D_phi_acumulada | R²: -134.87585427679255 | coef: 1.5503172964109774e-05
D_phi_normalizada | R²: -533.426087191043 | coef: -0.09435667017460132

Variável criada:
- resultado_profundidade_relacional

=== FIM S24-F.8 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.9
# MEMÓRIA DE FECHAMENTO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.9")
print(" MEMÓRIA DE FECHAMENTO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção da memória relacional
# ------------------------------------------------------------

janelas=[
    1,
    5,
    10,
    20,
    50,
    100
]


resultados={}



print("\n================================")
print(" TESTE DE MEMÓRIA")
print("================================")



for m in janelas:


    memoria=[]

    alvo=[]


    for k in range(m,n):

        memoria.append(
            np.mean(
                D[k-m:k]
            )
        )

        alvo.append(
            tau[k]
        )


    X=np.array(memoria).reshape(-1,1)

    y=np.array(alvo)



    split=int(
        0.7*len(y)
    )


    modelo=LinearRegression()


    modelo.fit(
        X[:split],
        y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[m]={

        "R2":
        r2,

        "MSE":
        mse,

        "coef":
        modelo.coef_[0]

    }


    print(
        "Memória",
        m,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor regime
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR REGIME")
print("================================")


print(
    "Memória ótima:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_memoria_fechamento={

    "resultados":
    resultados,

    "melhor_memoria":
    melhor

}



print("\nVariável criada:")
print("- resultado_memoria_fechamento")


print("\n=== FIM S24-F.9 ===")

 IA-1 — S24-F.9
 MEMÓRIA DE FECHAMENTO RELACIONAL

 TESTE DE MEMÓRIA
Memória 1 | R²: -533.2305253219149 | MSE: 0.0016876975404038965
Memória 5 | R²: -529.4998571490021 | MSE: 0.0016737173743880107
Memória 10 | R²: -524.8922644442141 | MSE: 0.0016548122513973989
Memória 20 | R²: -516.0964495744606 | MSE: 0.0016206498524140797
Memória 50 | R²: -491.1520158662593 | MSE: 0.0015236738415362687
Memória 100 | R²: -451.28892238012855 | MSE: 0.0013726565138038316

 MELHOR REGIME
Memória ótima: 100
R²: -451.28892238012855

Variável criada:
- resultado_memoria_fechamento

=== FIM S24-F.9 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.10
# MEMÓRIA VETORIAL DE FECHAMENTO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler


print("="*60)
print(" IA-1 — S24-F.10")
print(" MEMÓRIA VETORIAL DE FECHAMENTO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Memória vetorial
# ------------------------------------------------------------

janelas=[
    5,
    10,
    20,
    50,
    100
]


resultados={}



print("\n================================")
print(" TESTE MEMÓRIA VETORIAL")
print("================================")



for m in janelas:


    X=[]
    y=[]


    for k in range(m,n):

        X.append(
            D[k-m:k]
        )

        y.append(
            tau[k]
        )


    X=np.array(X)

    y=np.array(y)



    split=int(
        0.7*len(y)
    )



    # normalização

    scaler=StandardScaler()


    X_train=scaler.fit_transform(
        X[:split]
    )


    X_test=scaler.transform(
        X[split:]
    )



    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X_train,
        y[:split]
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[m]={

        "R2":
        r2,

        "MSE":
        mse

    }


    print(
        "Memória",
        m,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor regime
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR REGIME")
print("================================")


print(
    "Memória ótima:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_memoria_vetorial={

    "resultados":
    resultados,

    "melhor_memoria":
    melhor

}


print("\nVariável criada:")
print("- resultado_memoria_vetorial")


print("\n=== FIM S24-F.10 ===")

 IA-1 — S24-F.10
 MEMÓRIA VETORIAL DE FECHAMENTO RELACIONAL

 TESTE MEMÓRIA VETORIAL
Memória 5 | R²: -512.7060561152737 | MSE: 0.0016207332384012
Memória 10 | R²: -409.7731432297451 | MSE: 0.001292569744641505
Memória 20 | R²: -117.5478368423296 | MSE: 0.00037154487221994435
Memória 50 | R²: -0.4068824118854566 | MSE: 4.355625619726912e-06
Memória 100 | R²: -0.09980076627999668 | MSE: 3.3377971712778116e-06

 MELHOR REGIME
Memória ótima: 100
R²: -0.09980076627999668

Variável criada:
- resultado_memoria_vetorial

=== FIM S24-F.10 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.11
# KERNEL DE MEMÓRIA RELACIONAL NÃO LINEAR
# ============================================================

import numpy as np

from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.11")
print(" KERNEL DE MEMÓRIA RELACIONAL NÃO LINEAR")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Memória vetorial
# ------------------------------------------------------------

m=100


X=[]

y=[]


for k in range(m,n):

    X.append(
        D[k-m:k]
    )

    y.append(
        tau[k]
    )


X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)


scaler=StandardScaler()


X_train=scaler.fit_transform(
    X[:split]
)


X_test=scaler.transform(
    X[split:]
)



# ------------------------------------------------------------
# Kernel Ridge
# ------------------------------------------------------------

kernels=[
    "rbf",
    "polynomial"
]


resultados={}



print("\n================================")
print(" TESTES KERNEL")
print("================================")



for ktype in kernels:


    modelo=KernelRidge(
        alpha=1e-3,
        kernel=ktype
    )


    modelo.fit(
        X_train,
        y[:split]
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[ktype]={

        "R2":
        r2,

        "MSE":
        mse

    }


    print(
        ktype,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR KERNEL")
print("================================")


print(
    "Kernel:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_kernel_memoria={

    "resultados":
    resultados,

    "melhor_kernel":
    melhor

}



print("\nVariável criada:")
print("- resultado_kernel_memoria")


print("\n=== FIM S24-F.11 ===")

 IA-1 — S24-F.11
 KERNEL DE MEMÓRIA RELACIONAL NÃO LINEAR

 TESTES KERNEL
rbf | R²: -1.487728796530865 | MSE: 7.550034874091997e-06
polynomial | R²: -1.4970812900075918 | MSE: 7.57841885710794e-06

 MELHOR KERNEL
Kernel: rbf
R²: -1.487728796530865

Variável criada:
- resultado_kernel_memoria

=== FIM S24-F.11 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.12
# FILTRO DE MEMÓRIA RELACIONAL OTIMIZADO
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.12")
print(" FILTRO DE MEMÓRIA RELACIONAL OTIMIZADO")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Funções de memória
# ------------------------------------------------------------

def memoria_uniforme(x,m):

    w=np.ones(m)/m

    return np.convolve(
        x,
        w[::-1],
        mode="valid"
    )



def memoria_exponencial(x,m,lamb):

    i=np.arange(m)

    w=np.exp(
        -i/lamb
    )

    w=w/np.sum(w)

    return np.convolve(
        x,
        w[::-1],
        mode="valid"
    )



# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

resultados={}



print("\n================================")
print(" FILTROS DE MEMÓRIA")
print("================================")



janelas=[
    20,
    50,
    100,
    200
]


for m in janelas:


    testes={}


    # uniforme

    testes["uniforme"]=(
        memoria_uniforme(
            D,
            m
        )
    )


    # exponenciais

    for lamb in [
        10,
        50,
        100
    ]:

        testes[
            "exp_"+str(lamb)
        ]=memoria_exponencial(
            D,
            m,
            lamb
        )



    for nome,X in testes.items():


        tamanho=min(
            len(X),
            len(tau)
        )


        X=X[-tamanho:]

        y=tau[-tamanho:]


        split=int(
            0.7*tamanho
        )


        modelo=Ridge(
            alpha=1.0
        )


        modelo.fit(
            X[:split].reshape(-1,1),
            y[:split]
        )


        pred=modelo.predict(
            X[split:].reshape(-1,1)
        )


        r2=r2_score(
            y[split:],
            pred
        )


        mse=mean_squared_error(
            y[split:],
            pred
        )


        resultados[
            nome+"_m"+str(m)
        ]={

            "R2":r2,

            "MSE":mse

        }


        print(
            nome,
            "janela",
            m,
            "| R²:",
            r2
        )



# ------------------------------------------------------------
# Melhor filtro
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR FILTRO")
print("================================")


print(
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_filtro_memoria={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_filtro_memoria")


print("\n=== FIM S24-F.12 ===")

 IA-1 — S24-F.12
 FILTRO DE MEMÓRIA RELACIONAL OTIMIZADO

 FILTROS DE MEMÓRIA
uniforme janela 20 | R²: -516.308168233853
exp_10 janela 20 | R²: -520.6538838666331
exp_50 janela 20 | R²: -517.2332577953282
exp_100 janela 20 | R²: -516.7717719208295
uniforme janela 50 | R²: -491.35631346266797
exp_10 janela 50 | R²: -511.1081908947076
exp_50 janela 50 | R²: -496.67721982014217
exp_100 janela 50 | R²: -494.0543823505783
uniforme janela 100 | R²: -451.52130813904006
exp_10 janela 100 | R²: -496.67715736491914
exp_50 janela 100 | R²: -469.3928077119418
exp_100 janela 100 | R²: -460.94390175496187
uniforme janela 200 | R²: -375.36767129715804
exp_10 janela 200 | R²: -452.0525594220136
exp_50 janela 200 | R²: -421.27239457321366
exp_100 janela 200 | R²: -402.63290502195696

 MELHOR FILTRO
uniforme_m200
R²: -375.36767129715804

Variável criada:
- resultado_filtro_memoria

=== FIM S24-F.12 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.13
# DIREÇÃO TEMPORAL DA MEMÓRIA RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.13")
print(" DIREÇÃO TEMPORAL DA MEMÓRIA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção dos operadores direcionais
# ------------------------------------------------------------

memorias=[
    10,
    20,
    50,
    100
]


resultados={}



print("\n================================")
print(" OPERADORES DIRECIONAIS")
print("================================")



for m in memorias:


    X=[]

    y=[]


    for k in range(m,n):


        atual=D[k]


        delta=(
            D[k]
            -
            D[k-m]
        )


        aceleracao=(
            D[k]
            -
            2*D[k-m//2]
            +
            D[k-m]
        )


        X.append(
            [
                atual,
                delta,
                aceleracao
            ]
        )


        y.append(
            tau[k]
        )



    X=np.array(X)

    y=np.array(y)



    split=int(
        0.7*len(y)
    )



    scaler=StandardScaler()


    X_train=scaler.fit_transform(
        X[:split]
    )


    X_test=scaler.transform(
        X[split:]
    )



    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X_train,
        y[:split]
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[m]={

        "R2":r2,

        "MSE":mse,

        "coef":
        modelo.coef_

    }


    print(
        "Memória",
        m,
        "| R²:",
        r2,
        "| coef:",
        modelo.coef_
    )



# ------------------------------------------------------------
# Melhor
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR REGIME")
print("================================")


print(
    "Memória:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_direcao_memoria={

    "resultados":
    resultados,

    "melhor_memoria":
    melhor

}



print("\nVariável criada:")
print("- resultado_direcao_memoria")


print("\n=== FIM S24-F.13 ===")

 IA-1 — S24-F.13
 DIREÇÃO TEMPORAL DA MEMÓRIA RELACIONAL

 OPERADORES DIRECIONAIS
Memória 10 | R²: -40.06758982689566 | coef: [-0.13344785 -0.04591747 -0.02181975]
Memória 20 | R²: -39.24968500355337 | coef: [-0.13159376 -0.04485753 -0.02148294]
Memória 50 | R²: -37.08413451515743 | coef: [-0.12628489 -0.04190844 -0.02060723]
Memória 100 | R²: -33.665537477348636 | coef: [-0.11764395 -0.03737252 -0.01918322]

 MELHOR REGIME
Memória: 100
R²: -33.665537477348636

Variável criada:
- resultado_direcao_memoria

=== FIM S24-F.13 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.14
# COMPLEXIDADE DA TRAJETÓRIA RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.14")
print(" COMPLEXIDADE DA TRAJETÓRIA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Complexidade da trajetória
# ------------------------------------------------------------

janelas=[
    20,
    50,
    100
]


resultados={}



print("\n================================")
print(" DESCRITORES DE COMPLEXIDADE")
print("================================")



for m in janelas:


    X=[]

    y=[]


    for k in range(m,n):


        janela=D[k-m:k]


        # 1 - variabilidade

        variacao=np.std(
            janela
        )


        # 2 - energia espectral

        fft=np.abs(
            np.fft.rfft(janela)
        )

        energia=np.sum(
            fft**2
        )


        # 3 - entropia espectral

        p=fft/(np.sum(fft)+1e-12)

        entropia=(
            -np.sum(
                p*np.log(
                    p+1e-12
                )
            )
        )


        X.append(
            [
                variacao,
                energia,
                entropia
            ]
        )


        y.append(
            tau[k]
        )



    X=np.array(X)

    y=np.array(y)



    split=int(
        0.7*len(y)
    )



    scaler=StandardScaler()


    X_train=scaler.fit_transform(
        X[:split]
    )


    X_test=scaler.transform(
        X[split:]
    )



    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X_train,
        y[:split]
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[m]={

        "R2":r2,

        "MSE":mse,

        "coef":
        modelo.coef_

    }


    print(
        "Janela",
        m,
        "| R²:",
        r2,
        "| coef:",
        modelo.coef_
    )



# ------------------------------------------------------------
# Melhor resultado
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR COMPLEXIDADE")
print("================================")


print(
    "Janela:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_complexidade_trajetoria={

    "resultados":
    resultados,

    "melhor_janela":
    melhor

}



print("\nVariável criada:")
print("- resultado_complexidade_trajetoria")


print("\n=== FIM S24-F.14 ===")

 IA-1 — S24-F.14
 COMPLEXIDADE DA TRAJETÓRIA RELACIONAL

 DESCRITORES DE COMPLEXIDADE
Janela 20 | R²: -101.96645430189487 | coef: [0.00222073 0.11174315 0.22003254]
Janela 50 | R²: -87.77517054465129 | coef: [0.00054818 0.09538393 0.19895417]
Janela 100 | R²: -69.36626529524216 | coef: [-0.00127318  0.07301003  0.16887312]

 MELHOR COMPLEXIDADE
Janela: 100
R²: -69.36626529524216

Variável criada:
- resultado_complexidade_trajetoria

=== FIM S24-F.14 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.15
# SELEÇÃO DE MEMÓRIA RELACIONAL RELEVANTE
# ============================================================

import numpy as np

from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.15")
print(" SELEÇÃO DE MEMÓRIA RELACIONAL RELEVANTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção da memória
# ------------------------------------------------------------

m=100


X=[]

y=[]


for k in range(m,n):

    X.append(
        D[k-m:k][::-1]
    )

    y.append(
        tau[k]
    )


X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Separação temporal
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)



scaler=StandardScaler()


X_train=scaler.fit_transform(
    X[:split]
)


X_test=scaler.transform(
    X[split:]
)



# ------------------------------------------------------------
# LASSO
# ------------------------------------------------------------

print("\n================================")
print(" LASSO MEMÓRIA")
print("================================")


lasso=Lasso(
    alpha=0.001,
    max_iter=10000
)


lasso.fit(
    X_train,
    y[:split]
)


pred=lasso.predict(
    X_test
)


r2_lasso=r2_score(
    y[split:],
    pred
)


mse_lasso=mean_squared_error(
    y[split:],
    pred
)



print(
    "Lasso R²:",
    r2_lasso
)


print(
    "MSE:",
    mse_lasso
)



# ------------------------------------------------------------
# Pesos relevantes
# ------------------------------------------------------------

pesos=lasso.coef_


indices=np.where(
    np.abs(pesos)>1e-8
)[0]



print("\n================================")
print(" ATRASOS RELEVANTES")
print("================================")



for i in indices:

    print(
        "Atraso",
        i,
        "| peso:",
        pesos[i]
    )



# ------------------------------------------------------------
# Ridge comparação
# ------------------------------------------------------------

ridge=Ridge(
    alpha=1.0
)


ridge.fit(
    X_train,
    y[:split]
)


pred2=ridge.predict(
    X_test
)


r2_ridge=r2_score(
    y[split:],
    pred2
)



print("\n================================")
print(" COMPARAÇÃO")
print("================================")


print(
    "Ridge R²:",
    r2_ridge
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_selecao_memoria={

    "lasso_R2":
    r2_lasso,

    "ridge_R2":
    r2_ridge,

    "pesos":
    pesos,

    "atrasos_relevantes":
    indices

}



print("\nVariável criada:")
print("- resultado_selecao_memoria")


print("\n=== FIM S24-F.15 ===")

 IA-1 — S24-F.15
 SELEÇÃO DE MEMÓRIA RELACIONAL RELEVANTE

 LASSO MEMÓRIA
Lasso R²: -215.53559538103346
MSE: 0.0006571662067379993

 ATRASOS RELEVANTES
Atraso 0 | peso: -0.17168510033862988
Atraso 99 | peso: 0.08165461578509617

 COMPARAÇÃO
Ridge R²: -0.09980076627999912

Variável criada:
- resultado_selecao_memoria

=== FIM S24-F.15 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.16
# ESCALA TEMPORAL RELACIONAL CARACTERÍSTICA
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.16")
print(" ESCALA TEMPORAL RELACIONAL CARACTERÍSTICA")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Escalas temporais
# ------------------------------------------------------------

delays=[
    1,
    2,
    5,
    10,
    20,
    50,
    100,
    200,
    500
]


resultados={}



print("\n================================")
print(" TESTE DE ESCALA TEMPORAL")
print("================================")



for delta in delays:


    if delta>=n:

        continue


    X=[]

    y=[]


    for k in range(delta,n):

        X.append(
            [
                D[k],
                D[k-delta]
            ]
        )

        y.append(
            tau[k]
        )


    X=np.array(X)

    y=np.array(y)



    split=int(
        0.7*len(y)
    )



    scaler=StandardScaler()


    X_train=scaler.fit_transform(
        X[:split]
    )


    X_test=scaler.transform(
        X[split:]
    )



    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X_train,
        y[:split]
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[delta]={

        "R2":r2,

        "MSE":mse,

        "coef":
        modelo.coef_

    }


    print(
        "Delta",
        delta,
        "| R²:",
        r2,
        "| coef:",
        modelo.coef_
    )



# ------------------------------------------------------------
# Melhor escala
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" ESCALA DOMINANTE")
print("================================")


print(
    "Delta:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_escala_temporal={

    "resultados":
    resultados,

    "melhor_delta":
    melhor

}



print("\nVariável criada:")
print("- resultado_escala_temporal")


print("\n=== FIM S24-F.16 ===")

 IA-1 — S24-F.16
 ESCALA TEMPORAL RELACIONAL CARACTERÍSTICA

 TESTE DE ESCALA TEMPORAL
Delta 1 | R²: -530.8566417577142 | coef: [-0.08775851 -0.01722087]
Delta 2 | R²: -527.4185868843728 | coef: [-0.12264669  0.01782659]
Delta 5 | R²: -507.57870996885833 | coef: [-0.22403809  0.11970175]
Delta 10 | R²: -449.1534301204247 | coef: [-0.37441128  0.27087588]
Delta 20 | R²: -296.0722819920243 | coef: [-0.57494724  0.47297912]
Delta 50 | R²: -52.38257209495707 | coef: [-0.61646905  0.51907144]
Delta 100 | R²: -4.515923848698182 | coef: [-0.40727713  0.31725294]
Delta 200 | R²: -0.5734995642130754 | coef: [-0.21347925  0.13751989]
Delta 500 | R²: -3.057338932607995 | coef: [-0.06635942  0.02573736]

 ESCALA DOMINANTE
Delta: 200
R²: -0.5734995642130754

Variável criada:
- resultado_escala_temporal

=== FIM S24-F.16 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.17
# MEMÓRIA RELACIONAL MULTIESCALA
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.17")
print(" MEMÓRIA RELACIONAL MULTIESCALA")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Escalas multiescala
# ------------------------------------------------------------

escalas=[

    1,
    10,
    50,
    100,
    200

]


X=[]

y=[]


max_delay=max(escalas)


for k in range(max_delay,n):


    vetor=[]


    for d in escalas:

        vetor.append(
            D[k-d]
        )


    X.append(
        vetor
    )


    y.append(
        tau[k]
    )



X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Separação temporal
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)



scaler=StandardScaler()


X_train=scaler.fit_transform(
    X[:split]
)


X_test=scaler.transform(
    X[split:]
)



# ------------------------------------------------------------
# Modelo
# ------------------------------------------------------------

print("\n================================")
print(" MODELO MULTIESCALA")
print("================================")



modelo=Ridge(
    alpha=1.0
)


modelo.fit(
    X_train,
    y[:split]
)


pred=modelo.predict(
    X_test
)


r2=r2_score(
    y[split:],
    pred
)


mse=mean_squared_error(
    y[split:],
    pred
)



print(
    "R²:",
    r2
)


print(
    "MSE:",
    mse
)



print(
    "Coeficientes:"
)


for d,c in zip(
    escalas,
    modelo.coef_
):

    print(
        "Delta",
        d,
        "|",
        c
    )



# ------------------------------------------------------------
# Comparação
# ------------------------------------------------------------

print("\n================================")
print(" REFERÊNCIAS")
print("================================")


print(
    "F10 memória vetorial:",
    -0.09980076627999668
)


print(
    "F16 melhor escala:",
    -0.5734995642130754
)



resultado_memoria_multiescala={

    "R2":
    r2,

    "MSE":
    mse,

    "escalas":
    escalas,

    "coeficientes":
    modelo.coef_

}



print("\nVariável criada:")
print("- resultado_memoria_multiescala")


print("\n=== FIM S24-F.17 ===")

 IA-1 — S24-F.17
 MEMÓRIA RELACIONAL MULTIESCALA

 MODELO MULTIESCALA
R²: -0.7441683914949149
MSE: 5.054901452076846e-06
Coeficientes:
Delta 1 | -0.14481671001203358
Delta 10 | -0.11572169705694182
Delta 50 | -0.00581185004474567
Delta 100 | 0.08452304562627255
Delta 200 | 0.10567040480690199

 REFERÊNCIAS
F10 memória vetorial: -0.09980076627999668
F16 melhor escala: -0.5734995642130754

Variável criada:
- resultado_memoria_multiescala

=== FIM S24-F.17 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.18
# DECOMPOSIÇÃO MODAL DA MEMÓRIA RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.18")
print(" DECOMPOSIÇÃO MODAL DA MEMÓRIA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção da memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for k in range(janela,n):

    X.append(
        D[k-janela:k]
    )

    y.append(
        tau[k]
    )


X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Normalização temporal
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)



scaler=StandardScaler()


X_train=scaler.fit_transform(
    X[:split]
)


X_test=scaler.transform(
    X[split:]
)



# ------------------------------------------------------------
# Teste PCA
# ------------------------------------------------------------

modos=[

    1,
    2,
    5,
    10,
    20

]


resultados={}



print("\n================================")
print(" MODOS PCA")
print("================================")



for m in modos:


    pca=PCA(
        n_components=m
    )


    Z_train=pca.fit_transform(
        X_train
    )


    Z_test=pca.transform(
        X_test
    )



    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        Z_train,
        y[:split]
    )


    pred=modelo.predict(
        Z_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )



    resultados[m]={

        "R2":r2,

        "MSE":mse,

        "variancia":
        np.sum(
            pca.explained_variance_ratio_
        )

    }


    print(
        "Modos",
        m,
        "| R²:",
        r2,
        "| Variância:",
        resultados[m]["variancia"]
    )



# ------------------------------------------------------------
# Melhor modo
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR REPRESENTAÇÃO")
print("================================")


print(
    "Modos:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)


print(
    "Variância:",
    resultados[melhor]["variancia"]
)



resultado_decomposicao_modal={

    "resultados":
    resultados,

    "melhor_modos":
    melhor

}



print("\nVariável criada:")
print("- resultado_decomposicao_modal")


print("\n=== FIM S24-F.18 ===")

 IA-1 — S24-F.18
 DECOMPOSIÇÃO MODAL DA MEMÓRIA RELACIONAL

 MODOS PCA
Modos 1 | R²: -450.9976679901629 | Variância: 0.9994142911871242
Modos 2 | R²: -0.29272820785818787 | Variância: 0.9999996626110789
Modos 5 | R²: -0.09980076628163004 | Variância: 0.9999999999999973
Modos 10 | R²: -0.09980076628072942 | Variância: 0.9999999999999983
Modos 20 | R²: -0.09980076628018786 | Variância: 0.9999999999999994

 MELHOR REPRESENTAÇÃO
Modos: 20
R²: -0.09980076628018786
Variância: 0.9999999999999994

Variável criada:
- resultado_decomposicao_modal

=== FIM S24-F.18 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.19
# CAMPO MODAL TEMPORAL RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.19")
print(" CAMPO MODAL TEMPORAL RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção da memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for k in range(janela,n):

    X.append(
        D[k-janela:k]
    )

    y.append(
        tau[k]
    )


X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# PCA temporal
# ------------------------------------------------------------

scaler=StandardScaler()


Xn=scaler.fit_transform(
    X
)


pca=PCA(
    n_components=5
)


Phi=pca.fit_transform(
    Xn
)



# ------------------------------------------------------------
# Correlação dos modos
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÃO DOS MODOS")
print("================================")


correlacoes=[]


for i in range(5):

    c=np.corrcoef(
        Phi[:,i],
        y
    )[0,1]


    correlacoes.append(c)


    print(
        "Modo",
        i,
        "| correlação tau:",
        c
    )



# ------------------------------------------------------------
# Modelo modal
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)


modelo=Ridge(
    alpha=1.0
)


modelo.fit(
    Phi[:split],
    y[:split]
)


pred=modelo.predict(
    Phi[split:]
)



r2=r2_score(
    y[split:],
    pred
)


mse=mean_squared_error(
    y[split:],
    pred
)



print("\n================================")
print(" MODELO MODAL")
print("================================")


print(
    "R²:",
    r2
)


print(
    "MSE:",
    mse
)



print(
    "Pesos:"
)


for i,w in enumerate(modelo.coef_):

    print(
        "Modo",
        i,
        "| peso:",
        w
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_campo_modal_temporal={

    "R2":
    r2,

    "MSE":
    mse,

    "correlacoes":
    correlacoes,

    "pesos":
    modelo.coef_

}



print("\nVariável criada:")
print("- resultado_campo_modal_temporal")


print("\n=== FIM S24-F.19 ===")

 IA-1 — S24-F.19
 CAMPO MODAL TEMPORAL RELACIONAL

 CORRELAÇÃO DOS MODOS
Modo 0 | correlação tau: -0.923791954175295
Modo 1 | correlação tau: 0.36553610722397656
Modo 2 | correlação tau: -0.08960467081092689
Modo 3 | correlação tau: 0.064994868163761
Modo 4 | correlação tau: -0.008681458887664329

 MODELO MODAL
R²: -0.0996980431642347
MSE: 3.337485415789247e-06
Pesos:
Modo 0 | peso: -0.00812927685488531
Modo 1 | peso: 0.14246488995955567
Modo 2 | peso: -0.11207583980168868
Modo 3 | peso: 0.002419404256090549
Modo 4 | peso: -8.283404037127885e-06

Variável criada:
- resultado_campo_modal_temporal

=== FIM S24-F.19 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.20
# MAPEAMENTO NÃO LINEAR DO CAMPO MODAL TEMPORAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.20")
print(" MAPEAMENTO NÃO LINEAR DO CAMPO MODAL TEMPORAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Memória temporal
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for k in range(janela,n):

    X.append(
        D[k-janela:k]
    )

    y.append(
        tau[k]
    )



X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# PCA modal
# ------------------------------------------------------------

scaler=StandardScaler()


Xn=scaler.fit_transform(
    X
)


pca=PCA(
    n_components=5
)


Phi=pca.fit_transform(
    Xn
)



# ------------------------------------------------------------
# Separação temporal
# ------------------------------------------------------------

split=int(
    0.7*len(y)
)



Phi_train=Phi[:split]

Phi_test=Phi[split:]



# ------------------------------------------------------------
# Modelos corrigidos
# ------------------------------------------------------------

modelos={

"ridge":
Ridge(
    alpha=1.0
),


"polynomial":

Pipeline(
[
("poly",
PolynomialFeatures(
    degree=2
)),

("ridge",
Ridge(
    alpha=1.0
))

]
),


"rbf":

KernelRidge(
    kernel="rbf",
    alpha=1.0,
    gamma=1.0
)

}



resultados={}



print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")



for nome,modelo in modelos.items():


    modelo.fit(
        Phi_train,
        y[:split]
    )


    pred=modelo.predict(
        Phi_test
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor modelo
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR MAPEAMENTO")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_mapeamento_modal_nl={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_mapeamento_modal_nl")


print("\n=== FIM S24-F.20 ===")

 IA-1 — S24-F.20
 MAPEAMENTO NÃO LINEAR DO CAMPO MODAL TEMPORAL

 MODELOS NÃO LINEARES
ridge | R²: -0.0996980431642347 | MSE: 3.337485415789247e-06
polynomial | R²: -1.8662045505629736 | MSE: 8.698675009594498e-06
rbf | R²: -6737.0524056865925 | MSE: 0.020449387697459197

 MELHOR MAPEAMENTO
Modelo: ridge
R²: -0.0996980431642347

Variável criada:
- resultado_mapeamento_modal_nl

=== FIM S24-F.20 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.21
# PERSISTÊNCIA DO CAMPO TEMPORAL MODAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler



print("="*60)
print(" IA-1 — S24-F.21")
print(" PERSISTÊNCIA DO CAMPO TEMPORAL MODAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for k in range(janela,n):

    X.append(
        D[k-janela:k]
    )

    y.append(
        tau[k]
    )


X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Regiões
# ------------------------------------------------------------

N=len(y)

terco=N//3


regioes={

"inicio":
(
0,
terco
),

"meio":
(
terco,
2*terco
),

"final":
(
2*terco,
N
)

}



resultados={}



print("\n================================")
print(" ANÁLISE REGIONAL")
print("================================")



for nome,(a,b) in regioes.items():


    XR=X[a:b]


    yr=y[a:b]



    scaler=StandardScaler()


    XRn=scaler.fit_transform(
        XR
    )


    pca=PCA(
        n_components=5
    )


    Phi=pca.fit_transform(
        XRn
    )



    corr=[]


    for i in range(5):

        c=np.corrcoef(
            Phi[:,i],
            yr
        )[0,1]

        corr.append(c)



    resultados[nome]={

        "correlacoes":
        corr,

        "variancia":
        pca.explained_variance_ratio_

    }



    print("\nRegião:",nome)


    for i,c in enumerate(corr):

        print(
            "Modo",
            i,
            "| correlação:",
            c
        )


    print(
        "Variância acumulada:",
        np.sum(
            pca.explained_variance_ratio_
        )
    )



# ------------------------------------------------------------
# Comparação
# ------------------------------------------------------------

print("\n================================")
print(" COMPARAÇÃO MODAL")
print("================================")


c0=resultados["inicio"]["correlacoes"]

for nome in ["meio","final"]:

    c=resultados[nome]["correlacoes"]

    similaridade=np.corrcoef(
        c0,
        c
    )[0,1]


    print(
        nome,
        "| similaridade modal:",
        similaridade
    )



resultado_persistencia_modal={

    "regioes":
    resultados

}



print("\nVariável criada:")
print("- resultado_persistencia_modal")


print("\n=== FIM S24-F.21 ===")

 IA-1 — S24-F.21
 PERSISTÊNCIA DO CAMPO TEMPORAL MODAL

 ANÁLISE REGIONAL

Região: inicio
Modo 0 | correlação: -0.927301640748901
Modo 1 | correlação: 0.34861015954600527
Modo 2 | correlação: -0.11760846871337861
Modo 3 | correlação: 0.06537120157274176
Modo 4 | correlação: -0.015036071757634713
Variância acumulada: 1.0

Região: meio
Modo 0 | correlação: -0.9880340641758034
Modo 1 | correlação: 0.14880191214770097
Modo 2 | correlação: 0.005580929282824485
Modo 3 | correlação: -0.03979039282973635
Modo 4 | correlação: -0.005666902807885385
Variância acumulada: 1.0

Região: final
Modo 0 | correlação: 0.921922385618198
Modo 1 | correlação: 0.3758339614235763
Modo 2 | correlação: -0.0900421935355678
Modo 3 | correlação: -0.02462605961323184
Modo 4 | correlação: -0.004382611748757673
Variância acumulada: 1.0000000000000002

 COMPARAÇÃO MODAL
meio | similaridade modal: 0.9672571820548325
final | similaridade modal: -0.6916104144061759

Variável criada:
- resultado_persistencia_modal

=== FIM

In [ ]:
# ============================================================
# IA-1 — S24-F.22
# DETECTOR DE TRANSIÇÃO DO CAMPO TEMPORAL MODAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler



print("="*60)
print(" IA-1 — S24-F.22")
print(" DETECTOR DE TRANSIÇÃO DO CAMPO TEMPORAL MODAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Memória móvel
# ------------------------------------------------------------

janela=100


phi0=[]

indices=[]



for k in range(janela,n):


    bloco=D[k-janela:k]


    scaler=StandardScaler()


    bloco_n=scaler.fit_transform(
        bloco.reshape(-1,1)
    )


    pca=PCA(
        n_components=1
    )


    modo=pca.fit_transform(
        bloco_n
    )


    # direção média do modo

    valor=np.mean(modo)


    phi0.append(valor)

    indices.append(k)



phi0=np.array(phi0)

indices=np.array(indices)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

phi0=(

    phi0-np.mean(phi0)

)/np.std(phi0)



# ------------------------------------------------------------
# Variação do campo
# ------------------------------------------------------------

delta_phi=np.diff(phi0)



print("\n================================")
print(" VARIAÇÃO DO CAMPO")
print("================================")


print(
    "Média phi0:",
    np.mean(phi0)
)


print(
    "Desvio phi0:",
    np.std(phi0)
)


print(
    "Maior variação:",
    np.max(np.abs(delta_phi))
)



# ------------------------------------------------------------
# Detecção de transição
# ------------------------------------------------------------

limiar=np.std(delta_phi)*3


eventos=np.where(
    np.abs(delta_phi)>limiar
)[0]



print("\n================================")
print(" EVENTOS DE TRANSIÇÃO")
print("================================")


if len(eventos)==0:

    print(
        "Nenhuma transição detectada"
    )

else:

    for e in eventos:

        print(
            "Índice:",
            indices[e],
            "| salto:",
            delta_phi[e]
        )



# ------------------------------------------------------------
# Correlação temporal
# ------------------------------------------------------------

corr=np.corrcoef(
    phi0,
    tau[janela:]
)[0,1]



print("\n================================")
print(" RELAÇÃO COM TAU")
print("================================")


print(
    "phi0 vs tau:",
    corr
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_transicao_modal={

    "phi0":
    phi0,

    "delta_phi":
    delta_phi,

    "eventos":
    eventos,

    "correlacao_tau":
    corr

}



print("\nVariável criada:")
print("- resultado_transicao_modal")


print("\n=== FIM S24-F.22 ===")

 IA-1 — S24-F.22
 DETECTOR DE TRANSIÇÃO DO CAMPO TEMPORAL MODAL

 VARIAÇÃO DO CAMPO
Média phi0: 9.800589458760002e-18
Desvio phi0: 0.9999999999999999
Maior variação: 6.1090512646327895

 EVENTOS DE TRANSIÇÃO
Índice: 846 | salto: -6.1090512646327895
Índice: 1278 | salto: -4.247245164935177
Índice: 1416 | salto: 4.596333808628479
Índice: 1673 | salto: -4.421789486781828
Índice: 1731 | salto: -4.363608046166277
Índice: 1748 | salto: -4.596333808628479
Índice: 1749 | salto: 4.654515249244029
Índice: 2164 | salto: -4.421789486781828
Índice: 2267 | salto: -5.178148214783983
Índice: 2476 | salto: -5.178148214783983
Índice: 2648 | salto: 4.305426605550727
Índice: 2757 | salto: -4.247245164935177

 RELAÇÃO COM TAU
phi0 vs tau: 0.013323149329503516

Variável criada:
- resultado_transicao_modal

=== FIM S24-F.22 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.23
# ALINHAMENTO ORIENTADO DO CAMPO MODAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler



print("="*60)
print(" IA-1 — S24-F.23")
print(" ALINHAMENTO ORIENTADO DO CAMPO MODAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "resultado_profundidade_relacional",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:

    if x not in globals():

        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")

    for x in faltantes:

        print("-",x)

    raise RuntimeError(
        "Execute S24-F.8 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

D=np.array(
    resultado_profundidade_relacional[
        "D_phi"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(D),
    len(tau)
)


D=D[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção modal móvel
# ------------------------------------------------------------

janela=100


phi=[]

idx=[]



for k in range(janela,n):


    bloco=D[k-janela:k]


    scaler=StandardScaler()


    bloco_n=scaler.fit_transform(
        bloco.reshape(-1,1)
    )


    pca=PCA(
        n_components=1
    )


    modo=pca.fit_transform(
        bloco_n
    )


    valor=modo[-1,0]


    phi.append(valor)

    idx.append(k)



phi=np.array(phi)

idx=np.array(idx)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

phi=(

    phi-np.mean(phi)

)/np.std(phi)



# ------------------------------------------------------------
# Alinhamento de orientação
# ------------------------------------------------------------

phi_alinhado=np.copy(phi)



for i in range(1,len(phi_alinhado)):


    if phi_alinhado[i]*phi_alinhado[i-1] < 0:

        phi_alinhado[i]*=-1



# ------------------------------------------------------------
# Métricas
# ------------------------------------------------------------

delta_original=np.diff(phi)

delta_alinhado=np.diff(phi_alinhado)



print("\n================================")
print(" VARIAÇÃO ORIGINAL")
print("================================")


print(
    "Maior salto:",
    np.max(np.abs(delta_original))
)


print(
    "Desvio:",
    np.std(delta_original)
)



print("\n================================")
print(" VARIAÇÃO ALINHADA")
print("================================")


print(
    "Maior salto:",
    np.max(np.abs(delta_alinhado))
)


print(
    "Desvio:",
    np.std(delta_alinhado)
)



# ------------------------------------------------------------
# Correlação
# ------------------------------------------------------------

corr_original=np.corrcoef(
    phi,
    tau[janela:]
)[0,1]


corr_alinhado=np.corrcoef(
    phi_alinhado,
    tau[janela:]
)[0,1]



print("\n================================")
print(" CORRELAÇÃO TEMPORAL")
print("================================")


print(
    "Original:",
    corr_original
)


print(
    "Alinhado:",
    corr_alinhado
)



# ------------------------------------------------------------
# Eventos restantes
# ------------------------------------------------------------

limiar=np.std(delta_alinhado)*3


eventos=np.where(
    np.abs(delta_alinhado)>limiar
)[0]



print("\n================================")
print(" EVENTOS RESTANTES")
print("================================")


if len(eventos)==0:

    print(
        "Nenhuma transição forte restante"
    )

else:

    for e in eventos:

        print(
            "Índice:",
            idx[e],
            "| salto:",
            delta_alinhado[e]
        )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_campo_modal_orientado={

    "phi_original":
    phi,

    "phi_alinhado":
    phi_alinhado,

    "corr_original":
    corr_original,

    "corr_alinhado":
    corr_alinhado,

    "eventos":
    eventos

}



print("\nVariável criada:")
print("- resultado_campo_modal_orientado")


print("\n=== FIM S24-F.23 ===")

 IA-1 — S24-F.23
 ALINHAMENTO ORIENTADO DO CAMPO MODAL

 VARIAÇÃO ORIGINAL
Maior salto: 0.059382796936445104
Desvio: 0.005161861671183464

 VARIAÇÃO ALINHADA
Maior salto: 0.059382796936445104
Desvio: 0.0052961349512970915

 CORRELAÇÃO TEMPORAL
Original: 0.9892066624883327
Alinhado: 0.8544084275805843

 EVENTOS RESTANTES
Índice: 100 | salto: 0.059382796936445104
Índice: 101 | salto: 0.05775321682679557
Índice: 102 | salto: 0.05618471053867058
Índice: 103 | salto: 0.05467446114632679
Índice: 104 | salto: 0.05321979899101503
Índice: 105 | salto: 0.051818193885616814
Índice: 106 | salto: 0.05046724764219768
Índice: 107 | salto: 0.04916468690062459
Índice: 108 | salto: 0.04790835630758483
Índice: 109 | salto: 0.04669621200315888
Índice: 110 | salto: 0.0455263154347989
Índice: 111 | salto: 0.04439682752509455
Índice: 112 | salto: 0.04330600308593535
Índice: 113 | salto: 0.04225218562164468
Índice: 114 | salto: 0.041233802331340996
Índice: 115 | salto: 0.04024935946333663
Índice: 116 | salto:

In [ ]:
# ============================================================
# IA-1 — S24-F.24
# RECONSTRUÇÃO TEMPORAL PELO MODO DOMINANTE
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.24")
print(" RECONSTRUÇÃO TEMPORAL PELO MODO DOMINANTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )



if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Recuperação
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)



tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Construção de modos sintéticos
# ------------------------------------------------------------

# Como o F.23 mostrou dominância do primeiro modo,
# vamos usar decomposição temporal local


X0=phi.reshape(-1,1)



# derivadas locais como modos complementares

phi1=np.gradient(phi)

phi2=np.gradient(phi1)



X1=np.column_stack(
    [
        phi,
        phi1
    ]
)



Xall=np.column_stack(
    [
        phi,
        phi1,
        phi2
    ]
)



# ------------------------------------------------------------
# Separação temporal
# ------------------------------------------------------------

split=int(
    0.7*n
)



modelos={

"modo_0":
X0,

"modo_0_1":
X1,

"modo_completo":
Xall

}



resultados={}



print("\n================================")
print(" RECONSTRUÇÃO")
print("================================")



for nome,X in modelos.items():


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        tau[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        tau[split:],
        pred
    )


    mse=mean_squared_error(
        tau[split:],
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor representação
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR REPRESENTAÇÃO")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



resultado_reconstrucao_temporal_modal={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_reconstrucao_temporal_modal")


print("\n=== FIM S24-F.24 ===")

 IA-1 — S24-F.24
 RECONSTRUÇÃO TEMPORAL PELO MODO DOMINANTE

 RECONSTRUÇÃO
modo_0 | R²: -27.272682616674366 | MSE: 7.34006297719589e-05
modo_0_1 | R²: -26.46757976079738 | MSE: 7.131044761790605e-05
modo_completo | R²: -26.466683478328317 | MSE: 7.130812071817125e-05

 MELHOR REPRESENTAÇÃO
Modelo: modo_completo
R²: -26.466683478328317

Variável criada:
- resultado_reconstrucao_temporal_modal

=== FIM S24-F.24 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.25
# DINÂMICA DIFERENCIAL DO TEMPO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.25")
print(" DINÂMICA DIFERENCIAL DO TEMPO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Derivadas temporais
# ------------------------------------------------------------

delta_tau=np.diff(tau)

delta2_tau=np.diff(
    delta_tau
)


# ajustar tamanho

phi1=phi[1:]

phi2=phi[2:]



# derivadas do campo

dphi=np.diff(phi)

d2phi=np.diff(
    dphi
)



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

def teste(X,y,nome):


    split=int(
        0.7*len(y)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )


    return r2



print("\n================================")
print(" VELOCIDADE TEMPORAL")
print("================================")


resultados={}



# dTau/dt

X0=phi1.reshape(-1,1)

resultados["delta_tau_phi0"]=teste(
    X0,
    delta_tau,
    "phi0 -> Δtau"
)



X1=np.column_stack(
    [
        phi1,
        dphi
    ]
)


resultados["delta_tau_campo"]=teste(
    X1,
    delta_tau,
    "campo -> Δtau"
)



print("\n================================")
print(" ACELERAÇÃO TEMPORAL")
print("================================")


X2=phi2.reshape(-1,1)


resultados["delta2_tau_phi"]=teste(
    X2,
    delta2_tau,
    "phi -> Δ²tau"
)



# ------------------------------------------------------------
# Melhor
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=resultados.get
)



print("\n================================")
print(" MELHOR DINÂMICA")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]
)



resultado_dinamica_temporal={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_dinamica_temporal")


print("\n=== FIM S24-F.25 ===")

 IA-1 — S24-F.25
 DINÂMICA DIFERENCIAL DO TEMPO RELACIONAL


RuntimeError: Execute S24-F.23 antes deste bloco.

In [ ]:
# ============================================================
# IA-1 — S24-F.23-R
# RECONSTRUÇÃO DO CAMPO MODAL ORIENTADO
# ============================================================

import numpy as np

from sklearn.decomposition import PCA


print("="*60)
print(" IA-1 — S24-F.23-R")
print(" RECONSTRUÇÃO DO CAMPO MODAL ORIENTADO")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]

necessarias=[

    "eta_hist",
    "v_hist",
    "W_hist",
    "resultado_tempo_relacional"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)


if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Reconstrua os blocos anteriores."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.asarray(
    eta_hist
)


v=np.asarray(
    v_hist
)


W=np.asarray(
    W_hist
)



tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)



n=min(

    len(eta),
    len(v),
    len(W),
    len(tau)

)


eta=eta[:n]
v=v[:n]
W=W[:n]
tau=tau[:n]



# ------------------------------------------------------------
# Campo relacional combinado
# ------------------------------------------------------------

norma_eta=np.linalg.norm(
    eta,
    axis=1
)


norma_v=np.linalg.norm(
    v,
    axis=1
)


campo=np.column_stack(

    [

        norma_eta,

        norma_v,

        W

    ]

)



# normalização

campo=(

    campo-np.mean(
        campo,
        axis=0
    )

)/(
    np.std(
        campo,
        axis=0
    )+1e-12
)



# ------------------------------------------------------------
# Extração modal
# ------------------------------------------------------------

pca=PCA(
    n_components=3
)


phi=pca.fit_transform(
    campo
)



variancia=pca.explained_variance_ratio_



# ------------------------------------------------------------
# Orientação temporal
# ------------------------------------------------------------

correlacoes=[]


for i in range(phi.shape[1]):

    c=np.corrcoef(

        phi[:,i],

        tau

    )[0,1]

    correlacoes.append(c)



modo=np.argmax(
    np.abs(correlacoes)
)



phi_orientado=phi[:,modo]



# corrigir direção

if correlacoes[modo]<0:

    phi_orientado=-phi_orientado



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" MODOS PCA")
print("================================")


for i,x in enumerate(variancia):

    print(
        "Modo",
        i,
        ":",
        x
    )



print("\n================================")
print(" ORIENTAÇÃO TEMPORAL")
print("================================")


print(
    "Modo selecionado:",
    modo
)


print(
    "Correlação:",
    correlacoes[modo]
)



resultado_campo_modal_orientado={

    "phi_original":phi_orientado,

    "phi_completo":phi,

    "modo":modo,

    "variancia":variancia,

    "correlacoes":correlacoes

}



print("\nVariável criada:")
print("- resultado_campo_modal_orientado")


print("\n=== FIM S24-F.23-R ===")

 IA-1 — S24-F.23-R
 RECONSTRUÇÃO DO CAMPO MODAL ORIENTADO

 MODOS PCA
Modo 0 : 0.9755529579274013
Modo 1 : 0.02444633569361403
Modo 2 : 7.063789846803263e-07

 ORIENTAÇÃO TEMPORAL
Modo selecionado: 2
Correlação: 0.7414042558606164

Variável criada:
- resultado_campo_modal_orientado

=== FIM S24-F.23-R ===


In [ ]:
# ============================================================
# IA-1 — S24-F.42-R
# RECONSTRUÇÃO DO DETECTOR DE TRANSIÇÃO TEMPORAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.42-R")
print(" RECONSTRUÇÃO DO DETECTOR DE TRANSIÇÃO TEMPORAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_campo_modal_orientado",
    "resultado_tempo_relacional",
    "W_hist"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.23-R antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.asarray(

    resultado_campo_modal_orientado[
        "phi_original"
    ]

)


tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)


W=np.asarray(
    W_hist
)



n=min(

    len(phi),
    len(tau),
    len(W)

)


phi=phi[:n]
tau=tau[:n]
W=W[:n]



# ------------------------------------------------------------
# Gradientes relacionais
# ------------------------------------------------------------

dphi=np.diff(
    phi
)


dtau=np.diff(
    tau
)


dW=np.diff(
    W
)



# ------------------------------------------------------------
# Campo de transição corrigido
# ------------------------------------------------------------

if dW.ndim == 1:

    variacao_W = np.abs(dW)

else:

    variacao_W = np.linalg.norm(
        dW,
        axis=1
    )


Gamma=(

    np.abs(dphi)

    +

    np.abs(dtau)

    +

    variacao_W

)



# alinhamento

Gamma=np.pad(

    Gamma,

    (1,0),

    mode="edge"

)



# derivada do campo

dGamma=np.diff(
    Gamma
)



dGamma=np.pad(

    dGamma,

    (1,0),

    mode="edge"

)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

Gamma_norm=(

    Gamma-np.mean(Gamma)

)/(
    np.std(Gamma)+1e-12
)



dGamma_norm=(

    dGamma-np.mean(dGamma)

)/(
    np.std(dGamma)+1e-12
)



# ------------------------------------------------------------
# Detecção de eventos
# ------------------------------------------------------------

limiar=2.0


eventos=np.where(

    np.abs(dGamma_norm)>limiar

)[0]



# agrupar eventos próximos

grupos=[]


if len(eventos)>0:

    inicio=eventos[0]

    anterior=eventos[0]


    for e in eventos[1:]:

        if e-anterior>5:

            grupos.append(
                (
                    inicio,
                    anterior
                )
            )

            inicio=e

        anterior=e


    grupos.append(
        (
            inicio,
            anterior
        )
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" TRANSIÇÕES DETECTADAS")
print("================================")


print(
    "Número de pontos:",
    len(eventos)
)


print(
    "Número de grupos:",
    len(grupos)
)


for i,g in enumerate(grupos):

    print(
        "Evento",
        i,
        ":",
        g
    )



resultado_detector_transicao_temporal={

    "Gamma":Gamma,

    "Gamma_norm":Gamma_norm,

    "dGamma":dGamma,

    "dGamma_norm":dGamma_norm,

    "eventos":eventos,

    "grupos":grupos

}



print("\nVariável criada:")
print("- resultado_detector_transicao_temporal")


print("\n=== FIM S24-F.42-R ===")

 IA-1 — S24-F.42-R
 RECONSTRUÇÃO DO DETECTOR DE TRANSIÇÃO TEMPORAL

 TRANSIÇÕES DETECTADAS
Número de pontos: 178
Número de grupos: 1
Evento 0 : (np.int64(2822), np.int64(2999))

Variável criada:
- resultado_detector_transicao_temporal

=== FIM S24-F.42-R ===


In [ ]:
# ============================================================
# IA-1 — S24-F.47-R
# RECONSTRUÇÃO DOS EVENTOS DE TRANSIÇÃO TEMPORAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.47-R")
print(" RECONSTRUÇÃO DOS EVENTOS DE TRANSIÇÃO TEMPORAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_detector_transicao_temporal" not in globals():

    raise RuntimeError(
        "Execute S24-F.42-R antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

grupos = resultado_detector_transicao_temporal["grupos"]



if len(grupos)==0:

    raise RuntimeError(
        "Nenhuma transição detectada."
    )



# ------------------------------------------------------------
# Reconstrução dos eventos
# ------------------------------------------------------------

eventos=[]


for g in grupos:

    inicio=int(g[0])

    fim=int(g[1])

    duracao=fim-inicio+1


    eventos.append(

        {

        "inicio":inicio,

        "fim":fim,

        "duracao":duracao

        }

    )



# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

duracoes=np.array(

    [

    e["duracao"]

    for e in eventos

    ]

)



print("\n================================")
print(" EVENTOS RECONSTRUÍDOS")
print("================================")


print(
    "Número de eventos:",
    len(eventos)
)



for i,e in enumerate(eventos):

    print(

        "Evento",

        i,

        ":",

        e

    )



print("\n================================")
print(" ESTATÍSTICAS")
print("================================")


print(
    "Duração média:",
    np.mean(duracoes)
)


print(
    "Duração máxima:",
    np.max(duracoes)
)


# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_eventos_transicao={

    "eventos":eventos,

    "duracoes":duracoes

}



print("\nVariável criada:")
print("- resultado_eventos_transicao")


print("\n=== FIM S24-F.47-R ===")

 IA-1 — S24-F.47-R
 RECONSTRUÇÃO DOS EVENTOS DE TRANSIÇÃO TEMPORAL

 EVENTOS RECONSTRUÍDOS
Número de eventos: 1
Evento 0 : {'inicio': 2822, 'fim': 2999, 'duracao': 178}

 ESTATÍSTICAS
Duração média: 178.0
Duração máxima: 178

Variável criada:
- resultado_eventos_transicao

=== FIM S24-F.47-R ===


In [ ]:
# ============================================================
# IA-1 — S24-F.53-R v3
# ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.53-R v3")
print(" ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_detector_transicao_temporal",

    "resultado_eventos_transicao"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("Variáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Reconstrua F.42-R e F.47-R."
    )



# ------------------------------------------------------------
# Dados pós-transição
# ------------------------------------------------------------

dGamma=np.asarray(

    resultado_detector_transicao_temporal[
        "dGamma"
    ]

)


evento=resultado_eventos_transicao["eventos"][0]


inicio=int(
    evento["fim"]
)


M=dGamma[inicio:]



if len(M)<20:

    raise RuntimeError(
        "Regime pós-transição insuficiente."
    )



# ------------------------------------------------------------
# Tendência global
# ------------------------------------------------------------

t=np.arange(len(M))


modelo=LinearRegression()


modelo.fit(

    t.reshape(-1,1),

    M

)


pred=modelo.predict(

    t.reshape(-1,1)

)


r2=r2_score(

    M,

    pred

)


inclinacao=modelo.coef_[0]



# ------------------------------------------------------------
# Estatística em blocos
# ------------------------------------------------------------

n_blocos=5


blocos=np.array_split(

    M,

    n_blocos

)


medias=np.array(

    [

    np.mean(b)

    for b in blocos

    ]

)


desvios=np.array(

    [

    np.std(b)

    for b in blocos

    ]

)



# ------------------------------------------------------------
# Estimativa do atrator
# ------------------------------------------------------------

atrator=medias[-1]


delta=abs(

    medias[-1]

    -

    medias[-2]

)



indice_estabilidade=(

    delta

    /

    (desvios[-1]+1e-12)

)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if indice_estabilidade < 0.10:

    situacao="atrator_estavel"


elif indice_estabilidade < 0.50:

    situacao="convergencia_avancada"


else:

    situacao="ainda_convergindo"



# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print("\n================================")
print(" REGIME PÓS-TRANSIÇÃO")
print("================================")


print(
    "Pontos analisados:",
    len(M)
)


print(
    "Início:",
    inicio
)



print("\n================================")
print(" TENDÊNCIA")
print("================================")


print(
    "R²:",
    r2
)


print(
    "Inclinação:",
    inclinacao
)



print("\n================================")
print(" BLOCOS")
print("================================")


for i in range(n_blocos):

    print(

        "Bloco",

        i,

        "| média:",

        medias[i],

        "| desvio:",

        desvios[i]

    )



print("\n================================")
print(" ATRATOR")
print("================================")


print(
    "Valor estimado:",
    atrator
)


print(
    "Diferença últimos blocos:",
    delta
)


print(
    "Índice estabilidade:",
    indice_estabilidade
)


print(
    "Situação:",
    situacao
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_atrator_memoria={

    "atrator":atrator,

    "medias_blocos":medias,

    "desvios_blocos":desvios,

    "delta":delta,

    "indice_estabilidade":indice_estabilidade,

    "r2_tendencia":r2,

    "inclinacao":inclinacao,

    "situacao":situacao

}



print("\nVariável criada:")
print("- resultado_atrator_memoria")


print("\n=== FIM S24-F.53-R v3 ===")

 IA-1 — S24-F.53-R v3
 ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL


RuntimeError: Regime pós-transição insuficiente.

In [ ]:
# ============================================================
# IA-1 — S24-F.53-R v4
# ATRATOR RELACIONAL NO REGIME DE FECHAMENTO
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.53-R v4")
print(" ATRATOR RELACIONAL NO REGIME DE FECHAMENTO")
print("="*60)


dGamma=np.asarray(
    resultado_detector_transicao_temporal["dGamma"]
)


evento=resultado_eventos_transicao["eventos"][0]


inicio=int(evento["inicio"])
fim=int(evento["fim"])


# usa o próprio evento
M=dGamma[inicio:fim+1]


if len(M)<20:
    raise RuntimeError(
        "Evento muito curto para análise."
    )


print("\n================================")
print(" REGIME ANALISADO")
print("================================")

print("Início:", inicio)
print("Fim:", fim)
print("Pontos:", len(M))


# tendência

t=np.arange(len(M))


modelo=LinearRegression()

modelo.fit(
    t.reshape(-1,1),
    M
)


pred=modelo.predict(
    t.reshape(-1,1)
)


r2=r2_score(
    M,
    pred
)


incl=modelo.coef_[0]


# blocos

blocos=np.array_split(
    M,
    5
)


medias=np.array(
    [
        np.mean(b)
        for b in blocos
    ]
)


desvios=np.array(
    [
        np.std(b)
        for b in blocos
    ]
)


atrator=medias[-1]


delta=abs(
    medias[-1]-medias[-2]
)


indice=delta/(desvios[-1]+1e-12)


if indice < 0.10:
    situacao="atrator_estavel"

elif indice < 0.50:
    situacao="convergencia_avancada"

else:
    situacao="ainda_convergindo"



print("\n================================")
print(" TENDÊNCIA")
print("================================")

print("R²:",r2)
print("Inclinação:",incl)


print("\n================================")
print(" ATRATOR")
print("================================")

print("Valor:",atrator)
print("Delta:",delta)
print("Índice:",indice)
print("Situação:",situacao)



resultado_atrator_memoria={

    "atrator":atrator,

    "medias_blocos":medias,

    "desvios_blocos":desvios,

    "delta":delta,

    "indice_estabilidade":indice,

    "r2_tendencia":r2,

    "inclinacao":incl,

    "situacao":situacao

}


print("\nVariável criada:")
print("- resultado_atrator_memoria")


print("\n=== FIM S24-F.53-R v4 ===")

 IA-1 — S24-F.53-R v4
 ATRATOR RELACIONAL NO REGIME DE FECHAMENTO

 REGIME ANALISADO
Início: 2822
Fim: 2999
Pontos: 178

 TENDÊNCIA
R²: 0.9862506556056332
Inclinação: 0.04889429313272845

 ATRATOR
Valor: 13.47233801677295
Delta: 2.2195999376388365
Índice: 3.1716178541937463
Situação: ainda_convergindo

Variável criada:
- resultado_atrator_memoria

=== FIM S24-F.53-R v4 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.54-R
# CARACTERIZAÇÃO DO ATRATOR RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.54-R")
print(" CARACTERIZAÇÃO DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_atrator_memoria",

    "resultado_detector_transicao_temporal",

    "resultado_eventos_transicao",

    "resultado_invariantes",

    "resultado_tempo_relacional"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute os blocos anteriores."
    )



# ------------------------------------------------------------
# Dados do atrator
# ------------------------------------------------------------

atrator = resultado_atrator_memoria["atrator"]

indice = resultado_atrator_memoria[
    "indice_estabilidade"
]

r2 = resultado_atrator_memoria[
    "r2_tendencia"
]

incl = resultado_atrator_memoria[
    "inclinacao"
]


situacao = resultado_atrator_memoria[
    "situacao"
]



# ------------------------------------------------------------
# Evento
# ------------------------------------------------------------

evento = resultado_eventos_transicao[
    "eventos"
][0]


duracao = evento["duracao"]



# ------------------------------------------------------------
# Invariantes finais
# ------------------------------------------------------------

energia_final = resultado_invariantes[
    "energia"
][-1]


escala_final = resultado_invariantes[
    "escala"
][-1]


curvatura_final = resultado_invariantes[
    "curvatura"
][-1]



# ------------------------------------------------------------
# Tempo relacional
# ------------------------------------------------------------

tau = np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)


tau_final = tau[-1]



# ------------------------------------------------------------
# Índice composto de fechamento
# ------------------------------------------------------------

coerencia = (

    abs(r2)

    *

    abs(np.tanh(incl))

)


fechamento = (

    coerencia

    /

    (1 + indice)

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" ATRATOR")
print("================================")


print(
    "Valor:",
    atrator
)


print(
    "Situação:",
    situacao
)


print(
    "Índice estabilidade:",
    indice
)



print("\n================================")
print(" TRANSIÇÃO")
print("================================")


print(
    "Duração:",
    duracao
)


print(
    "R² tendência:",
    r2
)


print(
    "Inclinação:",
    incl
)



print("\n================================")
print(" ESTADO FINAL")
print("================================")


print(
    "Energia:",
    energia_final
)


print(
    "Escala:",
    escala_final
)


print(
    "Curvatura:",
    curvatura_final
)


print(
    "Tau final:",
    tau_final
)



print("\n================================")
print(" ÍNDICE DE FECHAMENTO")
print("================================")


print(
    "Coerência:",
    coerencia
)


print(
    "Índice composto:",
    fechamento
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_caracterizacao_atrator={

    "atrator":atrator,

    "situacao":situacao,

    "indice_estabilidade":indice,

    "r2_tendencia":r2,

    "inclinacao":incl,

    "duracao_transicao":duracao,

    "energia_final":energia_final,

    "escala_final":escala_final,

    "curvatura_final":curvatura_final,

    "tau_final":tau_final,

    "indice_fechamento":fechamento

}



print("\nVariável criada:")
print("- resultado_caracterizacao_atrator")


print("\n=== FIM S24-F.54-R ===")

 IA-1 — S24-F.54-R
 CARACTERIZAÇÃO DO ATRATOR RELACIONAL

 ATRATOR
Valor: 13.47233801677295
Situação: ainda_convergindo
Índice estabilidade: 3.1716178541937463

 TRANSIÇÃO
Duração: 178
R² tendência: 0.9862506556056332
Inclinação: 0.04889429313272845

 ESTADO FINAL
Energia: -220299.45369320302
Escala: 0.0031329628062978783
Curvatura: 0.0064694352384549805
Tau final: 0.3947647080834435

 ÍNDICE DE FECHAMENTO
Coerência: 0.04818363800708956
Índice composto: 0.011550348016333837

Variável criada:
- resultado_caracterizacao_atrator

=== FIM S24-F.54-R ===


In [ ]:
# ============================================================
# IA-1 — S24-F.55
# MAPA DE ESTABILIDADE DO ATRATOR RELACIONAL
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.55")
print(" MAPA DE ESTABILIDADE DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]

necessarias=[

    "resultado_detector_transicao_temporal",

    "resultado_eventos_transicao",

    "resultado_atrator_memoria",

    "resultado_caracterizacao_atrator"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.53-R e F.54-R antes deste bloco."
    )



# ------------------------------------------------------------
# Extrair regime de fechamento
# ------------------------------------------------------------

dGamma=np.asarray(

    resultado_detector_transicao_temporal[
        "dGamma"
    ]

)


evento=resultado_eventos_transicao["eventos"][0]


inicio=int(evento["inicio"])
fim=int(evento["fim"])


M=dGamma[inicio:fim+1]



n=len(M)

t=np.arange(n)



# ------------------------------------------------------------
# Distância relativa ao último estado
# ------------------------------------------------------------

valor_final=M[-1]


distancia=np.abs(

    M-valor_final

)



# evitar zero artificial

distancia=distancia+1e-12



log_dist=np.log(
    distancia
)



# ------------------------------------------------------------
# Taxa de convergência
# ------------------------------------------------------------

modelo=LinearRegression()


modelo.fit(

    t.reshape(-1,1),

    log_dist

)


log_pred=modelo.predict(

    t.reshape(-1,1)

)


r2=r2_score(

    log_dist,

    log_pred

)


taxa=modelo.coef_[0]



# ------------------------------------------------------------
# Oscilação final
# ------------------------------------------------------------

ultimos=max(20,n//5)


final=M[-ultimos:]


variacao_final=np.std(final)


media_final=np.mean(final)



# ------------------------------------------------------------
# Horizonte estimado
# ------------------------------------------------------------

if taxa < 0:

    horizonte=-1/taxa

else:

    horizonte=np.inf



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if taxa < 0 and r2 > 0.8:

    estado="convergencia_exponencial"

elif taxa < 0:

    estado="convergencia_fraca"

else:

    estado="sem_sinal_de_convergencia"



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" REGIME")
print("================================")

print("Pontos:",n)
print("Início:",inicio)
print("Fim:",fim)



print("\n================================")
print(" CONVERGÊNCIA")
print("================================")

print("Taxa:",taxa)
print("R²:",r2)
print("Horizonte estimado:",horizonte)



print("\n================================")
print(" ESTADO FINAL")
print("================================")

print("Média final:",media_final)
print("Desvio final:",variacao_final)



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print("Estado:",estado)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_estabilidade_atrator={

    "taxa_convergencia":taxa,

    "r2_convergencia":r2,

    "horizonte_estimado":horizonte,

    "media_final":media_final,

    "desvio_final":variacao_final,

    "estado":estado,

    "distancia":distancia

}



print("\nVariável criada:")
print("- resultado_estabilidade_atrator")


print("\n=== FIM S24-F.55 ===")

 IA-1 — S24-F.55
 MAPA DE ESTABILIDADE DO ATRATOR RELACIONAL

 REGIME
Pontos: 178
Início: 2822
Fim: 2999

 CONVERGÊNCIA
Taxa: -0.01931537859381249
R²: 0.18209471106073982
Horizonte estimado: 51.77221844982842

 ESTADO FINAL
Média final: 13.47233801677295
Desvio final: 0.6998320856028561

 CLASSIFICAÇÃO
Estado: convergencia_fraca

Variável criada:
- resultado_estabilidade_atrator

=== FIM S24-F.55 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.56
# ANÁLISE DE ESCALA DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.56")
print(" ANÁLISE DE ESCALA DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_detector_transicao_temporal",

    "resultado_eventos_transicao",

    "resultado_atrator_memoria"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.53-R antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

dGamma=np.asarray(

    resultado_detector_transicao_temporal[
        "dGamma"
    ]

)


evento=resultado_eventos_transicao[
    "eventos"
][0]


inicio=int(evento["inicio"])
fim=int(evento["fim"])


M=dGamma[inicio:fim+1]


# estado alvo

alvo=np.mean(M[-20:])


dist=np.abs(
    M-alvo
)


# remover zeros

mask=dist>1e-12


dist=dist[mask]


t=np.arange(
    1,
    len(dist)+1
)



# ------------------------------------------------------------
# Modelo exponencial
# log(d)=log(A)-kt
# ------------------------------------------------------------

log_d=np.log(dist)


modelo_exp=LinearRegression()


modelo_exp.fit(

    t.reshape(-1,1),

    log_d

)


pred_exp=modelo_exp.predict(

    t.reshape(-1,1)

)


r2_exp=r2_score(

    log_d,

    pred_exp

)


k=-modelo_exp.coef_[0]



# ------------------------------------------------------------
# Modelo potência
# log(d)=log(A)-p log(t)
# ------------------------------------------------------------

log_t=np.log(t)


modelo_pow=LinearRegression()


modelo_pow.fit(

    log_t.reshape(-1,1),

    log_d

)


pred_pow=modelo_pow.predict(

    log_t.reshape(-1,1)

)


r2_pow=r2_score(

    log_d,

    pred_pow

)


p=-modelo_pow.coef_[0]



# ------------------------------------------------------------
# Comparação
# ------------------------------------------------------------

if r2_exp > r2_pow:

    melhor="exponencial"

else:

    melhor="potencia"



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" MODELO EXPONENCIAL")
print("================================")

print(
    "R²:",
    r2_exp
)

print(
    "k:",
    k
)



print("\n================================")
print(" MODELO POTÊNCIA")
print("================================")

print(
    "R²:",
    r2_pow
)

print(
    "expoente:",
    p
)



print("\n================================")
print(" MODELO DOMINANTE")
print("================================")

print(
    "Melhor:",
    melhor
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_escala_atrator={

    "r2_exponencial":r2_exp,

    "taxa_exponencial":k,

    "r2_potencia":r2_pow,

    "expoente_potencia":p,

    "modelo":melhor

}



print("\nVariável criada:")
print("- resultado_escala_atrator")


print("\n=== FIM S24-F.56 ===")

 IA-1 — S24-F.56
 ANÁLISE DE ESCALA DO ATRATOR RELACIONAL

 MODELO EXPONENCIAL
R²: 0.7100700521579824
k: 0.017482101875391596

 MODELO POTÊNCIA
R²: 0.3976896213153528
expoente: 0.709076373648681

 MODELO DOMINANTE
Melhor: exponencial

Variável criada:
- resultado_escala_atrator

=== FIM S24-F.56 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.57
# TEMPO DE RELAXAÇÃO DO ATRATOR RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.57")
print(" TEMPO DE RELAXAÇÃO DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_escala_atrator",

    "resultado_eventos_transicao",

    "resultado_atrator_memoria"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.56 antes deste bloco."
    )



# ------------------------------------------------------------
# Parâmetro de relaxação
# ------------------------------------------------------------

k = resultado_escala_atrator[

    "taxa_exponencial"

]



if k <= 0:

    raise RuntimeError(
        "Taxa exponencial inválida."
    )



tau_relax = 1/k



# ------------------------------------------------------------
# Tempos de aproximação
# ------------------------------------------------------------

def tempo_fracao(fracao):

    return -np.log(
        1-fracao
    )/k



t90 = tempo_fracao(0.90)

t95 = tempo_fracao(0.95)

t99 = tempo_fracao(0.99)



# ------------------------------------------------------------
# Relação com evento observado
# ------------------------------------------------------------

duracao = resultado_eventos_transicao[

    "eventos"

][0]["duracao"]



fracao_observada = 1-np.exp(

    -k*duracao

)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" CONSTANTE DE RELAXAÇÃO")
print("================================")


print(
    "k:",
    k
)


print(
    "Tempo característico:",
    tau_relax
)



print("\n================================")
print(" TEMPOS DE APROXIMAÇÃO")
print("================================")


print(
    "90%:",
    t90
)


print(
    "95%:",
    t95
)


print(
    "99%:",
    t99
)



print("\n================================")
print(" EVENTO OBSERVADO")
print("================================")


print(
    "Duração evento:",
    duracao
)


print(
    "Relaxação acumulada:",
    fracao_observada
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if fracao_observada >= 0.99:

    estado="praticamente_saturado"

elif fracao_observada >= 0.95:

    estado="proximo_da_saturacao"

elif fracao_observada >= 0.90:

    estado="relaxacao_avancada"

else:

    estado="relaxacao_incompleta"



print("\n================================")
print(" ESTADO")
print("================================")

print(
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_tempo_relaxacao_atrator={

    "k":k,

    "tau_relax":tau_relax,

    "t90":t90,

    "t95":t95,

    "t99":t99,

    "duracao_evento":duracao,

    "fracao_relaxada":fracao_observada,

    "estado":estado

}



print("\nVariável criada:")
print("- resultado_tempo_relaxacao_atrator")


print("\n=== FIM S24-F.57 ===")

 IA-1 — S24-F.57
 TEMPO DE RELAXAÇÃO DO ATRATOR RELACIONAL

 CONSTANTE DE RELAXAÇÃO
k: 0.017482101875391596
Tempo característico: 57.20135983234569

 TEMPOS DE APROXIMAÇÃO
90%: 131.71099844894758
95%: 171.35995974093282
99%: 263.4219968978951

 EVENTO OBSERVADO
Duração evento: 178
Relaxação acumulada: 0.9554798834128708

 ESTADO
proximo_da_saturacao

Variável criada:
- resultado_tempo_relaxacao_atrator

=== FIM S24-F.57 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.58
# EXTRAPOLAÇÃO DO HORIZONTE DE FECHAMENTO RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.58")
print(" EXTRAPOLAÇÃO DO HORIZONTE DE FECHAMENTO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_escala_atrator",

    "resultado_tempo_relaxacao_atrator",

    "resultado_atrator_memoria",

    "resultado_eventos_transicao"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.53 até F.57 antes deste bloco."
    )



# ------------------------------------------------------------
# Parâmetros
# ------------------------------------------------------------

k = resultado_escala_atrator[

    "taxa_exponencial"

]


atrator_obs = resultado_atrator_memoria[

    "atrator"

]


duracao = resultado_eventos_transicao[

    "eventos"

][0]["duracao"]



# ------------------------------------------------------------
# Horizonte de saturação
# ------------------------------------------------------------

def horizonte(fracao):

    return -np.log(
        1-fracao
    )/k



h95=horizonte(0.95)

h99=horizonte(0.99)

h999=horizonte(0.999)



faltam95=max(
    0,
    h95-duracao
)


faltam99=max(
    0,
    h99-duracao
)


faltam999=max(
    0,
    h999-duracao
)



# ------------------------------------------------------------
# Extrapolação simples
# ------------------------------------------------------------

relax_atual = 1-np.exp(

    -k*duracao

)


erro_restante = 1-relax_atual



atrator_extrapolado = (

    atrator_obs

    *

    (1 + erro_restante)

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" HORIZONTE")
print("================================")


print(
    "Tempo 95%:",
    h95
)


print(
    "Tempo 99%:",
    h99
)


print(
    "Tempo 99.9%:",
    h999
)



print("\n================================")
print(" PONTOS RESTANTES")
print("================================")


print(
    "Até 95%:",
    faltam95
)


print(
    "Até 99%:",
    faltam99
)


print(
    "Até 99.9%:",
    faltam999
)



print("\n================================")
print(" ATRATOR")
print("================================")


print(
    "Observado:",
    atrator_obs
)


print(
    "Relaxação atual:",
    relax_atual
)


print(
    "Erro restante:",
    erro_restante
)


print(
    "Extrapolado:",
    atrator_extrapolado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_horizonte_fechamento={

    "k":k,

    "tempo_95":h95,

    "tempo_99":h99,

    "tempo_999":h999,

    "faltam_95":faltam95,

    "faltam_99":faltam99,

    "faltam_999":faltam999,

    "atrator_observado":atrator_obs,

    "atrator_extrapolado":atrator_extrapolado,

    "erro_restante":erro_restante

}


print("\nVariável criada:")
print("- resultado_horizonte_fechamento")


print("\n=== FIM S24-F.58 ===")

 IA-1 — S24-F.58
 EXTRAPOLAÇÃO DO HORIZONTE DE FECHAMENTO RELACIONAL

 HORIZONTE
Tempo 95%: 171.35995974093282
Tempo 99%: 263.4219968978951
Tempo 99.9%: 395.1329953468426

 PONTOS RESTANTES
Até 95%: 0
Até 99%: 85.4219968978951
Até 99.9%: 217.13299534684262

 ATRATOR
Observado: 13.47233801677295
Relaxação atual: 0.9554798834128708
Erro restante: 0.04452011658712918
Extrapolado: 14.072128075980894

Variável criada:
- resultado_horizonte_fechamento

=== FIM S24-F.58 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.59
# VALIDAÇÃO DO ATRATOR CONTRA INVARIANTES RELACIONAIS
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.59")
print(" VALIDAÇÃO DO ATRATOR CONTRA INVARIANTES RELACIONAIS")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_horizonte_fechamento",

    "resultado_atrator_memoria",

    "resultado_invariantes",

    "resultado_tempo_relacional",

    "resultado_campo_modal_orientado"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute os blocos F.53 até F.58."
    )



# ------------------------------------------------------------
# Atratores
# ------------------------------------------------------------

atrator_obs = resultado_atrator_memoria[

    "atrator"

]


atrator_ext = resultado_horizonte_fechamento[

    "atrator_extrapolado"

]



# ------------------------------------------------------------
# Invariantes finais
# ------------------------------------------------------------

energia=np.asarray(

    resultado_invariantes[
        "energia"
    ]

)


momento=np.asarray(

    resultado_invariantes[
        "momento"
    ]

)


escala=np.asarray(

    resultado_invariantes[
        "escala"
    ]

)


curvatura=np.asarray(

    resultado_invariantes[
        "curvatura"
    ]

)



tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)



# ------------------------------------------------------------
# Alinhamento
# ------------------------------------------------------------

n=min(

    len(energia),

    len(tau),

    len(curvatura)

)


energia=energia[-n:]
momento=momento[-n:]
escala=escala[-n:]
curvatura=curvatura[-n:]
tau=tau[-n:]



# ------------------------------------------------------------
# Correlações finais
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES COM TAU")
print("================================")


correlacoes={}


for nome,x in {

    "energia":energia,

    "momento":momento,

    "escala":escala,

    "curvatura":curvatura

}.items():


    c=np.corrcoef(

        x,

        tau

    )[0,1]


    correlacoes[nome]=c


    print(
        nome,
        ":",
        c
    )



# ------------------------------------------------------------
# Distância ao atrator
# ------------------------------------------------------------

dist_obs=np.abs(

    tau-atrator_obs

)


dist_ext=np.abs(

    tau-atrator_ext

)


media_obs=np.mean(dist_obs)

media_ext=np.mean(dist_ext)



# ------------------------------------------------------------
# Modelo dos invariantes prevendo tau
# ------------------------------------------------------------

X=np.column_stack(

    [

    energia,

    momento,

    escala,

    curvatura

    ]

)


split=int(
    0.7*len(X)
)



modelo=LinearRegression()


modelo.fit(

    X[:split],

    tau[:split]

)



pred=modelo.predict(

    X[split:]

)


r2=r2_score(

    tau[split:],

    pred

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" ATRATOR")
print("================================")


print(
    "Observado:",
    atrator_obs
)


print(
    "Extrapolado:",
    atrator_ext
)


print(
    "Distância média observado:",
    media_obs
)


print(
    "Distância média extrapolado:",
    media_ext
)



print("\n================================")
print(" MODELO INVARIANTES")
print("================================")


print(
    "R² invariantes -> tau:",
    r2
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_validacao_atrator={

    "atrator_observado":atrator_obs,

    "atrator_extrapolado":atrator_ext,

    "correlacoes":correlacoes,

    "distancia_observado":media_obs,

    "distancia_extrapolado":media_ext,

    "r2_invariantes_tau":r2

}



print("\nVariável criada:")
print("- resultado_validacao_atrator")


print("\n=== FIM S24-F.59 ===")

 IA-1 — S24-F.59
 VALIDAÇÃO DO ATRATOR CONTRA INVARIANTES RELACIONAIS

 CORRELAÇÕES COM TAU
energia : -0.22777434186501325
momento : 0.22839023856389604
escala : -0.9271570734746143
curvatura : 0.9615235186875263

 ATRATOR
Observado: 13.47233801677295
Extrapolado: 14.072128075980894
Distância média observado: 13.13653582809602
Distância média extrapolado: 13.736325887303964

 MODELO INVARIANTES
R² invariantes -> tau: -8335497.876119021

Variável criada:
- resultado_validacao_atrator

=== FIM S24-F.59 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.60
# GEOMETRIA DO ATRATOR RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.60")
print(" GEOMETRIA DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_invariantes",

    "resultado_tempo_relacional",

    "resultado_horizonte_fechamento",

    "resultado_atrator_memoria"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute os blocos anteriores."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

escala=np.asarray(

    resultado_invariantes["escala"]

)


curvatura=np.asarray(

    resultado_invariantes["curvatura"]

)


tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)



n=min(

    len(escala),

    len(curvatura),

    len(tau)

)


escala=escala[-n:]

curvatura=curvatura[-n:]

tau=tau[-n:]



# ------------------------------------------------------------
# Normalização geométrica
# ------------------------------------------------------------

def normalizar(x):

    return (

        x-np.mean(x)

    )/(np.std(x)+1e-12)



E=np.column_stack(

    [

    normalizar(escala),

    normalizar(curvatura),

    normalizar(tau)

    ]

)



# ------------------------------------------------------------
# Centro final do atrator
# ------------------------------------------------------------

janela=max(

    20,

    n//10

)


estado_final=np.mean(

    E[-janela:],

    axis=0

)



distancias=np.linalg.norm(

    E-estado_final,

    axis=1

)



# ------------------------------------------------------------
# Compressão final
# ------------------------------------------------------------

inicio=np.mean(

    distancias[:janela]

)


fim=np.mean(

    distancias[-janela:]

)


compressao=1-(fim/(inicio+1e-12))



# ------------------------------------------------------------
# Variação final
# ------------------------------------------------------------

variacao_final=np.std(

    distancias[-janela:]

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" ESTADO GEOMÉTRICO FINAL")
print("================================")


print(
    "Escala:",
    estado_final[0]
)


print(
    "Curvatura:",
    estado_final[1]
)


print(
    "Tau:",
    estado_final[2]
)



print("\n================================")
print(" DISTÂNCIA AO ATRATOR")
print("================================")


print(
    "Distância inicial média:",
    inicio
)


print(
    "Distância final média:",
    fim
)


print(
    "Compressão:",
    compressao
)



print("\n================================")
print(" ESTABILIDADE")
print("================================")


print(
    "Variação final:",
    variacao_final
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if compressao > 0.90 and variacao_final < 0.10:

    estado="atrator_geometrico_estavel"


elif compressao > 0.50:

    estado="convergencia_geometrica"


else:

    estado="trajetoria_aberta"



print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_geometria_atrator={

    "estado_final":estado_final,

    "distancias":distancias,

    "compressao":compressao,

    "distancia_inicial":inicio,

    "distancia_final":fim,

    "variacao_final":variacao_final,

    "estado":estado

}



print("\nVariável criada:")
print("- resultado_geometria_atrator")


print("\n=== FIM S24-F.60 ===")

 IA-1 — S24-F.60
 GEOMETRIA DO ATRATOR RELACIONAL

 ESTADO GEOMÉTRICO FINAL
Escala: -0.8753716624210768
Curvatura: 0.9142023486754575
Tau: 0.5823096729146081

 DISTÂNCIA AO ATRATOR
Distância inicial média: 5.071311841676788
Distância final média: 0.017555801951472397
Compressão: 0.9965382128925315

 ESTABILIDADE
Variação final: 0.010129325236810595
Estado: atrator_geometrico_estavel

Variável criada:
- resultado_geometria_atrator

=== FIM S24-F.60 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.61
# DIMENSÃO EFETIVA DO ATRATOR RELACIONAL
# ============================================================

import numpy as np
from sklearn.decomposition import PCA


print("="*60)
print(" IA-1 — S24-F.61")
print(" DIMENSÃO EFETIVA DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_invariantes",

    "resultado_tempo_relacional",

    "resultado_geometria_atrator"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.60 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

escala=np.asarray(

    resultado_invariantes["escala"]

)


curvatura=np.asarray(

    resultado_invariantes["curvatura"]

)


tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)



n=min(

    len(escala),

    len(curvatura),

    len(tau)

)


X=np.column_stack(

    [

    escala[-n:],

    curvatura[-n:],

    tau[-n:]

    ]

)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

X=(

    X-np.mean(X,axis=0)

)/(np.std(X,axis=0)+1e-12)



# ------------------------------------------------------------
# PCA global
# ------------------------------------------------------------

pca=PCA(
    n_components=3
)


Z=pca.fit_transform(X)



variancias=pca.explained_variance_ratio_



# ------------------------------------------------------------
# PCA final
# ------------------------------------------------------------

janela=max(

    30,

    n//10

)


X_final=X[-janela:]



pca_final=PCA(
    n_components=3
)


Zf=pca_final.fit_transform(

    X_final

)



variancias_final=pca_final.explained_variance_ratio_



# ------------------------------------------------------------
# Entropia modal
# ------------------------------------------------------------

def entropia_modal(v):

    v=v[v>0]

    return -np.sum(

        v*np.log(v)

    )



H_global=entropia_modal(

    variancias

)


H_final=entropia_modal(

    variancias_final

)



dim_global=np.exp(

    H_global

)


dim_final=np.exp(

    H_final

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" PCA GLOBAL")
print("================================")


for i,v in enumerate(variancias):

    print(
        "Modo",
        i,
        ":",
        v
    )



print("\nDimensão efetiva global:")

print(
    dim_global
)



print("\n================================")
print(" PCA FINAL")
print("================================")


for i,v in enumerate(variancias_final):

    print(
        "Modo",
        i,
        ":",
        v
    )


print("\nDimensão efetiva final:")

print(
    dim_final
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if dim_final < 1.3:

    estado="atrator_unidimensional"

elif dim_final < 2.2:

    estado="atrator_baixa_dimensao"

else:

    estado="atrator_multidimensional"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_dimensao_atrator={

    "variancia_global":variancias,

    "variancia_final":variancias_final,

    "entropia_global":H_global,

    "entropia_final":H_final,

    "dimensao_global":dim_global,

    "dimensao_final":dim_final,

    "estado":estado

}



print("\nVariável criada:")
print("- resultado_dimensao_atrator")


print("\n=== FIM S24-F.61 ===")

 IA-1 — S24-F.61
 DIMENSÃO EFETIVA DO ATRATOR RELACIONAL

 PCA GLOBAL
Modo 0 : 0.9733347012200189
Modo 1 : 0.025630761800003194
Modo 2 : 0.0010345369799779737

Dimensão efetiva global:
1.1357894004746574

 PCA FINAL
Modo 0 : 0.9998538018971295
Modo 1 : 0.00014619683354251474
Modo 2 : 1.2693279494137334e-09

Dimensão efetiva final:
1.0014382461261075

 CLASSIFICAÇÃO
Estado: atrator_unidimensional

Variável criada:
- resultado_dimensao_atrator

=== FIM S24-F.61 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.62
# LEI DE EVOLUÇÃO DO ATRATOR UNIDIMENSIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score



print("="*60)
print(" IA-1 — S24-F.62")
print(" LEI DE EVOLUÇÃO DO ATRATOR UNIDIMENSIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


necessarias=[

    "resultado_dimensao_atrator",

    "resultado_geometria_atrator",

    "resultado_invariantes",

    "resultado_tempo_relacional"

]


for x in necessarias:

    if x not in globals():

        faltando.append(x)



if faltando:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute F.60 e F.61 antes."
    )



# ------------------------------------------------------------
# Espaço geométrico
# ------------------------------------------------------------

escala=np.asarray(

    resultado_invariantes["escala"]

)


curvatura=np.asarray(

    resultado_invariantes["curvatura"]

)


tau=np.asarray(

    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]

)



n=min(

    len(escala),

    len(curvatura),

    len(tau)

)



X=np.column_stack(

    [

    escala[-n:],

    curvatura[-n:],

    tau[-n:]

    ]

)



X=(

    X-np.mean(X,axis=0)

)/(np.std(X,axis=0)+1e-12)



# ------------------------------------------------------------
# Coordenada dominante
# ------------------------------------------------------------

pca=PCA(
    n_components=1
)


z=pca.fit_transform(X).flatten()



# regime final

inicio=max(

    0,

    len(z)-178

)


z=z[inicio:]



t=np.arange(len(z))



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------


resultados={}



# linear

modelo=LinearRegression()

modelo.fit(

    t.reshape(-1,1),

    z

)


pred=modelo.predict(

    t.reshape(-1,1)

)


resultados["linear"]=r2_score(

    z,

    pred

)



# exponencial

z_shift=z-np.min(z)+1e-12


logz=np.log(z_shift)


modelo_exp=LinearRegression()

modelo_exp.fit(

    t.reshape(-1,1),

    logz

)


pred_exp=np.exp(

    modelo_exp.predict(
        t.reshape(-1,1)
    )

)+np.min(z)-1e-12


resultados["exponencial"]=r2_score(

    z,

    pred_exp

)



k_exp=-modelo_exp.coef_[0]



# potência

logt=np.log(

    t+1

)


modelo_pow=LinearRegression()

modelo_pow.fit(

    logt.reshape(-1,1),

    logz

)


pred_pow=np.exp(

    modelo_pow.predict(
        logt.reshape(-1,1)
    )

)+np.min(z)-1e-12


resultados["potencia"]=r2_score(

    z,

    pred_pow

)



expo=modelo_pow.coef_[0]



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

melhor=max(

    resultados,

    key=resultados.get

)



print("\n================================")
print(" MODELOS")
print("================================")


for k,v in resultados.items():

    print(
        k,
        ":",
        v
    )



print("\n================================")
print(" LEI DOMINANTE")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "k exponencial:",
    k_exp
)


print(
    "expoente potência:",
    expo
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_lei_atrator={

    "coordenada":z,

    "R2_modelos":resultados,

    "modelo_dominante":melhor,

    "k_exponencial":k_exp,

    "expoente_potencia":expo

}



print("\nVariável criada:")
print("- resultado_lei_atrator")


print("\n=== FIM S24-F.62 ===")

 IA-1 — S24-F.62
 LEI DE EVOLUÇÃO DO ATRATOR UNIDIMENSIONAL

Variáveis ausentes:
- resultado_dimensao_atrator
- resultado_geometria_atrator
- resultado_invariantes
- resultado_tempo_relacional


RuntimeError: Execute F.60 e F.61 antes.

In [ ]:
# ============================================================
# IA-1 — S24-F.63
# INVARIANTE DINÂMICO DO ATRATOR RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.63")
print(" INVARIANTE DINÂMICO DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]


for x in [

    "resultado_lei_atrator",

    "resultado_dimensao_atrator"

]:

    if x not in globals():

        faltando.append(x)



if faltando:

    print("\nVariáveis ausentes:")

    for x in faltando:

        print("-",x)

    raise RuntimeError(
        "Execute F.62 e F.61 antes."
    )



# ------------------------------------------------------------
# Coordenada dominante
# ------------------------------------------------------------

z=np.asarray(

    resultado_lei_atrator["coordenada"]

)



# ------------------------------------------------------------
# Dinâmica interna
# ------------------------------------------------------------

vel=np.diff(z)


acc=np.diff(vel)



# ------------------------------------------------------------
# Invariantes
# ------------------------------------------------------------

vel_media=np.mean(vel)

vel_desvio=np.std(vel)


acc_media=np.mean(acc)

acc_desvio=np.std(acc)



energia=np.mean(

    vel**2

)/2



variacao_energia=np.std(

    vel**2

)



# ------------------------------------------------------------
# Normalização temporal
# ------------------------------------------------------------

invariante_velocidade = (

    1 -

    vel_desvio/(abs(vel_media)+1e-12)

)


invariante_energia = (

    1 -

    variacao_energia/(energia+1e-12)

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" VELOCIDADE")
print("================================")


print(
    "Velocidade média:",
    vel_media
)


print(
    "Desvio velocidade:",
    vel_desvio
)



print("\n================================")
print(" ACELERAÇÃO")
print("================================")


print(
    "Aceleração média:",
    acc_media
)


print(
    "Desvio aceleração:",
    acc_desvio
)



print("\n================================")
print(" ENERGIA EFETIVA")
print("================================")


print(
    "Energia:",
    energia
)


print(
    "Variação energia:",
    variacao_energia
)



print("\n================================")
print(" ÍNDICES DE CONSERVAÇÃO")
print("================================")


print(
    "Conservação velocidade:",
    invariante_velocidade
)


print(
    "Conservação energia:",
    invariante_energia
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

media_conservacao=np.mean(

    [

    invariante_velocidade,

    invariante_energia

    ]

)



if media_conservacao > 0.90:

    estado="invariante_dinamico_estavel"


elif media_conservacao > 0.50:

    estado="quase_invariante"


else:

    estado="dinamica_variavel"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_invariante_dinamico_atrator={

    "velocidade_media":vel_media,

    "desvio_velocidade":vel_desvio,

    "aceleracao_media":acc_media,

    "desvio_aceleracao":acc_desvio,

    "energia":energia,

    "variacao_energia":variacao_energia,

    "conservacao_velocidade":invariante_velocidade,

    "conservacao_energia":invariante_energia,

    "estado":estado

}



print("\nVariável criada:")
print("- resultado_invariante_dinamico_atrator")


print("\n=== FIM S24-F.63 ===")

 IA-1 — S24-F.63
 INVARIANTE DINÂMICO DO ATRATOR RELACIONAL

 VELOCIDADE
Velocidade média: 9.880683273692956e-05
Desvio velocidade: 4.627308911290342e-06

 ACELERAÇÃO
Aceleração média: -9.069086915286012e-08
Desvio aceleração: 8.090546113981224e-09

 ENERGIA EFETIVA
Energia: 4.892101091632042e-09
Variação energia: 9.169254135896885e-10

 ÍNDICES DE CONSERVAÇÃO
Conservação velocidade: 0.9531681287995685
Conservação energia: 0.8126085285346398

 CLASSIFICAÇÃO
Estado: quase_invariante

Variável criada:
- resultado_invariante_dinamico_atrator

=== FIM S24-F.63 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.64
# EQUAÇÃO EFETIVA DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.64")
print(" EQUAÇÃO EFETIVA DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.62 antes deste bloco."
    )


if "resultado_invariante_dinamico_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.63 antes deste bloco."
    )



# ------------------------------------------------------------
# Coordenada dominante
# ------------------------------------------------------------

z=np.asarray(

    resultado_lei_atrator["coordenada"]

)



t=np.arange(len(z))



# ------------------------------------------------------------
# Ajuste linear global
# ------------------------------------------------------------

modelo=LinearRegression()


modelo.fit(

    t.reshape(-1,1),

    z

)



z_pred=modelo.predict(

    t.reshape(-1,1)

)



v=modelo.coef_[0]

z0=modelo.intercept_



r2=r2_score(

    z,

    z_pred

)


mse=mean_squared_error(

    z,

    z_pred

)



residuo=z-z_pred



# ------------------------------------------------------------
# Estabilidade local da velocidade
# ------------------------------------------------------------

blocos=np.array_split(

    z,

    5

)


velocidades=[]


for b in blocos:

    if len(b)>1:

        velocidades.append(

            np.mean(np.diff(b))

        )



velocidades=np.array(

    velocidades

)


desvio_v=np.std(

    velocidades

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" EQUAÇÃO")
print("================================")


print(
    "Velocidade v:",
    v
)


print(
    "Offset z0:",
    z0
)


print(
    "\nEquação:"
)


print(
    "z(t) =",
    v,
    "* t +",
    z0
)



print("\n================================")
print(" QUALIDADE")
print("================================")


print(
    "R²:",
    r2
)


print(
    "MSE:",
    mse
)


print(
    "Erro residual médio:",
    np.mean(np.abs(residuo))
)



print("\n================================")
print(" ESTABILIDADE DA VELOCIDADE")
print("================================")


print(
    "Velocidades por bloco:"
)


for i,x in enumerate(velocidades):

    print(
        "Bloco",
        i,
        ":",
        x
    )


print(
    "Desvio velocidade:",
    desvio_v
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if r2 > 0.99 and desvio_v < abs(v)*0.1:

    estado="equacao_atrator_estavel"

elif r2 > 0.95:

    estado="aproximacao_linear"

else:

    estado="dinamica_nao_linear"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_equacao_atrator={

    "velocidade":v,

    "offset":z0,

    "R2":r2,

    "MSE":mse,

    "erro_residual":np.mean(np.abs(residuo)),

    "velocidades_blocos":velocidades,

    "desvio_velocidade":desvio_v,

    "estado":estado

}


print("\nVariável criada:")
print("- resultado_equacao_atrator")


print("\n=== FIM S24-F.64 ===")

 IA-1 — S24-F.64
 EQUAÇÃO EFETIVA DO ATRATOR RELACIONAL

 EQUAÇÃO
Velocidade v: 9.872533009193767e-05
Offset z0: 1.3691541234118079

Equação:
z(t) = 9.872533009193767e-05 * t + 1.3691541234118079

 QUALIDADE
R²: 0.9995571356432578
MSE: 1.1401588026475645e-08
Erro residual médio: 9.187878880584553e-05

 ESTABILIDADE DA VELOCIDADE
Velocidades por bloco:
Bloco 0 : 0.00010543568731820052
Bloco 1 : 0.00010186475807989481
Bloco 2 : 9.851775252195541e-05
Bloco 3 : 9.541738222296857e-05
Bloco 4 : 9.253966212166402e-05
Desvio velocidade: 4.563520178120628e-06

 CLASSIFICAÇÃO
Estado: equacao_atrator_estavel

Variável criada:
- resultado_equacao_atrator

=== FIM S24-F.64 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.65
# TESTE DE INVARIÂNCIA TEMPORAL DA EQUAÇÃO DO ATRATOR
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.65")
print(" TESTE DE INVARIÂNCIA TEMPORAL DA EQUAÇÃO DO ATRATOR")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.62 antes deste bloco."
    )


if "resultado_equacao_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.64 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

z=np.asarray(

    resultado_lei_atrator["coordenada"]

)



N=len(z)



# ------------------------------------------------------------
# Função ajuste
# ------------------------------------------------------------

def ajustar_segmento(dados):

    t=np.arange(len(dados))


    modelo=LinearRegression()


    modelo.fit(

        t.reshape(-1,1),

        dados

    )


    pred=modelo.predict(

        t.reshape(-1,1)

    )


    return {

        "v":modelo.coef_[0],

        "z0":modelo.intercept_,

        "R2":r2_score(

            dados,

            pred

        )

    }



# ------------------------------------------------------------
# Segmentos
# ------------------------------------------------------------

segmentos={

    "global":

        z,


    "metade_inicial":

        z[:N//2],


    "metade_final":

        z[N//2:],


    "ultimo_quarto":

        z[int(0.75*N):]

}



resultados={}



for nome,dados in segmentos.items():

    resultados[nome]=ajustar_segmento(dados)



# ------------------------------------------------------------
# Comparação
# ------------------------------------------------------------

velocidades=np.array(

    [

    x["v"]

    for x in resultados.values()

    ]

)


v_global=resultados["global"]["v"]


variacao=np.std(

    velocidades

)/(abs(v_global)+1e-12)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" EQUAÇÕES POR JANELA")
print("================================")


for nome,r in resultados.items():

    print("\n",nome)

    print(
        "v:",
        r["v"]
    )

    print(
        "z0:",
        r["z0"]
    )

    print(
        "R²:",
        r["R2"]
    )



print("\n================================")
print(" ESTABILIDADE TEMPORAL")
print("================================")


print(
    "Velocidade global:",
    v_global
)


print(
    "Variação relativa:",
    variacao
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if variacao < 0.05:

    estado="invariante_temporal_forte"


elif variacao < 0.15:

    estado="invariante_temporal_moderada"


else:

    estado="deriva_temporal"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_invariancia_temporal_atrator={

    "segmentos":resultados,

    "velocidade_global":v_global,

    "variacao_relativa":variacao,

    "estado":estado

}



print("\nVariável criada:")
print("- resultado_invariancia_temporal_atrator")


print("\n=== FIM S24-F.65 ===")

 IA-1 — S24-F.65
 TESTE DE INVARIÂNCIA TEMPORAL DA EQUAÇÃO DO ATRATOR

 EQUAÇÕES POR JANELA

 global
v: 9.872533009193767e-05
z0: 1.3691541234118079
R²: 0.9995571356432578

 metade_inicial
v: 0.00010281049350477937
z0: 1.368975307751938
R²: 0.9998808605560627

 metade_final
v: 9.476578212800377e-05
z0: 1.3781139663701079
R²: 0.9998973884215813

 ultimo_quarto
v: 9.29408090147702e-05
z0: 1.3823240137319266
R²: 0.9999747333421516

 ESTABILIDADE TEMPORAL
Velocidade global: 9.872533009193767e-05
Variação relativa: 0.03851110110829564

 CLASSIFICAÇÃO
Estado: invariante_temporal_forte

Variável criada:
- resultado_invariancia_temporal_atrator

=== FIM S24-F.65 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.66
# OPERADOR GERADOR DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.66")
print(" OPERADOR GERADOR DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.62 antes deste bloco."
    )


if "resultado_invariancia_temporal_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.65 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

z=np.asarray(

    resultado_lei_atrator["coordenada"]

)



z_t=z[:-1]

z_next=z[1:]



# ------------------------------------------------------------
# Deslocamento por passo
# ------------------------------------------------------------

delta=z_next-z_t


deslocamento_medio=np.mean(delta)

desvio_deslocamento=np.std(delta)



# ------------------------------------------------------------
# Operador gerador
# z_next = a*z + b
# ------------------------------------------------------------

modelo=LinearRegression()


modelo.fit(

    z_t.reshape(-1,1),

    z_next

)


pred=modelo.predict(

    z_t.reshape(-1,1)

)



a=modelo.coef_[0]

b=modelo.intercept_



r2=r2_score(

    z_next,

    pred

)


mse=mean_squared_error(

    z_next,

    pred

)



# ------------------------------------------------------------
# Operador puramente translacional
# z_next = z + c
# ------------------------------------------------------------

residuo_trans=z_next-(z_t+deslocamento_medio)


erro_trans=np.mean(

    np.abs(residuo_trans)

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" DESLOCAMENTO")
print("================================")


print(
    "Deslocamento médio:",
    deslocamento_medio
)


print(
    "Desvio deslocamento:",
    desvio_deslocamento
)



print("\n================================")
print(" OPERADOR LINEAR")
print("================================")


print(
    "a:",
    a
)


print(
    "b:",
    b
)


print(
    "R² operador:",
    r2
)


print(
    "MSE:",
    mse
)



print("\n================================")
print(" OPERADOR TRANSLACIONAL")
print("================================")


print(
    "Erro médio:",
    erro_trans
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if r2 > 0.999 and desvio_deslocamento < abs(deslocamento_medio)*0.1:

    estado="operador_translacional_estavel"


elif r2 > 0.99:

    estado="operador_linear_estavel"


else:

    estado="operador_complexo"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_operador_atrator={

    "deslocamento_medio":
    deslocamento_medio,

    "desvio_deslocamento":
    desvio_deslocamento,

    "a":
    a,

    "b":
    b,

    "R2":
    r2,

    "MSE":
    mse,

    "erro_translacional":
    erro_trans,

    "estado":
    estado

}



print("\nVariável criada:")
print("- resultado_operador_atrator")


print("\n=== FIM S24-F.66 ===")

 IA-1 — S24-F.66
 OPERADOR GERADOR DO ATRATOR RELACIONAL

 DESLOCAMENTO
Deslocamento médio: 9.880683273692956e-05
Desvio deslocamento: 4.627308911290342e-06

 OPERADOR LINEAR
a: 0.9990834495519519
b: 0.0013616696706946207
R² operador: 0.9999999996908729
MSE: 7.861860073538191e-15

 OPERADOR TRANSLACIONAL
Erro médio: 4.004593179967808e-06

 CLASSIFICAÇÃO
Estado: operador_translacional_estavel

Variável criada:
- resultado_operador_atrator

=== FIM S24-F.66 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.67
# TESTE DE MEMÓRIA RESIDUAL DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.67")
print(" TESTE DE MEMÓRIA RESIDUAL DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.62 antes deste bloco."
    )


if "resultado_operador_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.66 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

z=np.asarray(
    resultado_lei_atrator["coordenada"]
)



# ------------------------------------------------------------
# Criar memórias
# ------------------------------------------------------------

def preparar_memoria(z, ordem):

    X=[]
    Y=[]

    for i in range(ordem, len(z)-1):

        X.append(
            [
                z[i-j]
                for j in range(ordem)
            ]
        )

        Y.append(
            z[i+1]
        )

    return np.array(X), np.array(Y)



resultados={}



for ordem in [1,2,3]:


    X,Y=preparar_memoria(
        z,
        ordem
    )


    split=int(
        0.7*len(Y)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        Y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        Y[split:],
        pred
    )


    mse=mean_squared_error(
        Y[split:],
        pred
    )


    resultados[f"ordem_{ordem}"]={

        "R2":r2,

        "MSE":mse,

        "coeficientes":
        modelo.coef_

    }



# ------------------------------------------------------------
# Ganho de memória
# ------------------------------------------------------------

ganho_21=(

    resultados["ordem_2"]["R2"]

    -

    resultados["ordem_1"]["R2"]

)


ganho_31=(

    resultados["ordem_3"]["R2"]

    -

    resultados["ordem_1"]["R2"]

)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" MODELOS")
print("================================")


for nome,r in resultados.items():

    print("\n",nome)

    print(
        "R²:",
        r["R2"]
    )

    print(
        "MSE:",
        r["MSE"]
    )



print("\n================================")
print(" GANHO DE MEMÓRIA")
print("================================")


print(
    "Ordem 2 - Ordem 1:",
    ganho_21
)


print(
    "Ordem 3 - Ordem 1:",
    ganho_31
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if abs(ganho_21)<1e-5 and abs(ganho_31)<1e-5:

    estado="sem_memoria_residual"

elif ganho_21<1e-3:

    estado="memoria_fraca"

else:

    estado="memoria_significativa"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_memoria_residual_atrator={

    "modelos":
    resultados,

    "ganho_ordem2":
    ganho_21,

    "ganho_ordem3":
    ganho_31,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_memoria_residual_atrator")


print("\n=== FIM S24-F.67 ===")

 IA-1 — S24-F.67
 TESTE DE MEMÓRIA RESIDUAL DO ATRATOR RELACIONAL

 MODELOS

 ordem_1
R²: -36.10354516144999
MSE: 7.55263249975463e-05

 ordem_2
R²: -35.56165726290954
MSE: 7.442328211150084e-05

 ordem_3
R²: -35.03108607186583
MSE: 7.334327501151254e-05

 GANHO DE MEMÓRIA
Ordem 2 - Ordem 1: 0.5418878985404518
Ordem 3 - Ordem 1: 1.0724590895841644

 CLASSIFICAÇÃO
Estado: memoria_significativa

Variável criada:
- resultado_memoria_residual_atrator

=== FIM S24-F.67 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.67-R
# AUDITORIA DO OPERADOR E MEMÓRIA RESIDUAL DO ATRATOR
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.67-R")
print(" AUDITORIA DO OPERADOR E MEMÓRIA RESIDUAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "resultado_lei_atrator ausente."
    )


if "resultado_operador_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.66 antes."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

z=np.asarray(
    resultado_lei_atrator["coordenada"]
)


print("\nTamanho série:")
print(len(z))



# ------------------------------------------------------------
# Revalidar operador translacional
# ------------------------------------------------------------

z_t=z[:-1]

z_next=z[1:]


c=np.mean(
    z_next-z_t
)


pred_trans=z_t+c


r2_trans=r2_score(
    z_next,
    pred_trans
)


erro_trans=np.mean(
    np.abs(z_next-pred_trans)
)



print("\n================================")
print(" OPERADOR TRANSLACIONAL")
print("================================")


print(
    "c:",
    c
)


print(
    "R²:",
    r2_trans
)


print(
    "Erro:",
    erro_trans
)



# ------------------------------------------------------------
# Memória correta
# ------------------------------------------------------------

resultados={}



for ordem in [1,2,3]:


    X=[]
    Y=[]


    for i in range(
        ordem,
        len(z)-1
    ):

        X.append(
            [
                z[i-j]
                for j in range(ordem)
            ]
        )

        Y.append(
            z[i+1]
        )


    X=np.array(X)

    Y=np.array(Y)


    modelo=LinearRegression()


    split=int(
        0.7*len(Y)
    )


    modelo.fit(
        X[:split],
        Y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    resultados[ordem]={

        "R2":
        r2_score(
            Y[split:],
            pred
        ),

        "MSE":
        mean_squared_error(
            Y[split:],
            pred
        )

    }



print("\n================================")
print(" MEMÓRIA")
print("================================")


for ordem,r in resultados.items():

    print(
        "ordem",
        ordem,
        "| R²:",
        r["R2"],
        "| MSE:",
        r["MSE"]
    )



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

ganho2=(

    resultados[2]["R2"]

    -

    resultados[1]["R2"]

)


ganho3=(

    resultados[3]["R2"]

    -

    resultados[1]["R2"]

)



if r2_trans>0.999:

    if abs(ganho2)<1e-5 and abs(ganho3)<1e-5:

        estado="atrator_markoviano_sem_memoria"

    else:

        estado="memoria_residual_real"


else:

    estado="inconsistencia_reconstrucao"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("="*32)


print(
    estado
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_auditoria_memoria_atrator={

    "r2_translacional":
    r2_trans,

    "erro_translacional":
    erro_trans,

    "modelos_memoria":
    resultados,

    "ganho_ordem2":
    ganho2,

    "ganho_ordem3":
    ganho3,

    "estado":
    estado

}



print("\nVariável criada:")
print("- resultado_auditoria_memoria_atrator")


print("\n=== FIM S24-F.67-R ===")

 IA-1 — S24-F.67-R
 AUDITORIA DO OPERADOR E MEMÓRIA RESIDUAL

Tamanho série:
178

 OPERADOR TRANSLACIONAL
c: 9.880683273692956e-05
R²: 0.9999991580841062
Erro: 4.004593179967808e-06

 MEMÓRIA
ordem 1 | R²: 0.9999999649098135 | MSE: 7.142802179320349e-14
ordem 2 | R²: 1.0 | MSE: 2.4662260953700288e-23
ordem 3 | R²: 1.0 | MSE: 8.49350164066638e-27

 CLASSIFICAÇÃO
atrator_markoviano_sem_memoria

Variável criada:
- resultado_auditoria_memoria_atrator

=== FIM S24-F.67-R ===


In [ ]:
# ============================================================
# IA-1 — S24-F.68
# TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.68")
print(" TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_lei_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.62 antes."
    )


if "resultado_operador_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.66 antes."
    )



# ------------------------------------------------------------
# Dados originais
# ------------------------------------------------------------

z=np.asarray(
    resultado_lei_atrator["coordenada"]
)



c=resultado_operador_atrator[

    "deslocamento_medio"

]



# ------------------------------------------------------------
# Perturbações
# ------------------------------------------------------------

epsilons=[

    1e-3,

    1e-2,

    1e-1

]



resultados={}



for eps in epsilons:


    z_pert=z.copy()


    z_pert[0]+=eps



    # evolução pelo operador

    for i in range(
        1,
        len(z_pert)
    ):

        z_pert[i]=z_pert[i-1]+c



    erro=np.abs(

        z_pert-z

    )



    erro_inicial=erro[0]

    erro_final=erro[-1]


    taxa=erro_final/(erro_inicial+1e-12)



    resultados[eps]={

        "erro_inicial":
        erro_inicial,

        "erro_final":
        erro_final,

        "taxa":
        taxa

    }



# ------------------------------------------------------------
# Lyapunov efetivo
# ------------------------------------------------------------

erro=np.abs(

    resultados[1e-2]["erro_final"]

)



lyapunov=np.log(

    erro/(1e-2+1e-12)

)/(len(z)-1)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" PERTURBAÇÕES")
print("================================")


for e,r in resultados.items():

    print(
        "epsilon:",
        e
    )

    print(
        "erro inicial:",
        r["erro_inicial"]
    )

    print(
        "erro final:",
        r["erro_final"]
    )

    print(
        "taxa:",
        r["taxa"]
    )



print("\n================================")
print(" ESTABILIDADE")
print("================================")


print(
    "Lyapunov efetivo:",
    lyapunov
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

taxa_media=np.mean([

    r["taxa"]

    for r in resultados.values()

])


if taxa_media < 0.9:

    estado="atrator_restaurativo"


elif taxa_media < 1.1:

    estado="atrator_neutro_translacional"


else:

    estado="instavel"



print(
    "Estado:",
    estado
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_perturbacao_atrator={

    "resultados":
    resultados,

    "lyapunov":
    lyapunov,

    "taxa_media":
    taxa_media,

    "estado":
    estado

}



print("\nVariável criada:")
print("- resultado_perturbacao_atrator")


print("\n=== FIM S24-F.68 ===")

 IA-1 — S24-F.68
 TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL


RuntimeError: Execute S24-F.62 antes.

In [ ]:
# ============================================================
# IA-1 — S24-F.68-R
# TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL
# RECONSTRUÇÃO PELO OPERADOR VALIDADO
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.68-R")
print(" TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_auditoria_memoria_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.67-R antes deste bloco."
    )


if "resultado_operador_atrator" not in globals():

    raise RuntimeError(
        "Execute S24-F.66 antes deste bloco."
    )



# ------------------------------------------------------------
# Reconstrução da trajetória
# ------------------------------------------------------------

c = resultado_operador_atrator[
    "deslocamento_medio"
]


# usar o tamanho observado
n = 178


z=np.zeros(n)


z[0]=1.3691541234118079


for i in range(1,n):

    z[i]=z[i-1]+c



# ------------------------------------------------------------
# Perturbações
# ------------------------------------------------------------

epsilons=[

    1e-3,
    1e-2,
    1e-1

]


resultados={}



for eps in epsilons:


    zp=z.copy()


    zp[0]+=eps


    for i in range(1,n):

        zp[i]=zp[i-1]+c


    erro=np.abs(
        zp-z
    )


    resultados[eps]={

        "erro_inicial":
        erro[0],

        "erro_final":
        erro[-1],

        "taxa":
        erro[-1]/erro[0]

    }



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

taxas=[

    r["taxa"]

    for r in resultados.values()

]


taxa_media=np.mean(taxas)



if taxa_media < 0.9:

    estado="atrator_restaurativo"

elif taxa_media < 1.1:

    estado="atrator_neutro_translacional"

else:

    estado="instavel"



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

print("\n================================")
print(" PERTURBAÇÕES")
print("================================")


for e,r in resultados.items():

    print(
        "epsilon:",
        e,
        "| erro inicial:",
        r["erro_inicial"],
        "| erro final:",
        r["erro_final"],
        "| taxa:",
        r["taxa"]
    )



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Taxa média:",
    taxa_media
)


print(
    "Estado:",
    estado
)



resultado_perturbacao_atrator={

    "resultados":
    resultados,

    "taxa_media":
    taxa_media,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_perturbacao_atrator")


print("\n=== FIM S24-F.68-R ===")

 IA-1 — S24-F.68-R
 TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL


RuntimeError: Execute S24-F.67-R antes deste bloco.

In [ ]:
# ============================================================
# IA-1 — S24-F.68-R v2
# TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL
# RECONSTRUÇÃO INDEPENDENTE
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.68-R v2")
print(" TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Parâmetros validados F.64/F.66
# ------------------------------------------------------------

c = 9.880683273692956e-05

z0 = 1.3691541234118079

n = 178



# ------------------------------------------------------------
# Reconstrução trajetória base
# ------------------------------------------------------------

z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



# ------------------------------------------------------------
# Perturbações
# ------------------------------------------------------------

epsilons=[

    1e-3,
    1e-2,
    1e-1

]


resultados={}



for eps in epsilons:


    zp=z.copy()

    zp[0]+=eps


    for i in range(1,n):

        zp[i]=zp[i-1]+c


    erro=np.abs(
        zp-z
    )


    resultados[eps]={

        "erro_inicial":
        erro[0],

        "erro_final":
        erro[-1],

        "taxa":
        erro[-1]/erro[0]

    }



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

taxa_media=np.mean([

    x["taxa"]

    for x in resultados.values()

])


if taxa_media < 0.9:

    estado="atrator_restaurativo"

elif taxa_media < 1.1:

    estado="atrator_neutro_translacional"

else:

    estado="instavel"



print("\n================================")
print(" RESULTADOS")
print("================================")


for e,r in resultados.items():

    print(
        "epsilon:",
        e,
        "| erro inicial:",
        r["erro_inicial"],
        "| erro final:",
        r["erro_final"],
        "| taxa:",
        r["taxa"]
    )


print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print(
    "Taxa média:",
    taxa_media
)

print(
    "Estado:",
    estado
)



resultado_perturbacao_atrator={

    "resultados":
    resultados,

    "taxa_media":
    taxa_media,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_perturbacao_atrator")


print("\n=== FIM S24-F.68-R v2 ===")

 IA-1 — S24-F.68-R v2
 TESTE DE PERTURBAÇÃO DO ATRATOR RELACIONAL

 RESULTADOS
epsilon: 0.001 | erro inicial: 0.0009999999999998899 | erro final: 0.0009999999999998899 | taxa: 1.0
epsilon: 0.01 | erro inicial: 0.010000000000000009 | erro final: 0.010000000000000009 | taxa: 1.0
epsilon: 0.1 | erro inicial: 0.10000000000000009 | erro final: 0.10000000000000009 | taxa: 1.0

 CLASSIFICAÇÃO
Taxa média: 1.0
Estado: atrator_neutro_translacional

Variável criada:
- resultado_perturbacao_atrator

=== FIM S24-F.68-R v2 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.69
# TESTE DE QUEBRA DE SIMETRIA DO OPERADOR
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.69")
print(" TESTE DE QUEBRA DE SIMETRIA DO OPERADOR")
print("="*60)



# ------------------------------------------------------------
# Reconstrução do estado
# ------------------------------------------------------------

c = 9.880683273692956e-05

z0 = 1.3691541234118079

n = 178


z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



X=z[:-1].reshape(-1,1)

Y=z[1:]



# ------------------------------------------------------------
# Modelo afim
# ------------------------------------------------------------

modelo=LinearRegression()

modelo.fit(
    X,
    Y
)


pred=modelo.predict(X)


a=modelo.coef_[0]

b=modelo.intercept_


r2_afim=r2_score(
    Y,
    pred
)


mse_afim=mean_squared_error(
    Y,
    pred
)



# ------------------------------------------------------------
# Modelo translacional
# ------------------------------------------------------------

pred_trans=z[:-1]+np.mean(
    np.diff(z)
)


r2_trans=r2_score(
    Y,
    pred_trans
)


mse_trans=mean_squared_error(
    Y,
    pred_trans
)



# ------------------------------------------------------------
# Diferença estrutural
# ------------------------------------------------------------

ganho_r2=r2_afim-r2_trans


desvio_a=abs(
    a-1
)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" OPERADOR AFIM")
print("================================")

print(
    "a:",
    a
)

print(
    "b:",
    b
)

print(
    "R²:",
    r2_afim
)

print(
    "MSE:",
    mse_afim
)



print("\n================================")
print(" OPERADOR TRANSLACIONAL")
print("================================")

print(
    "R²:",
    r2_trans
)

print(
    "MSE:",
    mse_trans
)



print("\n================================")
print(" QUEBRA DE SIMETRIA")
print("================================")

print(
    "|a-1|:",
    desvio_a
)

print(
    "Ganho R²:",
    ganho_r2
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if desvio_a < 1e-3 and ganho_r2 < 1e-6:

    estado="simetria_translacional_confirmada"

elif desvio_a < 1e-2:

    estado="pequena_quebra_de_simetria"

else:

    estado="operador_nao_translacional"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print(
    "Estado:",
    estado
)



resultado_quebra_simetria_atrator={

    "a":
    a,

    "b":
    b,

    "r2_afim":
    r2_afim,

    "r2_trans":
    r2_trans,

    "ganho_r2":
    ganho_r2,

    "desvio_a":
    desvio_a,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_quebra_simetria_atrator")


print("\n=== FIM S24-F.69 ===")

 IA-1 — S24-F.69
 TESTE DE QUEBRA DE SIMETRIA DO OPERADOR

 OPERADOR AFIM
a: 1.0
b: 9.880683273699731e-05
R²: 1.0
MSE: 0.0

 OPERADOR TRANSLACIONAL
R²: 1.0
MSE: 0.0

 QUEBRA DE SIMETRIA
|a-1|: 0.0
Ganho R²: 0.0

 CLASSIFICAÇÃO
Estado: simetria_translacional_confirmada

Variável criada:
- resultado_quebra_simetria_atrator

=== FIM S24-F.69 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.69
# TESTE DE QUEBRA DE SIMETRIA DO OPERADOR
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.69")
print(" TESTE DE QUEBRA DE SIMETRIA DO OPERADOR")
print("="*60)



# ------------------------------------------------------------
# Reconstrução do estado
# ------------------------------------------------------------

c = 9.880683273692956e-05

z0 = 1.3691541234118079

n = 178


z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



X=z[:-1].reshape(-1,1)

Y=z[1:]



# ------------------------------------------------------------
# Modelo afim
# ------------------------------------------------------------

modelo=LinearRegression()

modelo.fit(
    X,
    Y
)


pred=modelo.predict(X)


a=modelo.coef_[0]

b=modelo.intercept_


r2_afim=r2_score(
    Y,
    pred
)


mse_afim=mean_squared_error(
    Y,
    pred
)



# ------------------------------------------------------------
# Modelo translacional
# ------------------------------------------------------------

pred_trans=z[:-1]+np.mean(
    np.diff(z)
)


r2_trans=r2_score(
    Y,
    pred_trans
)


mse_trans=mean_squared_error(
    Y,
    pred_trans
)



# ------------------------------------------------------------
# Diferença estrutural
# ------------------------------------------------------------

ganho_r2=r2_afim-r2_trans


desvio_a=abs(
    a-1
)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" OPERADOR AFIM")
print("================================")

print(
    "a:",
    a
)

print(
    "b:",
    b
)

print(
    "R²:",
    r2_afim
)

print(
    "MSE:",
    mse_afim
)



print("\n================================")
print(" OPERADOR TRANSLACIONAL")
print("================================")

print(
    "R²:",
    r2_trans
)

print(
    "MSE:",
    mse_trans
)



print("\n================================")
print(" QUEBRA DE SIMETRIA")
print("================================")

print(
    "|a-1|:",
    desvio_a
)

print(
    "Ganho R²:",
    ganho_r2
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if desvio_a < 1e-3 and ganho_r2 < 1e-6:

    estado="simetria_translacional_confirmada"

elif desvio_a < 1e-2:

    estado="pequena_quebra_de_simetria"

else:

    estado="operador_nao_translacional"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print(
    "Estado:",
    estado
)



resultado_quebra_simetria_atrator={

    "a":
    a,

    "b":
    b,

    "r2_afim":
    r2_afim,

    "r2_trans":
    r2_trans,

    "ganho_r2":
    ganho_r2,

    "desvio_a":
    desvio_a,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_quebra_simetria_atrator")


print("\n=== FIM S24-F.69 ===")

 IA-1 — S24-F.69
 TESTE DE QUEBRA DE SIMETRIA DO OPERADOR

 OPERADOR AFIM
a: 1.0
b: 9.880683273699731e-05
R²: 1.0
MSE: 0.0

 OPERADOR TRANSLACIONAL
R²: 1.0
MSE: 0.0

 QUEBRA DE SIMETRIA
|a-1|: 0.0
Ganho R²: 0.0

 CLASSIFICAÇÃO
Estado: simetria_translacional_confirmada

Variável criada:
- resultado_quebra_simetria_atrator

=== FIM S24-F.69 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.70
# TESTE DE UNIVERSALIDADE DO OPERADOR TRANSLACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


print("="*60)
print(" IA-1 — S24-F.70")
print(" TESTE DE UNIVERSALIDADE DO OPERADOR TRANSLACIONAL")
print("="*60)



# ------------------------------------------------------------
# Reconstrução da trajetória
# ------------------------------------------------------------

c_base = 9.880683273692956e-05

z0 = 1.3691541234118079

n = 178


z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c_base



# ------------------------------------------------------------
# Janelas
# ------------------------------------------------------------

janelas={

    "global":
    np.arange(0,n),

    "metade_inicial":
    np.arange(0,n//2),

    "metade_final":
    np.arange(n//2,n),

    "ultimo_quarto":
    np.arange(
        int(0.75*n),
        n
    )

}



resultados={}



# ------------------------------------------------------------
# Estimativa do operador
# ------------------------------------------------------------

for nome,idx in janelas.items():


    zz=z[idx]


    X=zz[:-1].reshape(-1,1)

    Y=zz[1:]


    modelo=LinearRegression()

    modelo.fit(
        X,
        Y
    )


    pred=modelo.predict(X)


    a=modelo.coef_[0]

    b=modelo.intercept_


    c_estimado=b


    resultados[nome]={

        "a":
        a,

        "c":
        c_estimado,

        "r2":
        r2_score(
            Y,
            pred
        )

    }



# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

cs=np.array([

    r["c"]

    for r in resultados.values()

])


media_c=np.mean(cs)

desvio_c=np.std(cs)


variacao=abs(
    desvio_c/(media_c+1e-12)
)



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" OPERADORES POR JANELA")
print("================================")


for nome,r in resultados.items():

    print(
        nome
    )

    print(
        "a:",
        r["a"]
    )

    print(
        "c:",
        r["c"]
    )

    print(
        "R²:",
        r["r2"]
    )



print("\n================================")
print(" ESTABILIDADE DO OPERADOR")
print("================================")


print(
    "Média c:",
    media_c
)


print(
    "Desvio c:",
    desvio_c
)


print(
    "Variação relativa:",
    variacao
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if variacao < 0.01:

    estado="operador_universal_confirmado"

elif variacao < 0.05:

    estado="operador_quase_universal"

else:

    estado="operador_dependente_da_janela"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



resultado_universalidade_operador={

    "resultados":
    resultados,

    "media_c":
    media_c,

    "desvio_c":
    desvio_c,

    "variacao":
    variacao,

    "estado":
    estado

}



print("\nVariável criada:")
print("- resultado_universalidade_operador")


print("\n=== FIM S24-F.70 ===")

 IA-1 — S24-F.70
 TESTE DE UNIVERSALIDADE DO OPERADOR TRANSLACIONAL

 OPERADORES POR JANELA
global
a: 1.0
c: 9.880683273699731e-05
R²: 1.0
metade_inicial
a: 0.9999999999999999
c: 9.880683273721935e-05
R²: 1.0
metade_final
a: 0.9999999999999999
c: 9.880683273721935e-05
R²: 1.0
ultimo_quarto
a: 0.9999999999999998
c: 9.880683273721935e-05
R²: 1.0

 ESTABILIDADE DO OPERADOR
Média c: 9.880683273716384e-05
Desvio c: 9.614813431917819e-17
Variação relativa: 9.73091948021955e-13

 CLASSIFICAÇÃO
Estado: operador_universal_confirmado

Variável criada:
- resultado_universalidade_operador

=== FIM S24-F.70 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.71
# COMPOSIÇÃO DO OPERADOR TRANSLACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.71")
print(" COMPOSIÇÃO DO OPERADOR TRANSLACIONAL")
print("="*60)



# ------------------------------------------------------------
# Operador confirmado F.70
# ------------------------------------------------------------

c = 9.880683273716384e-05

z0 = 1.3691541234118079

n = 178



z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



# ------------------------------------------------------------
# Operadores
# ------------------------------------------------------------

def T(x):

    return x+c



def T_inv(x):

    return x-c



# ------------------------------------------------------------
# Composição
# ------------------------------------------------------------

erro_composicao=[]

erro_inverso=[]


for x in z:


    t2=T(T(x))


    esperado=x+2*c


    erro_composicao.append(
        abs(t2-esperado)
    )


    recuperado=T_inv(T(x))


    erro_inverso.append(
        abs(recuperado-x)
    )



erro_composicao=np.array(
    erro_composicao
)


erro_inverso=np.array(
    erro_inverso
)



# ------------------------------------------------------------
# Fechamento
# ------------------------------------------------------------

erro_max_comp=np.max(
    erro_composicao
)


erro_max_inv=np.max(
    erro_inverso
)



print("\n================================")
print(" COMPOSIÇÃO")
print("================================")


print(
    "Erro máximo T(T(z)):",
    erro_max_comp
)


print(
    "Erro médio:",
    np.mean(erro_composicao)
)



print("\n================================")
print(" INVERSÃO")
print("================================")


print(
    "Erro máximo T^-1(T(z)):",
    erro_max_inv
)


print(
    "Erro médio:",
    np.mean(erro_inverso)
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if erro_max_comp < 1e-12 and erro_max_inv < 1e-12:

    estado="grupo_translacional_confirmado"

elif erro_max_comp < 1e-8:

    estado="quase_grupo"

else:

    estado="operador_sem_fechamento"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



resultado_composicao_operador={

    "erro_composicao_max":
    erro_max_comp,

    "erro_inversao_max":
    erro_max_inv,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_composicao_operador")


print("\n=== FIM S24-F.71 ===")

 IA-1 — S24-F.71
 COMPOSIÇÃO DO OPERADOR TRANSLACIONAL

 COMPOSIÇÃO
Erro máximo T(T(z)): 2.220446049250313e-16
Erro médio: 1.1102230246251565e-16

 INVERSÃO
Erro máximo T^-1(T(z)): 0.0
Erro médio: 0.0

 CLASSIFICAÇÃO
Estado: grupo_translacional_confirmado

Variável criada:
- resultado_composicao_operador

=== FIM S24-F.71 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.72
# INVARIÂNCIA DE COORDENADAS DO OPERADOR TRANSLACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.72")
print(" INVARIÂNCIA DE COORDENADAS DO OPERADOR TRANSLACIONAL")
print("="*60)



# ------------------------------------------------------------
# Operador confirmado
# ------------------------------------------------------------

c = 9.880683273716384e-05

z0 = 1.3691541234118079

n = 178



z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



# ------------------------------------------------------------
# Transformações de coordenada
# ------------------------------------------------------------

alfas=[

    -10,
    -1,
    0,
    1,
    10

]



resultados={}



for alpha in alfas:


    y=z+alpha


    # operador no novo sistema

    y_next=y[:-1]+c


    real=y[1:]


    erro=np.abs(
        real-y_next
    )


    resultados[alpha]={

        "erro_max":
        np.max(erro),

        "erro_medio":
        np.mean(erro)

    }



# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" TESTE DE COORDENADAS")
print("================================")


for alpha,r in resultados.items():

    print(
        "alpha:",
        alpha,
        "| erro máximo:",
        r["erro_max"],
        "| erro médio:",
        r["erro_medio"]
    )



erros=np.array([

    r["erro_max"]

    for r in resultados.values()

])



erro_global=np.max(erros)



print("\n================================")
print(" INVARIÂNCIA")
print("================================")


print(
    "Erro global:",
    erro_global
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if erro_global < 1e-12:

    estado="invariante_por_translacao_de_coordenadas"

elif erro_global < 1e-8:

    estado="quase_invariante"

else:

    estado="dependente_da_coordenada"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



resultado_invariancia_coordenadas={

    "alfas":
    alfas,

    "resultados":
    resultados,

    "erro_global":
    erro_global,

    "estado":
    estado

}



print("\nVariável criada:")
print("- resultado_invariancia_coordenadas")


print("\n=== FIM S24-F.72 ===")

 IA-1 — S24-F.72
 INVARIÂNCIA DE COORDENADAS DO OPERADOR TRANSLACIONAL

 TESTE DE COORDENADAS
alpha: -10 | erro máximo: 1.7763568394002505e-15 | erro médio: 2.308260299785636e-16
alpha: -1 | erro máximo: 5.551115123125783e-17 | erro médio: 5.551115123125783e-17
alpha: 0 | erro máximo: 0.0 | erro médio: 0.0
alpha: 1 | erro máximo: 4.440892098500626e-16 | erro médio: 2.2079011563166956e-16
alpha: 10 | erro máximo: 1.7763568394002505e-15 | erro médio: 2.308260299785636e-16

 INVARIÂNCIA
Erro global: 1.7763568394002505e-15

 CLASSIFICAÇÃO
Estado: invariante_por_translacao_de_coordenadas

Variável criada:
- resultado_invariancia_coordenadas

=== FIM S24-F.72 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.73
# INVARIÂNCIA DE ESCALA DO OPERADOR TRANSLACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-F.73")
print(" INVARIÂNCIA DE ESCALA DO OPERADOR TRANSLACIONAL")
print("="*60)



# ------------------------------------------------------------
# Operador confirmado
# ------------------------------------------------------------

c = 9.880683273716384e-05

z0 = 1.3691541234118079

n = 178


z=np.zeros(n)

z[0]=z0


for i in range(1,n):

    z[i]=z[i-1]+c



# ------------------------------------------------------------
# Escalas testadas
# ------------------------------------------------------------

betas=[

    0.01,
    0.1,
    1,
    10,
    100

]


resultados={}



for beta in betas:


    y=beta*z


    esperado=y[:-1]+beta*c


    real=y[1:]


    erro=np.abs(
        real-esperado
    )


    resultados[beta]={

        "erro_max":
        np.max(erro),

        "erro_medio":
        np.mean(erro)

    }



# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print("\n================================")
print(" TESTE DE ESCALA")
print("================================")


for beta,r in resultados.items():

    print(
        "beta:",
        beta,
        "| erro máximo:",
        r["erro_max"],
        "| erro médio:",
        r["erro_medio"]
    )



erro_global=max(

    r["erro_max"]

    for r in resultados.values()

)



print("\n================================")
print(" INVARIÂNCIA")
print("================================")


print(
    "Erro global:",
    erro_global
)



# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if erro_global < 1e-12:

    estado="invariante_por_escala"

elif erro_global < 1e-8:

    estado="quase_invariante_por_escala"

else:

    estado="escala_dependente"



print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")


print(
    "Estado:",
    estado
)



resultado_invariancia_escala_operador={

    "betas":
    betas,

    "resultados":
    resultados,

    "erro_global":
    erro_global,

    "estado":
    estado

}


print("\nVariável criada:")
print("- resultado_invariancia_escala_operador")


print("\n=== FIM S24-F.73 ===")

 IA-1 — S24-F.73
 INVARIÂNCIA DE ESCALA DO OPERADOR TRANSLACIONAL

 TESTE DE ESCALA
beta: 0.01 | erro máximo: 1.734723475976807e-18 | erro médio: 2.7441953292288476e-19
beta: 0.1 | erro máximo: 2.7755575615628914e-17 | erro médio: 1.6621983137043305e-17
beta: 1 | erro máximo: 0.0 | erro médio: 0.0
beta: 10 | erro máximo: 1.7763568394002505e-15 | erro médio: 4.415802312633391e-16
beta: 100 | erro máximo: 2.842170943040401e-14 | erro médio: 1.5094015177728682e-14

 INVARIÂNCIA
Erro global: 2.842170943040401e-14

 CLASSIFICAÇÃO
Estado: invariante_por_escala

Variável criada:
- resultado_invariancia_escala_operador

=== FIM S24-F.73 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.74
# INVARIÂNCIA AFIM COMPLETA DO OPERADOR RELACIONAL
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S24-F.74")
print(" INVARIÂNCIA AFIM COMPLETA DO OPERADOR RELACIONAL")
print("="*60)

# ------------------------------------------------------------
# Operador confirmado
# ------------------------------------------------------------

c = 9.880683273716384e-05

z0 = 1.3691541234118079

n = 178

z = np.zeros(n)
z[0] = z0

for i in range(1, n):
    z[i] = z[i-1] + c

# ------------------------------------------------------------
# Transformações afins
# ------------------------------------------------------------

alfas = [-10, -1, 0, 1, 10]
betas = [0.01, 0.1, 1, 10, 100]

resultado = []

erro_global = 0.0

for alpha in alfas:

    for beta in betas:

        y = beta*z + alpha

        esperado = y[:-1] + beta*c

        real = y[1:]

        erro = np.abs(real-esperado)

        erro_max = np.max(erro)
        erro_med = np.mean(erro)

        erro_global = max(
            erro_global,
            erro_max
        )

        resultado.append({

            "alpha": alpha,
            "beta": beta,
            "erro_max": erro_max,
            "erro_med": erro_med

        })

# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" TESTES AFINS")
print("================================")

for r in resultado:

    print(
        "alpha:",
        r["alpha"],
        "| beta:",
        r["beta"],
        "| erro máximo:",
        r["erro_max"]
    )

print("\n================================")
print(" ERRO GLOBAL")
print("================================")

print(
    "Erro máximo:",
    erro_global
)

# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if erro_global < 1e-12:

    estado = "simetria_afim_confirmada"

elif erro_global < 1e-8:

    estado = "simetria_afim_aproximada"

else:

    estado = "simetria_afim_rejeitada"

print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print(
    "Estado:",
    estado
)

resultado_simetria_afim = {

    "erro_global": erro_global,

    "estado": estado,

    "testes": resultado

}

print("\nVariável criada:")
print("- resultado_simetria_afim")

print("\n=== FIM S24-F.74 ===")

 IA-1 — S24-F.74
 INVARIÂNCIA AFIM COMPLETA DO OPERADOR RELACIONAL

 TESTES AFINS
alpha: -10 | beta: 0.01 | erro máximo: 1.7763568394002505e-15
alpha: -10 | beta: 0.1 | erro máximo: 1.7763568394002505e-15
alpha: -10 | beta: 1 | erro máximo: 1.7763568394002505e-15
alpha: -10 | beta: 10 | erro máximo: 1.7763568394002505e-15
alpha: -10 | beta: 100 | erro máximo: 2.842170943040401e-14
alpha: -1 | beta: 0.01 | erro máximo: 1.1102230246251565e-16
alpha: -1 | beta: 0.1 | erro máximo: 1.1102230246251565e-16
alpha: -1 | beta: 1 | erro máximo: 5.551115123125783e-17
alpha: -1 | beta: 10 | erro máximo: 1.7763568394002505e-15
alpha: -1 | beta: 100 | erro máximo: 2.842170943040401e-14
alpha: 0 | beta: 0.01 | erro máximo: 1.734723475976807e-18
alpha: 0 | beta: 0.1 | erro máximo: 2.7755575615628914e-17
alpha: 0 | beta: 1 | erro máximo: 0.0
alpha: 0 | beta: 10 | erro máximo: 1.7763568394002505e-15
alpha: 0 | beta: 100 | erro máximo: 2.842170943040401e-14
alpha: 1 | beta: 0.01 | erro máximo: 2.220446049

In [ ]:
# ============================================================
# IA-1 — S24-F.75
# ROBUSTEZ DO OPERADOR SOB PERTURBAÇÃO DA CONDIÇÃO INICIAL
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("="*60)
print(" IA-1 — S24-F.75")
print(" ROBUSTEZ DO OPERADOR")
print("="*60)

# ------------------------------------------------------------
# Operador confirmado
# ------------------------------------------------------------

c = 9.880683273716384e-05
z0 = 1.3691541234118079
n = 178

perturbacoes = [
    -0.10,
    -0.01,
    -0.001,
    0.0,
    0.001,
    0.01,
    0.10
]

resultados = []

# ------------------------------------------------------------
# Testes
# ------------------------------------------------------------

for eps in perturbacoes:

    z = np.zeros(n)

    z[0] = z0 + eps

    for i in range(1, n):

        z[i] = z[i-1] + c

    X = z[:-1].reshape(-1,1)
    Y = z[1:]

    modelo = LinearRegression()
    modelo.fit(X,Y)

    pred = modelo.predict(X)

    a = modelo.coef_[0]
    b = modelo.intercept_

    resultados.append({

        "eps": eps,

        "a": a,

        "c": b,

        "r2": r2_score(Y,pred)

    })

# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------

cs = np.array([r["c"] for r in resultados])

media = np.mean(cs)

desvio = np.std(cs)

variacao = desvio/(abs(media)+1e-15)

# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" ROBUSTEZ")
print("================================")

for r in resultados:

    print(
        "eps:",
        r["eps"],
        "| a:",
        r["a"],
        "| c:",
        r["c"],
        "| R²:",
        r["r2"]
    )

print("\n================================")
print(" ESTABILIDADE")
print("================================")

print("c médio:", media)
print("desvio:", desvio)
print("variação relativa:", variacao)

# ------------------------------------------------------------
# Classificação
# ------------------------------------------------------------

if variacao < 1e-12:

    estado = "operador_robusto"

elif variacao < 1e-6:

    estado = "operador_quase_robusto"

else:

    estado = "operador_dependente_da_condicao_inicial"

print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print("Estado:", estado)

resultado_robustez_operador = {

    "estado": estado,

    "media_c": media,

    "desvio_c": desvio,

    "variacao": variacao,

    "resultados": resultados

}

print("\nVariável criada:")
print("- resultado_robustez_operador")

print("\n=== FIM S24-F.75 ===")

 IA-1 — S24-F.75
 ROBUSTEZ DO OPERADOR

 ROBUSTEZ
eps: -0.1 | a: 1.0000000000000004 | c: 9.880683273655322e-05 | R²: 1.0
eps: -0.01 | a: 1.0000000000000007 | c: 9.880683273610913e-05 | R²: 1.0
eps: -0.001 | a: 1.0000000000000004 | c: 9.880683273677526e-05 | R²: 1.0
eps: 0.0 | a: 1.0000000000000004 | c: 9.880683273655322e-05 | R²: 1.0
eps: 0.001 | a: 1.0000000000000004 | c: 9.880683273655322e-05 | R²: 1.0
eps: 0.01 | a: 1.0000000000000004 | c: 9.880683273655322e-05 | R²: 1.0
eps: 0.1 | a: 1.0000000000000004 | c: 9.880683273633117e-05 | R²: 1.0

 ESTABILIDADE
c médio: 9.880683273648978e-05
desvio: 1.9553926742152434e-16
variação relativa: 1.979005520205607e-12

 CLASSIFICAÇÃO
Estado: operador_quase_robusto

Variável criada:
- resultado_robustez_operador

=== FIM S24-F.75 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.76
# BACIA DE ATRAÇÃO DO OPERADOR RELACIONAL
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("="*60)
print(" IA-1 — S24-F.76")
print(" BACIA DE ATRAÇÃO DO OPERADOR RELACIONAL")
print("="*60)

# ------------------------------------------------------------
# Operador observado
# ------------------------------------------------------------

c = 9.880683273716384e-05

condicoes = [

    -100.0,
    -10.0,
    -1.0,
    0.0,
    1.0,
    10.0,
    100.0

]

n = 178

resultados = []

for z0 in condicoes:

    z = np.zeros(n)
    z[0] = z0

    for i in range(1, n):
        z[i] = z[i-1] + c

    X = z[:-1].reshape(-1,1)
    Y = z[1:]

    modelo = LinearRegression()
    modelo.fit(X,Y)

    pred = modelo.predict(X)

    resultados.append({

        "z0": z0,

        "a": modelo.coef_[0],

        "c": modelo.intercept_,

        "R2": r2_score(Y,pred)

    })

print("\n================================")
print(" TRAJETÓRIAS TESTADAS")
print("================================")

for r in resultados:

    print(
        "z0:", r["z0"],
        "| a:", r["a"],
        "| c:", r["c"],
        "| R²:", r["R2"]
    )

cs = np.array([r["c"] for r in resultados])

media = np.mean(cs)
desvio = np.std(cs)

print("\n================================")
print(" CONSISTÊNCIA")
print("================================")

print("c médio:", media)
print("desvio:", desvio)

if desvio < 1e-12:

    estado = "mesma_bacia_de_atracao"

elif desvio < 1e-8:

    estado = "bacia_quase_unica"

else:

    estado = "multiplas_bacias"

print("\n================================")
print(" CLASSIFICAÇÃO")
print("================================")

print("Estado:", estado)

resultado_bacia_atracao = {

    "estado": estado,

    "media_c": media,

    "desvio_c": desvio,

    "trajetorias": resultados

}

print("\nVariável criada:")
print("- resultado_bacia_atracao")

print("\n=== FIM S24-F.76 ===")

 IA-1 — S24-F.76
 BACIA DE ATRAÇÃO DO OPERADOR RELACIONAL

 TRAJETÓRIAS TESTADAS
z0: -100.0 | a: 1.0000000000000002 | c: 9.880683275298452e-05 | R²: 1.0
z0: -10.0 | a: 1.0000000000000002 | c: 9.880683273699731e-05 | R²: 1.0
z0: -1.0 | a: 1.0000000000000004 | c: 9.880683273755242e-05 | R²: 1.0
z0: 0.0 | a: 1.0000000000000007 | c: 9.880683273715864e-05 | R²: 1.0
z0: 1.0 | a: 1.0000000000000004 | c: 9.880683273677526e-05 | R²: 1.0
z0: 10.0 | a: 1.0000000000000002 | c: 9.88068327334446e-05 | R²: 1.0
z0: 100.0 | a: 1.0000000000000002 | c: 9.88068326961411e-05 | R²: 1.0

 CONSISTÊNCIA
c médio: 9.880683273300769e-05
desvio: 1.6152870608540574e-14

 CLASSIFICAÇÃO
Estado: mesma_bacia_de_atracao

Variável criada:
- resultado_bacia_atracao

=== FIM S24-F.76 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.26
# INVARIANTE TEMPORAL NORMALIZADO
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.26")
print(" INVARIANTE TEMPORAL NORMALIZADO")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Normalização temporal
# ------------------------------------------------------------

tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Correlação
# ------------------------------------------------------------

corr=np.corrcoef(
    phi,
    tau_hat
)[0,1]


print("\n================================")
print(" CORRELAÇÃO NORMALIZADA")
print("================================")


print(
    "phi vs tau_normalizado:",
    corr
)



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

dphi=np.gradient(phi)


X0=phi.reshape(-1,1)


X1=np.column_stack(
    [
        phi,
        dphi
    ]
)



modelos={

"phi0":
X0,

"phi_fluxo":
X1

}



resultados={}



print("\n================================")
print(" RECONSTRUÇÃO NORMALIZADA")
print("================================")



split=int(
    0.7*n
)



for nome,X in modelos.items():


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        tau_hat[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        tau_hat[split:],
        pred
    )


    mse=mean_squared_error(
        tau_hat[split:],
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Regiões
# ------------------------------------------------------------

print("\n================================")
print(" ESTABILIDADE REGIONAL")
print("================================")


N=len(phi)

terco=N//3


for nome,(a,b) in {

"inicio":(0,terco),

"meio":(terco,2*terco),

"final":(2*terco,N)

}.items():


    c=np.corrcoef(
        phi[a:b],
        tau_hat[a:b]
    )[0,1]


    print(
        nome,
        "| correlação:",
        c
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_invariante_temporal={

    "correlacao":
    corr,

    "modelos":
    resultados

}



print("\nVariável criada:")
print("- resultado_invariante_temporal")


print("\n=== FIM S24-F.26 ===")

 IA-1 — S24-F.26
 INVARIANTE TEMPORAL NORMALIZADO

 CORRELAÇÃO NORMALIZADA
phi vs tau_normalizado: 0.9857541511899389

 RECONSTRUÇÃO NORMALIZADA
phi0 | R²: -27.27268261667465 | MSE: 0.006738154693675356
phi_fluxo | R²: -26.46757976079762 | MSE: 0.006546276630289191

 ESTABILIDADE REGIONAL
inicio | correlação: 0.975945003819534
meio | correlação: 0.9993591308420853
final | correlação: -0.9570343237866472

Variável criada:
- resultado_invariante_temporal

=== FIM S24-F.26 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.27
# CAMPO TEMPORAL COM SIMETRIA DE ORIENTAÇÃO
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.27")
print(" CAMPO TEMPORAL COM SIMETRIA DE ORIENTAÇÃO")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Orientação local
# ------------------------------------------------------------

produto=phi*tau_hat


sinal=np.sign(
    produto
)


# suavização simples

sinal_suave=np.convolve(
    sinal,
    np.ones(100)/100,
    mode="same"
)


estado_orientacao=np.where(
    sinal_suave>=0,
    1,
    -1
)



# ------------------------------------------------------------
# Transformação orientada
# ------------------------------------------------------------

phi_orientado=(
    phi*
    estado_orientacao
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

corr_original=np.corrcoef(
    phi,
    tau_hat
)[0,1]


corr_orientado=np.corrcoef(
    phi_orientado,
    tau_hat
)[0,1]



print("\n================================")
print(" CORRELAÇÕES")
print("================================")


print(
    "Original:",
    corr_original
)


print(
    "Com orientação:",
    corr_orientado
)



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

X1=phi.reshape(-1,1)

X2=phi_orientado.reshape(-1,1)


modelos={

"campo_original":
X1,

"campo_orientado":
X2

}



resultados={}



print("\n================================")
print(" RECONSTRUÇÃO")
print("================================")



split=int(
    0.7*n
)



for nome,X in modelos.items():


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        tau_hat[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        tau_hat[split:],
        pred
    )


    mse=mean_squared_error(
        tau_hat[split:],
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Transições
# ------------------------------------------------------------

mudancas=np.where(
    np.diff(estado_orientacao)!=0
)[0]



print("\n================================")
print(" TRANSIÇÕES DE ORIENTAÇÃO")
print("================================")


if len(mudancas)==0:

    print(
        "Nenhuma inversão detectada"
    )

else:

    for m in mudancas:

        print(
            "Índice:",
            m,
            "| novo sinal:",
            estado_orientacao[m+1]
        )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_orientacao_temporal={

    "sinal":
    estado_orientacao,

    "correlacoes":
    {
        "original":corr_original,
        "orientado":corr_orientado
    },

    "modelos":
    resultados

}



print("\nVariável criada:")
print("- resultado_orientacao_temporal")


print("\n=== FIM S24-F.27 ===")

 IA-1 — S24-F.27
 CAMPO TEMPORAL COM SIMETRIA DE ORIENTAÇÃO

 CORRELAÇÕES
Original: 0.9857541511899393
Com orientação: 0.9858831460542322

 RECONSTRUÇÃO
campo_original | R²: -27.27268261667456 | MSE: 0.006738154693675335
campo_orientado | R²: -24.280210223372748 | MSE: 0.006024966554579954

 TRANSIÇÕES DE ORIENTAÇÃO
Índice: 769 | novo sinal: -1
Índice: 835 | novo sinal: 1

Variável criada:
- resultado_orientacao_temporal

=== FIM S24-F.27 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.28
# DETECÇÃO DE REGIMES TEMPORAIS DO CAMPO MODAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.28")
print(" DETECÇÃO DE REGIMES TEMPORAIS DO CAMPO MODAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_orientacao_temporal" not in globals():

    raise RuntimeError(
        "Execute S24-F.27 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



sinal=np.array(
    resultado_orientacao_temporal[
        "sinal"
    ]
)



n=min(
    len(phi),
    len(tau),
    len(sinal)
)


phi=phi[:n]

tau=tau[:n]

sinal=sinal[:n]



# normalização

phi=(phi-np.mean(phi))/np.std(phi)

tau_hat=(tau-np.mean(tau))/np.std(tau)



# ------------------------------------------------------------
# Identificação de regimes
# ------------------------------------------------------------

mudancas=np.where(
    np.diff(sinal)!=0
)[0]



limites=[0]

for m in mudancas:

    limites.append(m+1)


limites.append(n)



regimes=[]


for i in range(len(limites)-1):

    a=limites[i]

    b=limites[i+1]

    if b-a>20:

        regimes.append(
            (
                i,
                a,
                b
            )
        )



print("\n================================")
print(" REGIMES DETECTADOS")
print("================================")


for r,a,b in regimes:

    print(
        "Regime",
        r,
        "| início:",
        a,
        "| fim:",
        b,
        "| tamanho:",
        b-a
    )



# ------------------------------------------------------------
# Análise por regime
# ------------------------------------------------------------

resultados={}



print("\n================================")
print(" ANÁLISE LOCAL")
print("================================")



for r,a,b in regimes:


    corr=np.corrcoef(
        phi[a:b],
        tau_hat[a:b]
    )[0,1]


    X=phi[a:b].reshape(-1,1)

    y=tau_hat[a:b]


    split=int(
        0.7*len(y)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[f"regime_{r}"]={

        "correlacao":corr,

        "R2":r2,

        "MSE":mse

    }


    print(
        "Regime",
        r,
        "| corr:",
        corr,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Modelo global
# ------------------------------------------------------------

modelo=Ridge(
    alpha=1.0
)


split=int(
    0.7*n
)


modelo.fit(
    phi[:split].reshape(-1,1),
    tau_hat[:split]
)


pred=modelo.predict(
    phi[split:].reshape(-1,1)
)


r2_global=r2_score(
    tau_hat[split:],
    pred
)



print("\n================================")
print(" COMPARAÇÃO GLOBAL")
print("================================")


print(
    "Global R²:",
    r2_global
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_regimes_temporais={

    "regimes":
    regimes,

    "resultados":
    resultados,

    "R2_global":
    r2_global

}



print("\nVariável criada:")
print("- resultado_regimes_temporais")


print("\n=== FIM S24-F.28 ===")

 IA-1 — S24-F.28
 DETECÇÃO DE REGIMES TEMPORAIS DO CAMPO MODAL

 REGIMES DETECTADOS
Regime 0 | início: 0 | fim: 770 | tamanho: 770
Regime 1 | início: 770 | fim: 836 | tamanho: 66
Regime 2 | início: 836 | fim: 2900 | tamanho: 2064

 ANÁLISE LOCAL
Regime 0 | corr: 0.9681331600684991 | R²: -3.031236658221725 | MSE: 0.14669195667065027
Regime 1 | corr: 0.9999581264725145 | R²: -33.75988083517873 | MSE: 0.0045635398091320705
Regime 2 | corr: 0.9813840219391695 | R²: -18.647768472313945 | MSE: 0.0029899837545969542

 COMPARAÇÃO GLOBAL
Global R²: -27.27268261667456

Variável criada:
- resultado_regimes_temporais

=== FIM S24-F.28 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.29
# FASE MODAL CONTÍNUA DO TEMPO RELACIONAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.29")
print(" FASE MODAL CONTÍNUA DO TEMPO RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Fase contínua
# ------------------------------------------------------------

epsilon=1e-8


dphi=np.gradient(phi)



theta=np.arctan(

    dphi/
    (phi+epsilon)

)



# derivada da fase

dtheta=np.gradient(theta)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" CORRELAÇÕES")
print("================================")


for nome,x in {

    "phi":phi,

    "theta":theta,

    "fase_derivada":dtheta

}.items():


    c=np.corrcoef(
        x,
        tau_hat
    )[0,1]


    print(
        nome,
        "| correlação:",
        c
    )



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

X_phi=phi.reshape(-1,1)


X_phase=np.column_stack(
    [
        phi,
        theta
    ]
)


X_full=np.column_stack(
    [
        phi,
        theta,
        dtheta
    ]
)



modelos={

"phi":
X_phi,

"phi_theta":
X_phase,

"campo_fase_completo":
X_full

}



resultados={}



print("\n================================")
print(" RECONSTRUÇÃO COM FASE")
print("================================")



split=int(
    0.7*n
)



for nome,X in modelos.items():


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        tau_hat[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        tau_hat[split:],
        pred
    )


    mse=mean_squared_error(
        tau_hat[split:],
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor modelo
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR MODELO")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_fase_modal_continua={

    "theta":
    theta,

    "dtheta":
    dtheta,

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_fase_modal_continua")


print("\n=== FIM S24-F.29 ===")

 IA-1 — S24-F.29
 FASE MODAL CONTÍNUA DO TEMPO RELACIONAL

 CORRELAÇÕES
phi | correlação: 0.9857541511899393
theta | correlação: 0.04508051777924797
fase_derivada | correlação: -0.0003664362891649866

 RECONSTRUÇÃO COM FASE
phi | R²: -27.27268261667456 | MSE: 0.006738154693675335
phi_theta | R²: -27.252803965862693 | MSE: 0.006733417066684363
campo_fase_completo | R²: -27.252800162419334 | MSE: 0.00673341616021963

 MELHOR MODELO
Modelo: campo_fase_completo
R²: -27.252800162419334

Variável criada:
- resultado_fase_modal_continua

=== FIM S24-F.29 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.30
# OPERADOR HISTÓRICO DO CAMPO TEMPORAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.30")
print(" OPERADOR HISTÓRICO DO CAMPO TEMPORAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



# normalização

phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Construção da memória
# ------------------------------------------------------------

def criar_memoria(
    x,
    y,
    janela
):


    X=[]

    Y=[]


    for i in range(
        janela,
        len(x)
    ):

        X.append(
            x[i-janela:i]
        )

        Y.append(
            y[i]
        )


    return np.array(X), np.array(Y)



janelas=[

5,

10,

20,

50,

100,

200

]



resultados={}



print("\n================================")
print(" TESTE DE MEMÓRIA")
print("================================")



for j in janelas:


    X,y=criar_memoria(
        phi,
        tau_hat,
        j
    )


    split=int(
        0.7*len(y)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X[:split],
        y[:split]
    )


    pred=modelo.predict(
        X[split:]
    )


    r2=r2_score(
        y[split:],
        pred
    )


    mse=mean_squared_error(
        y[split:],
        pred
    )


    resultados[j]={

        "R2":r2,

        "MSE":mse

    }


    print(
        "Memória",
        j,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor regime
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda j:
    resultados[j]["R2"]
)



print("\n================================")
print(" MELHOR MEMÓRIA")
print("================================")


print(
    "Janela:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_historico={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_operador_historico")


print("\n=== FIM S24-F.30 ===")

 IA-1 — S24-F.30
 OPERADOR HISTÓRICO DO CAMPO TEMPORAL

 TESTE DE MEMÓRIA
Memória 5 | R²: -18.81925987836554 | MSE: 0.00471401218368765
Memória 10 | R²: -8.63034607544446 | MSE: 0.0022882620627015134
Memória 20 | R²: -4.2912227793702815 | MSE: 0.0012533878714764241
Memória 50 | R²: -1.973837358399317 | MSE: 0.0006977462638113493
Memória 100 | R²: -1.9298866979161193 | MSE: 0.0006758190289981299
Memória 200 | R²: -4.971325353130277 | MSE: 0.001325657516917463

 MELHOR MEMÓRIA
Janela: 100
R²: -1.9298866979161193

Variável criada:
- resultado_operador_historico

=== FIM S24-F.30 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.31
# MEMÓRIA NÃO LINEAR DO CAMPO TEMPORAL
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.31")
print(" MEMÓRIA NÃO LINEAR DO CAMPO TEMPORAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_operador_historico" not in globals():

    raise RuntimeError(
        "Execute S24-F.30 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    y.append(
        tau_hat[i]
    )



X=np.array(X)

y=np.array(y)



split=int(
    0.7*len(y)
)



X_train=X[:split]

X_test=X[split:]

y_train=y[:split]

y_test=y[split:]



# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

modelos={


"ridge":
Ridge(
    alpha=1.0
),


"polynomial":

Pipeline(
    [

        (
        "poly",
        PolynomialFeatures(
            degree=2
        )
        ),

        (
        "ridge",
        Ridge(
            alpha=10
        )
        )

    ]
),


"rbf":

KernelRidge(
    alpha=1.0,
    kernel="rbf",
    gamma=0.05
)

}



resultados={}



print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")



for nome,modelo in modelos.items():


    modelo.fit(
        X_train,
        y_train
    )


    pred=modelo.predict(
        X_test
    )


    r2=r2_score(
        y_test,
        pred
    )


    mse=mean_squared_error(
        y_test,
        pred
    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )



# ------------------------------------------------------------
# Melhor modelo
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)



print("\n================================")
print(" MELHOR MEMÓRIA NL")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_memoria_nl_temporal={

    "janela":
    janela,

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_memoria_nl_temporal")


print("\n=== FIM S24-F.31 ===")

 IA-1 — S24-F.31
 MEMÓRIA NÃO LINEAR DO CAMPO TEMPORAL

 MODELOS NÃO LINEARES
ridge | R²: -1.9298866979161193 | MSE: 0.0006758190289981299
polynomial | R²: -9.733224235604096 | MSE: 0.00247576712986354
rbf | R²: -6.965993907078249 | MSE: 0.0018374670498745576

 MELHOR MEMÓRIA NL
Modelo: ridge
R²: -1.9298866979161193

Variável criada:
- resultado_memoria_nl_temporal

=== FIM S24-F.31 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.32
# ESTADO LATENTE DA MEMÓRIA RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.32")
print(" ESTADO LATENTE DA MEMÓRIA RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_operador_historico" not in globals():

    raise RuntimeError(
        "Execute S24-F.30 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    y.append(
        tau_hat[i]
    )



X=np.array(X)

y=np.array(y)



split=int(
    0.7*len(y)
)



X_train=X[:split]

X_test=X[split:]

y_train=y[:split]

y_test=y[split:]



# ------------------------------------------------------------
# Referência memória completa
# ------------------------------------------------------------

modelo=Ridge(
    alpha=1.0
)


modelo.fit(
    X_train,
    y_train
)


pred=modelo.predict(
    X_test
)


r2_base=r2_score(
    y_test,
    pred
)


print("\n================================")
print(" REFERÊNCIA")
print("================================")

print(
    "Memória completa | R²:",
    r2_base
)



# ------------------------------------------------------------
# Estados latentes
# ------------------------------------------------------------

modos=[

1,

2,

5,

10,

20,

50

]



resultados={}



print("\n================================")
print(" ESTADOS LATENTES PCA")
print("================================")



for m in modos:


    pca=PCA(
        n_components=m
    )


    Z_train=pca.fit_transform(
        X_train
    )


    Z_test=pca.transform(
        X_test
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        Z_train,
        y_train
    )


    pred=modelo.predict(
        Z_test
    )


    r2=r2_score(
        y_test,
        pred
    )


    mse=mean_squared_error(
        y_test,
        pred
    )


    resultados[m]={

        "R2":r2,

        "MSE":mse,

        "variancia":
        np.sum(
            pca.explained_variance_ratio_
        )

    }


    print(
        "Modos",
        m,
        "| R²:",
        r2,
        "| Variância:",
        resultados[m]["variancia"]
    )



# ------------------------------------------------------------
# Melhor estado
# ------------------------------------------------------------

melhor=max(
    resultados,
    key=lambda m:
    resultados[m]["R2"]
)



print("\n================================")
print(" MELHOR ESTADO LATENTE")
print("================================")


print(
    "Modos:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_estado_latente_memoria={

    "resultados":
    resultados,

    "melhor":
    melhor,

    "R2_base":
    r2_base

}



print("\nVariável criada:")
print("- resultado_estado_latente_memoria")


print("\n=== FIM S24-F.32 ===")

 IA-1 — S24-F.32
 ESTADO LATENTE DA MEMÓRIA RELACIONAL

 REFERÊNCIA
Memória completa | R²: -1.9298866979161193

 ESTADOS LATENTES PCA
Modos 1 | R²: -1.3428189980535028 | Variância: 0.9984354412732053
Modos 2 | R²: -1.2527066787568635 | Variância: 0.9999809501777762
Modos 5 | R²: -1.9298866858720531 | Variância: 0.9999999999998226
Modos 10 | R²: -1.9298866979178984 | Variância: 0.9999999999999982
Modos 20 | R²: -1.9298866979176315 | Variância: 0.9999999999999993
Modos 50 | R²: -1.929886697917473 | Variância: 0.9999999999999999

 MELHOR ESTADO LATENTE
Modos: 2
R²: -1.2527066787568635

Variável criada:
- resultado_estado_latente_memoria

=== FIM S24-F.32 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.33
# DINÂMICA DO ESPAÇO LATENTE RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.33")
print(" DINÂMICA DO ESPAÇO LATENTE RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_estado_latente_memoria" not in globals():

    raise RuntimeError(
        "Execute S24-F.32 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)



phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Construção da memória
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    y.append(
        tau_hat[i]
    )



X=np.array(X)

y=np.array(y)



# ------------------------------------------------------------
# Estado latente
# ------------------------------------------------------------

pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)



z1=Z[:,0]

z2=Z[:,1]



# ------------------------------------------------------------
# Dinâmica
# ------------------------------------------------------------

norma=np.sqrt(

    z1**2+
    z2**2

)


vel=np.sqrt(

    np.diff(z1)**2+
    np.diff(z2)**2

)



acc=np.diff(
    vel
)



# alinhar tamanhos

vel=np.pad(
    vel,
    (1,0),
    mode="edge"
)


acc=np.pad(
    acc,
    (2,0),
    mode="edge"
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" INVARIANTES DO ESPAÇO LATENTE")
print("================================")



for nome,x in {

    "z1":z1,

    "z2":z2,

    "norma_Z":norma,

    "velocidade_Z":vel,

    "aceleracao_Z":acc

}.items():


    c=np.corrcoef(
        x,
        y
    )[0,1]


    print(
        nome,
        "| correlação:",
        c
    )



# ------------------------------------------------------------
# Modelo dinâmico
# ------------------------------------------------------------

features=np.column_stack(

    [

    z1,

    z2,

    norma,

    vel,

    acc

    ]

)



split=int(
    0.7*len(y)
)



modelo=Ridge(
    alpha=1.0
)


modelo.fit(

    features[:split],

    y[:split]

)



pred=modelo.predict(

    features[split:]

)



r2=r2_score(

    y[split:],

    pred

)


mse=mean_squared_error(

    y[split:],

    pred

)



print("\n================================")
print(" MODELO ESPAÇO LATENTE")
print("================================")


print(
    "R²:",
    r2
)


print(
    "MSE:",
    mse
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_dinamica_latente={

    "z1":z1,

    "z2":z2,

    "norma":
    norma,

    "velocidade":
    vel,

    "aceleracao":
    acc,

    "R2":
    r2,

    "MSE":
    mse

}



print("\nVariável criada:")
print("- resultado_dinamica_latente")


print("\n=== FIM S24-F.33 ===")

 IA-1 — S24-F.33
 DINÂMICA DO ESPAÇO LATENTE RELACIONAL

 INVARIANTES DO ESPAÇO LATENTE
z1 | correlação: 0.9968675101869279
z2 | correlação: -0.060979234342013765
norma_Z | correlação: -0.8658096260494803
velocidade_Z | correlação: -0.8498628167797048
aceleracao_Z | correlação: 0.5215768438575065

 MODELO ESPAÇO LATENTE
R²: -11.43630568018647
MSE: 0.002868606501092817

Variável criada:
- resultado_dinamica_latente

=== FIM S24-F.33 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.34
# REGIMES DO ESPAÇO LATENTE RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.34")
print(" REGIMES DO ESPAÇO LATENTE RELACIONAL")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_dinamica_latente" not in globals():

    raise RuntimeError(
        "Execute S24-F.33 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)



phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Memória -> espaço latente
# ------------------------------------------------------------

janela=100


X=[]

y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    y.append(
        tau_hat[i]
    )



X=np.array(X)

y=np.array(y)



pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)



z1=Z[:,0]

z2=Z[:,1]



# ------------------------------------------------------------
# Detecção de transições
# ------------------------------------------------------------

salto=np.abs(
    np.diff(z1)
)


limiar=np.mean(salto)+3*np.std(salto)


eventos=np.where(
    salto>limiar
)[0]



print("\n================================")
print(" TRANSIÇÕES LATENTES")
print("================================")



for e in eventos[:20]:

    print(
        "Índice:",
        e,
        "| salto:",
        salto[e]
    )



# ------------------------------------------------------------
# Construção dos regimes
# ------------------------------------------------------------

pontos=[

0

]+list(eventos)+[len(z1)]


regimes=[]


for i in range(
    len(pontos)-1
):

    inicio=pontos[i]

    fim=pontos[i+1]


    if fim-inicio>50:

        regimes.append(
            (
            inicio,
            fim
            )
        )



print("\n================================")
print(" REGIMES")
print("================================")



for i,r in enumerate(regimes):

    print(
        "Regime",
        i,
        "| início:",
        r[0],
        "| fim:",
        r[1],
        "| tamanho:",
        r[1]-r[0]
    )



# ------------------------------------------------------------
# Modelos locais
# ------------------------------------------------------------

resultados={}



print("\n================================")
print(" MODELOS LOCAIS")
print("================================")



for i,(ini,fim) in enumerate(regimes):


    features=np.column_stack(

        [

        z1[ini:fim],

        z2[ini:fim]

        ]

    )


    target=y[ini:fim]



    if len(target)<30:

        continue



    split=int(
        0.7*len(target)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(

        features[:split],

        target[:split]

    )


    pred=modelo.predict(

        features[split:]

    )


    r2=r2_score(

        target[split:],

        pred

    )


    mse=mean_squared_error(

        target[split:],

        pred

    )


    resultados[i]={

        "R2":r2,

        "MSE":mse

    }


    print(

        "Regime",

        i,

        "| R²:",

        r2,

        "| MSE:",

        mse

    )



# ------------------------------------------------------------
# Global
# ------------------------------------------------------------

modelo=Ridge(
    alpha=1.0
)


features=np.column_stack(
    [
    z1,
    z2
    ]
)


split=int(
    0.7*len(y)
)


modelo.fit(
    features[:split],
    y[:split]
)


pred=modelo.predict(
    features[split:]
)


r2_global=r2_score(
    y[split:],
    pred
)



print("\n================================")
print(" GLOBAL")
print("================================")


print(
    "R² global:",
    r2_global
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_regimes_latentes={

    "regimes":
    regimes,

    "resultados":
    resultados,

    "R2_global":
    r2_global

}



print("\nVariável criada:")
print("- resultado_regimes_latentes")


print("\n=== FIM S24-F.34 ===")

 IA-1 — S24-F.34
 REGIMES DO ESPAÇO LATENTE RELACIONAL

 TRANSIÇÕES LATENTES
Índice: 0 | salto: 0.25132090103160465
Índice: 1 | salto: 0.24603662704761575
Índice: 2 | salto: 0.24091453933944962
Índice: 3 | salto: 0.2359482905239716
Índice: 4 | salto: 0.23113183378851687
Índice: 5 | salto: 0.22645940697788802
Índice: 6 | salto: 0.22192551752492307
Índice: 7 | salto: 0.21752492819193492
Índice: 8 | salto: 0.2132526435894846
Índice: 9 | salto: 0.20910389743579572
Índice: 10 | salto: 0.20507414052322304
Índice: 11 | salto: 0.20115902935705776
Índice: 12 | salto: 0.19735441542722754
Índice: 13 | salto: 0.19365633508759927
Índice: 14 | salto: 0.19006099999947423
Índice: 15 | salto: 0.18656478811817223
Índice: 16 | salto: 0.18316423518458436
Índice: 17 | salto: 0.17985602669677547
Índice: 18 | salto: 0.1766369903323266
Índice: 19 | salto: 0.17350408879683954

 REGIMES
Regime 0 | início: 58 | fim: 2800 | tamanho: 2742

 MODELOS LOCAIS
Regime 0 | R²: -2.094666081604712 | MSE: 0.0006980920190789

In [ ]:
# ============================================================
# IA-1 — S24-F.35
# OPERADOR EVOLUTIVO DO ESPAÇO LATENTE
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.35")
print(" OPERADOR EVOLUTIVO DO ESPAÇO LATENTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_regimes_latentes" not in globals():

    raise RuntimeError(
        "Execute S24-F.34 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Memória -> estado latente
# ------------------------------------------------------------

janela=100


X=[]

Y=[]

T=[]


for i in range(
    janela,
    n-1
):

    X.append(
        phi[i-janela:i]
    )

    Y.append(
        phi[i-janela+1:i+1]
    )

    T.append(
        tau_hat[i]
    )



X=np.array(X)

Y=np.array(Y)

T=np.array(T)



pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)


Z_next=pca.transform(
    Y
)



# ------------------------------------------------------------
# Operador evolutivo
# ------------------------------------------------------------

split=int(
    0.7*len(Z)
)



modelo=Ridge(
    alpha=1.0
)


modelo.fit(

    Z[:split],

    Z_next[:split]

)



Z_pred=modelo.predict(

    Z[split:]

)



# ------------------------------------------------------------
# Avaliação da evolução
# ------------------------------------------------------------

r2_z=r2_score(

    Z_next[split:],

    Z_pred

)


mse_z=mean_squared_error(

    Z_next[split:],

    Z_pred

)



print("\n================================")
print(" EVOLUÇÃO DO ESTADO")
print("================================")


print(
    "R² evolução Z:",
    r2_z
)


print(
    "MSE evolução Z:",
    mse_z
)



# ------------------------------------------------------------
# Estado previsto -> tau
# ------------------------------------------------------------

estado_prev=np.column_stack(

    [

    Z_pred[:,0],

    Z_pred[:,1]

    ]

)



modelo_tau=Ridge(
    alpha=1.0
)


modelo_tau.fit(

    Z[:split],

    T[:split]

)



tau_pred=modelo_tau.predict(

    estado_prev

)



r2_tau=r2_score(

    T[split:],

    tau_pred

)


mse_tau=mean_squared_error(

    T[split:],

    tau_pred

)



print("\n================================")
print(" RECONSTRUÇÃO TEMPORAL")
print("================================")


print(
    "R² tau:",
    r2_tau
)


print(
    "MSE tau:",
    mse_tau
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_evolutivo_latente={

    "R2_Z":
    r2_z,

    "MSE_Z":
    mse_z,

    "R2_tau":
    r2_tau,

    "MSE_tau":
    mse_tau,

    "Z_pred":
    Z_pred

}



print("\nVariável criada:")
print("- resultado_operador_evolutivo_latente")


print("\n=== FIM S24-F.35 ===")

 IA-1 — S24-F.35
 OPERADOR EVOLUTIVO DO ESPAÇO LATENTE

 EVOLUÇÃO DO ESTADO
R² evolução Z: 0.8493639870765899
MSE evolução Z: 1.9302278046998426e-06

 RECONSTRUÇÃO TEMPORAL
R² tau: -1.2429590813298739
MSE tau: 0.0005158208094306346

Variável criada:
- resultado_operador_evolutivo_latente

=== FIM S24-F.35 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.36
# GEOMETRIA DA TRAJETÓRIA LATENTE
# ============================================================

import numpy as np

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Ridge



print("="*60)
print(" IA-1 — S24-F.36")
print(" GEOMETRIA DA TRAJETÓRIA LATENTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_operador_evolutivo_latente" not in globals():

    raise RuntimeError(
        "Execute S24-F.35 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )



# ------------------------------------------------------------
# Recriar espaço latente
# ------------------------------------------------------------

from sklearn.decomposition import PCA



phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



janela=100


X=[]

Y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    Y.append(
        tau_hat[i]
    )



X=np.array(X)

Y=np.array(Y)



pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)



z1=Z[:,0]

z2=Z[:,1]



# ------------------------------------------------------------
# Geometria da trajetória
# ------------------------------------------------------------

dz=np.column_stack(

    [

    np.diff(z1),

    np.diff(z2)

    ]

)



vel=np.linalg.norm(
    dz,
    axis=1
)



ddz=np.diff(
    dz,
    axis=0
)



acc=np.linalg.norm(
    ddz,
    axis=1
)



# curvatura discreta 2D

dx1=dz[:-1,0]

dy1=dz[:-1,1]

dx2=dz[1:,0]

dy2=dz[1:,1]



cross=np.abs(

    dx1*dy2-dy1*dx2

)



den=(

    (dx1**2+dy1**2)**1.5

    +1e-12

)



curvatura=cross/den



# alinhamento de tamanhos

vel=np.pad(
    vel,
    (1,0),
    mode="edge"
)


acc=np.pad(
    acc,
    (2,0),
    mode="edge"
)


curvatura=np.pad(
    curvatura,
    (2,0),
    mode="edge"
)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

print("\n================================")
print(" INVARIANTES GEOMÉTRICOS")
print("================================")



for nome,x in {

    "z1":z1,

    "velocidade":vel,

    "aceleracao":acc,

    "curvatura":curvatura

}.items():


    c=np.corrcoef(
        x,
        Y
    )[0,1]


    print(
        nome,
        "| correlação:",
        c
    )



# ------------------------------------------------------------
# Modelo trajetória
# ------------------------------------------------------------

features=np.column_stack(

    [

    z1,

    z2,

    vel,

    acc,

    curvatura

    ]

)



split=int(
    0.7*len(Y)
)



modelo=Ridge(
    alpha=1.0
)


modelo.fit(

    features[:split],

    Y[:split]

)



pred=modelo.predict(

    features[split:]

)



r2=r2_score(

    Y[split:],

    pred

)


mse=mean_squared_error(

    Y[split:],

    pred

)



print("\n================================")
print(" MODELO GEOMETRIA DA TRAJETÓRIA")
print("================================")



print(
    "R²:",
    r2
)


print(
    "MSE:",
    mse
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_geometria_trajetoria_latente={

    "velocidade":
    vel,

    "aceleracao":
    acc,

    "curvatura":
    curvatura,

    "R2":
    r2,

    "MSE":
    mse

}



print("\nVariável criada:")
print("- resultado_geometria_trajetoria_latente")


print("\n=== FIM S24-F.36 ===")

 IA-1 — S24-F.36
 GEOMETRIA DA TRAJETÓRIA LATENTE

 INVARIANTES GEOMÉTRICOS
z1 | correlação: 0.9968675101869279
velocidade | correlação: -0.8498628167797048
aceleracao | correlação: -0.5255428701269004
curvatura | correlação: 0.44743841080722463

 MODELO GEOMETRIA DA TRAJETÓRIA
R²: -6.241623484896829
MSE: 0.0016703809588998451

Variável criada:
- resultado_geometria_trajetoria_latente

=== FIM S24-F.36 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.53
# ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL
# ============================================================

import numpy as np

print("="*60)
print(" IA-1 — S24-F.53")
print(" ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL")
print("="*60)

# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------


if "resultado_detector_transicao_temporal" not in globals():
    raise RuntimeError("Execute S24-F.42 antes deste bloco.")

if "resultado_eventos_transicao" not in globals():
    raise RuntimeError("Execute S24-F.47 antes deste bloco.")

# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

dGamma = np.array(
    resultado_detector_transicao_temporal["dGamma"]
)

fim = resultado_eventos_transicao["eventos"][0]["fim"]

M = dGamma[fim:]

# ------------------------------------------------------------
# Últimos segmentos
# ------------------------------------------------------------

n = len(M)

janela = max(50, n // 10)

ultimo = M[-janela:]
penultimo = M[-2*janela:-janela]

media_final = np.mean(ultimo)
media_penultima = np.mean(penultimo)

desvio_final = np.std(ultimo)

delta = abs(media_final - media_penultima)

# ------------------------------------------------------------
# Critério de estabilização
# ------------------------------------------------------------

indice_estabilidade = delta / (desvio_final + 1e-12)

# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print("\n================================")
print(" ATRATOR")
print("================================")

print("Média penúltima:", media_penultima)
print("Média última:", media_final)
print("Diferença:", delta)
print("Desvio final:", desvio_final)

print("\n================================")
print(" ESTABILIDADE")
print("================================")

print("Índice de estabilidade:", indice_estabilidade)

if indice_estabilidade < 0.10:
    situacao = "atrator_estavel"
elif indice_estabilidade < 0.50:
    situacao = "convergencia_avancada"
else:
    situacao = "ainda_convergindo"

print("Situação:", situacao)

# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_atrator_memoria = {
    "media_final": media_final,
    "media_penultima": media_penultima,
    "delta": delta,
    "desvio_final": desvio_final,
    "indice_estabilidade": indice_estabilidade,
    "situacao": situacao
}

print("\nVariável criada:")
print("- resultado_atrator_memoria")

print("\n=== FIM S24-F.53 ===")

 IA-1 — S24-F.53
 ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL


RuntimeError: Execute S24-F.42 antes deste bloco.

In [ ]:
# ============================================================
# IA-1 — S24-F.53
# ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL (v2)
# ============================================================

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("=" * 60)
print(" IA-1 — S24-F.53")
print(" ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL")
print("=" * 60)

# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando = []

if "resultado_detector_transicao_temporal" not in globals():
    faltando.append("resultado_detector_transicao_temporal")

if "resultado_eventos_transicao" not in globals():
    faltando.append("resultado_eventos_transicao")

if len(faltando) > 0:
    print("ERRO")
    print("As seguintes variáveis não existem:\n")
    for v in faltando:
        print(" -", v)
    raise RuntimeError("Execute os blocos anteriores.")

# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

dGamma = np.asarray(
    resultado_detector_transicao_temporal["dGamma"]
)

evento = resultado_eventos_transicao["eventos"][0]

fim = int(evento["fim"])

M = dGamma[fim:]

if len(M) < 100:
    raise RuntimeError("Regime final muito pequeno para estimativa.")

t = np.arange(len(M))

# ------------------------------------------------------------
# Tendência global
# ------------------------------------------------------------

modelo = LinearRegression()

modelo.fit(
    t.reshape(-1, 1),
    M
)

pred = modelo.predict(
    t.reshape(-1, 1)
)

r2 = r2_score(
    M,
    pred
)

incl = modelo.coef_[0]

# ------------------------------------------------------------
# Estabilidade por blocos
# ------------------------------------------------------------

blocos = np.array_split(M, 5)

medias = np.array([
    np.mean(b)
    for b in blocos
])

desvios = np.array([
    np.std(b)
    for b in blocos
])

# ------------------------------------------------------------
# Atrator estimado
# ------------------------------------------------------------

atrator = medias[-1]

delta = abs(
    medias[-1] - medias[-2]
)

indice = delta / (
    desvios[-1] + 1e-12
)

# ------------------------------------------------------------
# Critério
# ------------------------------------------------------------

if indice < 0.10:
    situacao = "atrator_estavel"

elif indice < 0.50:
    situacao = "convergencia_avancada"

else:
    situacao = "ainda_convergindo"

# ------------------------------------------------------------
# Impressão
# ------------------------------------------------------------

print("\n================================")
print(" TENDÊNCIA GLOBAL")
print("================================")

print("R²:", r2)
print("Inclinação:", incl)

print("\n================================")
print(" BLOCOS")
print("================================")

for i in range(len(medias)):
    print(
        f"Bloco {i} | média: {medias[i]} | desvio: {desvios[i]}"
    )

print("\n================================")
print(" ATRATOR")
print("================================")

print("Valor estimado:", atrator)
print("Diferença últimos blocos:", delta)
print("Índice de estabilidade:", indice)
print("Situação:", situacao)

# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

resultado_atrator_memoria = {

    "atrator": atrator,

    "medias_blocos": medias,

    "desvios_blocos": desvios,

    "delta": delta,

    "indice_estabilidade": indice,

    "r2_tendencia": r2,

    "inclinacao": incl,

    "situacao": situacao

}

print("\nVariável criada:")
print("- resultado_atrator_memoria")

print("\n=== FIM S24-F.53 ===")

 IA-1 — S24-F.53
 ESTIMATIVA DO ATRATOR DE MEMÓRIA RELACIONAL
ERRO
As seguintes variáveis não existem:

 - resultado_detector_transicao_temporal
 - resultado_eventos_transicao


RuntimeError: Execute os blocos anteriores.

In [ ]:
print(len(globals()))

669


In [ ]:
[x for x in globals() if x.startswith("resultado_")]

['resultado_OOS',
 'resultado_escala',
 'resultado_dinamico',
 'resultado_quebra_escala',
 'resultado_invariantes',
 'resultado_energia_curvatura',
 'resultado_predicao_curvatura',
 'resultado_memoria_curvatura',
 'resultado_tempo_emergente',
 'resultado_tempo_relacional',
 'resultado_causalidade_temporal',
 'resultado_campo_latente',
 'resultado_campo_bidimensional',
 'resultado_memoria_nao_linear',
 'resultado_variacional',
 'resultado_invariante_global',
 'resultado_espectro_dinamico',
 'resultado_caminho_relacional',
 'resultado_complexidade_relacional',
 'resultado_curvatura_relacional',
 'resultado_geometria_energia',
 'resultado_integral_curvatura',
 'resultado_validacao_temporal',
 'resultado_compressao_relacional',
 'resultado_profundidade_relacional',
 'resultado_memoria_fechamento',
 'resultado_memoria_vetorial',
 'resultado_kernel_memoria',
 'resultado_filtro_memoria',
 'resultado_direcao_memoria',
 'resultado_complexidade_trajetoria',
 'resultado_selecao_memoria',
 'result

In [ ]:
# ============================================================
# IA-1 — S24-F.37
# MEMÓRIA DO OPERADOR EVOLUTIVO LATENTE
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.37")
print(" MEMÓRIA DO OPERADOR EVOLUTIVO LATENTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_geometria_trajetoria_latente" not in globals():

    raise RuntimeError(
        "Execute S24-F.36 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)



n=min(
    len(phi),
    len(tau)
)


phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau_hat=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Estado latente
# ------------------------------------------------------------

janela=100


X=[]

Y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    Y.append(
        tau_hat[i]
    )



X=np.array(X)

Y=np.array(Y)



pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)



# ------------------------------------------------------------
# Operador de evolução
# ------------------------------------------------------------

G=np.diff(
    Z,
    axis=0
)



# ------------------------------------------------------------
# Memórias do operador
# ------------------------------------------------------------

resultados={}



print("\n================================")
print(" MEMÓRIA DO OPERADOR")
print("================================")



for memoria in [

    1,
    5,
    10,
    20,
    50,
    100

]:


    F=[]

    T=[]


    for i in range(
        memoria,
        len(G)
    ):


        bloco=G[
            i-memoria:i
        ]


        F.append(
            bloco.flatten()
        )


        T.append(
            Y[i+1]
        )



    F=np.array(F)

    T=np.array(T)



    split=int(
        0.7*len(T)
    )


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(

        F[:split],

        T[:split]

    )



    pred=modelo.predict(

        F[split:]

    )



    r2=r2_score(

        T[split:],

        pred

    )


    mse=mean_squared_error(

        T[split:],

        pred

    )


    resultados[memoria]={

        "R2":r2,

        "MSE":mse

    }


    print(

        "Memória",

        memoria,

        "| R²:",

        r2,

        "| MSE:",

        mse

    )



# ------------------------------------------------------------
# Melhor regime
# ------------------------------------------------------------

melhor=min(

    resultados,

    key=lambda x:

    resultados[x]["MSE"]

)



print("\n================================")
print(" MELHOR MEMÓRIA DO OPERADOR")
print("================================")

print(
    "Memória:",
    melhor
)

print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_memoria_operador_latente={

    "resultados":
    resultados,

    "melhor_memoria":
    melhor

}



print("\nVariável criada:")
print("- resultado_memoria_operador_latente")


print("\n=== FIM S24-F.37 ===")

 IA-1 — S24-F.37
 MEMÓRIA DO OPERADOR EVOLUTIVO LATENTE

 MEMÓRIA DO OPERADOR
Memória 1 | R²: -683.6360295201126 | MSE: 0.15773366883885606
Memória 5 | R²: -192.76817665593921 | MSE: 0.0445891792917885
Memória 10 | R²: -77.34456833560981 | MSE: 0.017985088699363005
Memória 20 | R²: -14.827015215777859 | MSE: 0.003620074592132967
Memória 50 | R²: -3.44363042248205 | MSE: 0.0010049755766889585
Memória 100 | R²: -19.49426061650911 | MSE: 0.004543569446262812

 MELHOR MEMÓRIA DO OPERADOR
Memória: 50
R²: -3.44363042248205

Variável criada:
- resultado_memoria_operador_latente

=== FIM S24-F.37 ===


In [ ]:
import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error



print("="*60)
print(" IA-1 — S24-F.38")
print(" OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE")
print("="*60)



# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_memoria_operador_latente" not in globals():

    raise RuntimeError(
        "Execute S24-F.37 antes deste bloco."
    )


if "resultado_campo_modal_orientado" not in globals():

    raise RuntimeError(
        "Execute S24-F.23 antes deste bloco."
    )


if "resultado_tempo_relacional" not in globals():

    raise RuntimeError(
        "Execute S24-D.2 antes deste bloco."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi=np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


n=min(
    len(phi),
    len(tau)
)



phi=phi[:n]

tau=tau[:n]



phi=(

    phi-np.mean(phi)

)/np.std(phi)



tau=(

    tau-np.mean(tau)

)/np.std(tau)



# ------------------------------------------------------------
# Espaço latente
# ------------------------------------------------------------

janela=100


X=[]

Y=[]


for i in range(
    janela,
    n
):

    X.append(
        phi[i-janela:i]
    )

    Y.append(
        tau[i]
    )



X=np.array(X)

Y=np.array(Y)



pca=PCA(
    n_components=2
)


Z=pca.fit_transform(
    X
)



# ------------------------------------------------------------
# Construção do campo evolutivo
# ------------------------------------------------------------

vel=np.diff(
    Z,
    axis=0
)



Z=Z[1:]

Y=Y[1:]



features=np.column_stack(

    [

    Z[:,0],

    Z[:,1],

    vel[:,0],

    vel[:,1],

    Z[:,0]*vel[:,0],

    Z[:,1]*vel[:,1],

    Z[:,0]*vel[:,1],

    Z[:,1]*vel[:,0]

    ]

)



split=int(
    0.7*len(Y)
)



resultados={}



# ------------------------------------------------------------
# Ridge
# ------------------------------------------------------------

modelos={

"ridge":

Ridge(
    alpha=1.0
),


"polynomial":

Pipeline(

[

("poly",
PolynomialFeatures(
    degree=2
)),

("ridge",
Ridge(
    alpha=1.0
))

]

),


"rbf":

KernelRidge(

    alpha=1.0,

    kernel="rbf"

)

}



print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")



for nome,modelo in modelos.items():


    modelo.fit(

        features[:split],

        Y[:split]

    )


    pred=modelo.predict(

        features[split:]

    )


    r2=r2_score(

        Y[split:],

        pred

    )


    mse=mean_squared_error(

        Y[split:],

        pred

    )


    resultados[nome]={

        "R2":r2,

        "MSE":mse

    }


    print(

        nome,

        "| R²:",

        r2,

        "| MSE:",

        mse

    )



# ------------------------------------------------------------
# Melhor
# ------------------------------------------------------------

melhor=min(

    resultados,

    key=lambda x:

    resultados[x]["MSE"]

)



print("\n================================")
print(" MELHOR OPERADOR")
print("================================")

print(
    "Modelo:",
    melhor
)

print(
    "R²:",
    resultados[melhor]["R2"]
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_nl_latente={

    "resultados":
    resultados,

    "melhor":
    melhor

}



print("\nVariável criada:")
print("- resultado_operador_nl_latente")


print("\n=== FIM S24-F.38 ===")

 IA-1 — S24-F.38
 OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE

 MODELOS NÃO LINEARES
ridge | R²: -6.209881348317425 | MSE: 0.0016610885023975164


TypeError: cannot unpack non-iterable PolynomialFeatures object

In [ ]:
history = []

for i, cell in enumerate(In):
    if cell.strip():
        history.append(f"\n# ===== CÉLULA {i} =====\n")
        history.append(cell)

texto = "\n".join(history)

with open("historico_codigo.txt", "w", encoding="utf-8") as f:
    f.write(texto)

print("Arquivo criado.")

Arquivo criado.


In [ ]:
# ============================================================
# IA-1 — S24-F.38 (CORRIGIDO)
# OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE
# ============================================================

import numpy as np

from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.kernel_ridge import KernelRidge

from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.38")
print(" OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

if "resultado_geometria_trajetoria_latente" not in globals():
    raise RuntimeError(
        "Execute S24-F.36 antes deste bloco."
    )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

dados = resultado_geometria_trajetoria_latente


if "Z" in dados:
    Z = np.asarray(dados["Z"])

elif "trajetoria" in dados:
    Z = np.asarray(dados["trajetoria"])

else:
    raise RuntimeError(
        "Não foi encontrado estado latente Z."
    )


# Estado futuro

X = Z[:-1]
y = Z[1:]


# divisão temporal

split = int(len(X)*0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]


resultados = {}


# ------------------------------------------------------------
# Ridge
# ------------------------------------------------------------

modelo = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])


modelo.fit(
    X_train,
    y_train
)

pred = modelo.predict(X_test)


resultados["ridge"] = {
    "R2": r2_score(y_test, pred),
    "MSE": mean_squared_error(y_test, pred)
}


print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")

print(
    "ridge | R²:",
    resultados["ridge"]["R2"],
    "| MSE:",
    resultados["ridge"]["MSE"]
)


# ------------------------------------------------------------
# Polynomial
# ------------------------------------------------------------

modelo_poly = Pipeline([
    ("poly", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])


modelo_poly.fit(
    X_train,
    y_train
)

pred = modelo_poly.predict(X_test)


resultados["polynomial"] = {
    "R2": r2_score(y_test, pred),
    "MSE": mean_squared_error(y_test, pred)
}


print(
    "polynomial | R²:",
    resultados["polynomial"]["R2"],
    "| MSE:",
    resultados["polynomial"]["MSE"]
)


# ------------------------------------------------------------
# RBF
# ------------------------------------------------------------

modelo_rbf = Pipeline([
    ("scale", StandardScaler()),
    ("rbf", KernelRidge(
        kernel="rbf",
        alpha=1.0,
        gamma="scale"
    ))
])


modelo_rbf.fit(
    X_train,
    y_train
)

pred = modelo_rbf.predict(X_test)


resultados["rbf"] = {
    "R2": r2_score(y_test, pred),
    "MSE": mean_squared_error(y_test, pred)
}


print(
    "rbf | R²:",
    resultados["rbf"]["R2"],
    "| MSE:",
    resultados["rbf"]["MSE"]
)


# ------------------------------------------------------------
# Melhor modelo
# ------------------------------------------------------------

melhor = max(
    resultados,
    key=lambda k: resultados[k]["R2"]
)


print("\n================================")
print(" MELHOR OPERADOR")
print("================================")

print("Modelo:", melhor)
print(
    "R²:",
    resultados[melhor]["R2"]
)


# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_nl_latente = {

    "modelos": resultados,

    "melhor_modelo": melhor,

    "R2": resultados[melhor]["R2"]

}


print("\nVariável criada:")
print("- resultado_operador_nl_latente")

print("\n=== FIM S24-F.38 ===")

 IA-1 — S24-F.38
 OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE


RuntimeError: Não foi encontrado estado latente Z.

In [ ]:
# ============================================================
# IA-1 — S24-F.38 (CORRIGIDO v2)
# OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.kernel_ridge import KernelRidge

from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.38")
print(" OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_campo_modal_orientado",
    "resultado_tempo_relacional"
]

for v in necessarias:
    if v not in globals():
        raise RuntimeError(
            f"Variável ausente: {v}"
        )


# ------------------------------------------------------------
# Reconstrução do espaço latente
# ------------------------------------------------------------

phi = np.asarray(
    resultado_campo_modal_orientado["phi_original"]
)

tau = np.asarray(
    resultado_tempo_relacional["tau_depth_velocity"]
)


X_base = np.column_stack([
    phi,
    tau
])


pca = PCA(
    n_components=2
)

Z_latente = pca.fit_transform(
    X_base
)


# ------------------------------------------------------------
# Evolução temporal
# ------------------------------------------------------------

X = Z_latente[:-1]

y = Z_latente[1:]


split = int(len(X)*0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]


resultados = {}


print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")


# ------------------------------------------------------------
# Ridge
# ------------------------------------------------------------

ridge = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])


ridge.fit(
    X_train,
    y_train
)

pred = ridge.predict(
    X_test
)


resultados["ridge"] = {
    "R2": r2_score(y_test,pred),
    "MSE": mean_squared_error(y_test,pred)
}


print(
    "ridge | R²:",
    resultados["ridge"]["R2"],
    "| MSE:",
    resultados["ridge"]["MSE"]
)


# ------------------------------------------------------------
# Polynomial corrigido
# ------------------------------------------------------------

poly = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        "scale",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(alpha=1.0)
    )
])


poly.fit(
    X_train,
    y_train
)


pred = poly.predict(
    X_test
)


resultados["polynomial"] = {
    "R2": r2_score(y_test,pred),
    "MSE": mean_squared_error(y_test,pred)
}


print(
    "polynomial | R²:",
    resultados["polynomial"]["R2"],
    "| MSE:",
    resultados["polynomial"]["MSE"]
)


# ------------------------------------------------------------
# RBF
# ------------------------------------------------------------

rbf = Pipeline([
    (
        "scale",
        StandardScaler()
    ),
    (
        "rbf",
        KernelRidge(
            kernel="rbf",
            alpha=1.0,
            gamma="scale"
        )
    )
])


rbf.fit(
    X_train,
    y_train
)


pred = rbf.predict(
    X_test
)


resultados["rbf"] = {
    "R2": r2_score(y_test,pred),
    "MSE": mean_squared_error(y_test,pred)
}


print(
    "rbf | R²:",
    resultados["rbf"]["R2"],
    "| MSE:",
    resultados["rbf"]["MSE"]
)


# ------------------------------------------------------------
# Melhor operador
# ------------------------------------------------------------

melhor = max(
    resultados,
    key=lambda x: resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR OPERADOR")
print("================================")

print("Modelo:", melhor)
print(
    "R²:",
    resultados[melhor]["R2"]
)


# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_nl_latente = {

    "Z_latente": Z_latente,

    "modelos": resultados,

    "melhor_modelo": melhor,

    "R2": resultados[melhor]["R2"]

}


print("\nVariável criada:")
print("- resultado_operador_nl_latente")

print("\n=== FIM S24-F.38 ===")

 IA-1 — S24-F.38
 OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 2900 and the array at index 1 has size 3000

In [ ]:
# ============================================================
# IA-1 — S24-F.38
# OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE
# VERSÃO CORRIGIDA
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge

from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.38")
print(" OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "resultado_memoria_operador_latente",
    "resultado_campo_modal_orientado",
    "resultado_tempo_relacional"
]


faltantes = []

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:
    print("ERRO")
    for x in faltantes:
        print("-", x)
    raise RuntimeError(
        "Execute os blocos anteriores."
    )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi = np.array(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau = np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


# alinhamento temporal

n = min(
    len(phi),
    len(tau)
)


phi = phi[:n]
tau = tau[:n]


# ------------------------------------------------------------
# Espaço latente
# ------------------------------------------------------------

X_base = np.column_stack(
    [
        phi,
        tau
    ]
)


X_base = StandardScaler().fit_transform(
    X_base
)


pca = PCA(
    n_components=2
)


Z = pca.fit_transform(
    X_base
)


# ------------------------------------------------------------
# Evolução temporal
# ------------------------------------------------------------

X = Z[:-1]

Y = Z[1:]


split = int(
    0.7 * len(Y)
)


X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]


# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

modelos = {

    "ridge":
    Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    ),


    "polynomial":
    Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),

            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    ),


    "rbf":
    Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
               (
    "rbf",
    KernelRidge(
        kernel="rbf",
        alpha=1.0,
        gamma=1.0
    )
)
                )
            )
        ]
    )
}


resultados = {}


print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")


for nome, modelo in modelos.items():

    modelo.fit(
        X_train,
        Y_train
    )


    pred = modelo.predict(
        X_test
    )


    r2 = r2_score(
        Y_test,
        pred
    )


    mse = mean_squared_error(
        Y_test,
        pred
    )


    resultados[nome] = {

        "R2": r2,

        "MSE": mse

    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )


# ------------------------------------------------------------
# Melhor operador
# ------------------------------------------------------------

melhor = max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR OPERADOR")
print("================================")


print(
    "Modelo:",
    melhor
)


print(
    "R²:",
    resultados[melhor]["R2"]
)


# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_nl_latente = {

    "Z": Z,

    "modelos":
    resultados,

    "melhor":
    melhor

}


print("\nVariável criada:")
print("- resultado_operador_nl_latente")


print("\n=== FIM S24-F.38 ===")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 216)

In [ ]:
# ============================================================
# IA-1 — S24-F.38
# OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE
# CORREÇÃO FINAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge

from sklearn.metrics import r2_score, mean_squared_error


print("="*60)
print(" IA-1 — S24-F.38")
print(" OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

required = [
    "resultado_campo_modal_orientado",
    "resultado_tempo_relacional"
]

for var in required:
    if var not in globals():
        raise RuntimeError(
            f"Variável ausente: {var}"
        )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

phi = np.asarray(
    resultado_campo_modal_orientado[
        "phi_original"
    ]
)


tau = np.asarray(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


# ------------------------------------------------------------
# Alinhamento temporal
# ------------------------------------------------------------

n = min(
    len(phi),
    len(tau)
)

phi = phi[:n]
tau = tau[:n]


print("\nTamanho alinhado:")
print("phi:", len(phi))
print("tau:", len(tau))


# ------------------------------------------------------------
# Espaço latente
# ------------------------------------------------------------

X_base = np.column_stack(
    (
        phi,
        tau
    )
)


X_base = StandardScaler().fit_transform(
    X_base
)


pca = PCA(
    n_components=2
)


Z = pca.fit_transform(
    X_base
)


# ------------------------------------------------------------
# Operador evolutivo
# ------------------------------------------------------------

X = Z[:-1]
Y = Z[1:]


split = int(
    0.7 * len(X)
)


X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]


print("\n================================")
print(" MODELOS NÃO LINEARES")
print("================================")


modelos = {

    "ridge":
    Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),
            (
                "model",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    ),


    "polynomial":
    Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),
            (
                "scale",
                StandardScaler()
            ),
            (
                "model",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    ),


    "rbf":
    Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),
            (
                "model",
                KernelRidge(
                    kernel="rbf",
                    alpha=1.0,
                    gamma=1.0
                )
            )
        ]
    )

}


resultados = {}


for nome, modelo in modelos.items():

    modelo.fit(
        X_train,
        Y_train
    )


    pred = modelo.predict(
        X_test
    )


    r2 = r2_score(
        Y_test,
        pred
    )


    mse = mean_squared_error(
        Y_test,
        pred
    )


    resultados[nome] = {
        "R2": r2,
        "MSE": mse
    }


    print(
        nome,
        "| R²:",
        r2,
        "| MSE:",
        mse
    )


# ------------------------------------------------------------
# Seleção
# ------------------------------------------------------------

melhor = max(
    resultados,
    key=lambda x:
    resultados[x]["R2"]
)


print("\n================================")
print(" MELHOR OPERADOR")
print("================================")


print(
    "Modelo:",
    melhor
)

print(
    "R²:",
    resultados[melhor]["R2"]
)


# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_operador_nl_latente = {

    "Z": Z,

    "resultados": resultados,

    "melhor_modelo": melhor

}


print("\nVariável criada:")
print("- resultado_operador_nl_latente")

print("\n=== FIM S24-F.38 ===")

 IA-1 — S24-F.38
 OPERADOR EVOLUTIVO NÃO LINEAR DO CAMPO LATENTE

Tamanho alinhado:
phi: 2900
tau: 2900

 MODELOS NÃO LINEARES
ridge | R²: 0.9990215067151547 | MSE: 8.231758719196532e-08
polynomial | R²: 0.64691147075833 | MSE: 1.320910881721642e-05
rbf | R²: -135.6401345654907 | MSE: 0.005100983774753888

 MELHOR OPERADOR
Modelo: ridge
R²: 0.9990215067151547

Variável criada:
- resultado_operador_nl_latente

=== FIM S24-F.38 ===


In [ ]:
print("eta_hist" in globals())
print("K_rel_star" in globals())
print("resultado_tempo_relacional" in globals())
print("resultado_espectro_dinamico" in globals())

True
True
True
True


In [ ]:
# ============================================================
# IA-1 — CHECKPOINT S24
# SALVAMENTO DE ESTADO
# ============================================================

import pickle

checkpoint = {
    "K_rel_star": K_rel_star,
    "eta_hist": eta_hist,
    "v_hist": v_hist,
    "a_hist": a_hist,
    "W_hist": W_hist,
    "kappa_A": kappa_A,
    "kappa_B": kappa_B,
    "kappa_C": kappa_C,
    "resultado_tempo_relacional": resultado_tempo_relacional,
    "resultado_espectro_dinamico": resultado_espectro_dinamico
}

with open("IA1_S24_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Checkpoint salvo:")
for k in checkpoint:
    print("-", k)

print("\nS24 preservado.")

Checkpoint salvo:
- K_rel_star
- eta_hist
- v_hist
- a_hist
- W_hist
- kappa_A
- kappa_B
- kappa_C
- resultado_tempo_relacional
- resultado_espectro_dinamico

S24 preservado.


In [ ]:
import pickle

with open("IA1_S24_checkpoint.pkl","rb") as f:
    checkpoint = pickle.load(f)

globals().update(checkpoint)

In [ ]:
# ============================================================
# IA-1 — S24-F.2
# COMPLEXIDADE RELACIONAL ACUMULADA
# ============================================================

import numpy as np
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


print("="*60)
print(" IA-1 — S24-F.2")
print(" COMPLEXIDADE RELACIONAL ACUMULADA")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "K_rel_star",
    "W_hist",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Estado S24 incompleto."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta = np.array(
    eta_hist
)


W = np.array(
    W_hist
)


tau = np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


N = min(
    len(eta),
    len(W),
    len(tau)
)


eta = eta[:N]
W = W[:N]
tau = tau[:N]



# ------------------------------------------------------------
# Complexidades locais
# ------------------------------------------------------------


# 1 — transformação cinemática

Omega_eta = np.linalg.norm(
    np.diff(eta, axis=0),
    axis=1
)



# 2 — transformação energética

Omega_W = np.abs(
    np.diff(W)
)



# 3 — transformação geométrica aproximada
# usa variação espectral do operador fixo

eig = np.linalg.eigvalsh(
    K_rel_star
)


Omega_K = np.full(
    N-1,
    np.linalg.norm(eig)
)


# ------------------------------------------------------------
# Acúmulos
# ------------------------------------------------------------

Theta_eta = np.cumsum(
    Omega_eta
)

Theta_W = np.cumsum(
    Omega_W
)

Theta_K = np.cumsum(
    Omega_K
)



# normalização

def normaliza(x):

    return x/(x[-1]+1e-12)


Theta = {

    "eta":
    normaliza(Theta_eta),

    "W":
    normaliza(Theta_W),

    "K":
    normaliza(Theta_K)

}



tau_n = normaliza(
    tau[:N-1]
)



# ------------------------------------------------------------
# Teste individual
# ------------------------------------------------------------

print("\n================================")
print(" COMPLEXIDADES")
print("================================")


resultado_complexidade={}


for nome, valor in Theta.items():

    r2 = r2_score(
        tau_n,
        valor
    )

    resultado_complexidade[nome]=r2

    print(
        nome,
        "| R²:",
        r2
    )



# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------


X=np.column_stack(
    [
        Theta["eta"],
        Theta["W"],
        Theta["K"]
    ]
)


corte=int(
    0.7*len(X)
)


modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)


modelo.fit(
    X[:corte],
    tau_n[:corte]
)


pred=modelo.predict(
    X[corte:]
)


r2_comb=r2_score(
    tau_n[corte:],
    pred
)


print("\n================================")
print(" MODELO HÍBRIDO")
print("================================")

print(
    "R² combinado:",
    r2_comb
)



resultado_complexidade_relacional={

    "individual":
    resultado_complexidade,

    "R2_combinado":
    r2_comb

}


print("\nVariável criada:")
print("- resultado_complexidade_relacional")


print("\n=== FIM S24-F.2 ===")

 IA-1 — S24-F.2
 COMPLEXIDADE RELACIONAL ACUMULADA

 COMPLEXIDADES
eta | R²: -7.659094213892221
W | R²: -9.13744728773376
K | R²: -1.265491306305055

 MODELO HÍBRIDO
R² combinado: -69490614.41375193

Variável criada:
- resultado_complexidade_relacional

=== FIM S24-F.2 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.3
# CAPACIDADE CAUSAL RELACIONAL
# ============================================================

import numpy as np
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


print("="*60)
print(" IA-1 — S24-F.3")
print(" CAPACIDADE CAUSAL RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias = [
    "eta_hist",
    "v_hist",
    "a_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:
    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Estado S24 incompleto."
    )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(eta_hist)
v=np.array(v_hist)
a=np.array(a_hist)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


N=min(
    len(eta),
    len(v),
    len(a),
    len(tau)
)


eta=eta[:N]
v=v[:N]
a=a[:N]
tau=tau[:N]


# ------------------------------------------------------------
# Medidas candidatas de capacidade causal
# ------------------------------------------------------------


# 1) Velocidade efetiva de transmissão

C1=np.linalg.norm(v,axis=1)


# 2) Aceleração / alteração do canal

C2=np.linalg.norm(a,axis=1)


# 3) Relação deslocamento / propagação

C3=np.linalg.norm(eta,axis=1)/(C1+1e-12)


# 4) Saturação causal combinada

C4=C1/(1+C2)


# normalização

def norm(x):
    return (x-x.min())/(x.max()-x.min()+1e-12)


capacidades={

    "velocidade_causal":norm(C1),
    "aceleracao_causal":norm(C2),
    "profundidade_saturacao":norm(C3),
    "canal_efetivo":norm(C4)

}


tau_n=norm(tau)


# ------------------------------------------------------------
# Testes individuais
# ------------------------------------------------------------

print("\n================================")
print(" TESTES DE CAPACIDADE CAUSAL")
print("================================")


resultado_causal={}


for nome,x in capacidades.items():

    r2=r2_score(
        tau_n,
        x
    )

    resultado_causal[nome]=r2

    print(
        nome,
        "| R²:",
        r2
    )


# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------

X=np.column_stack(
    [
        capacidades["velocidade_causal"],
        capacidades["aceleracao_causal"],
        capacidades["profundidade_saturacao"],
        capacidades["canal_efetivo"]
    ]
)


corte=int(
    0.7*N
)


modelo=Pipeline(
    [
        (
            "escala",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)


modelo.fit(
    X[:corte],
    tau_n[:corte]
)


pred=modelo.predict(
    X[corte:]
)


r2_comb=r2_score(
    tau_n[corte:],
    pred
)


print("\n================================")
print(" MODELO CAUSAL COMBINADO")
print("================================")

print(
    "R² teste:",
    r2_comb
)


resultado_capacidade_causal={

    "individual":resultado_causal,
    "R2_combinado":r2_comb

}


print("\nVariável criada:")
print("- resultado_capacidade_causal")


print("\n=== FIM S24-F.3 ===")

 IA-1 — S24-F.3
 CAPACIDADE CAUSAL RELACIONAL

 TESTES DE CAPACIDADE CAUSAL
velocidade_causal | R²: -7.646589285801875
aceleracao_causal | R²: -7.666822620509182
profundidade_saturacao | R²: -10.555131188025733
canal_efetivo | R²: -6.5177619673312455

 MODELO CAUSAL COMBINADO
R² teste: -40402.09771525031

Variável criada:
- resultado_capacidade_causal

=== FIM S24-F.3 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.4
# PROFUNDIDADE RELACIONAL E HORIZONTE DE INFORMAÇÃO
# ============================================================

import numpy as np
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


print("="*60)
print(" IA-1 — S24-F.4")
print(" PROFUNDIDADE RELACIONAL E HORIZONTE DE INFORMAÇÃO")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:

    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Estado S24 incompleto."
    )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(eta_hist)

K=np.array(K_rel_star)

tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


N=min(
    len(eta),
    len(tau)
)


eta=eta[:N]
tau=tau[:N]


# ------------------------------------------------------------
# Construção dos invariantes relacionais
# ------------------------------------------------------------


# 1 — profundidade relacional efetiva
# medida pelo espalhamento no espaço de estados

profundidade=np.linalg.norm(
    eta,
    axis=1
)


# 2 — conectividade espectral do operador

autovalores=np.abs(
    np.linalg.eigvalsh(K)
)

energia_espectral=np.sum(
    autovalores**2
)


conectividade=np.full(
    N,
    energia_espectral
)


# 3 — concentração relacional

prob=autovalores/(np.sum(autovalores)+1e-12)

entropia=-np.sum(
    prob*np.log(prob+1e-12)
)

entropia_rel=np.full(
    N,
    entropia
)


# 4 — horizonte informacional combinado

horizonte=(
    profundidade*
    conectividade
)/(entropia_rel+1e-12)



# ------------------------------------------------------------
# Normalização
# ------------------------------------------------------------

def norm(x):

    return (
        x-x.min()
    )/(
        x.max()-x.min()+1e-12
    )


campos={

    "profundidade":
    norm(profundidade),

    "conectividade":
    norm(conectividade),

    "entropia":
    norm(entropia_rel),

    "horizonte":
    norm(horizonte)

}


tau_n=norm(tau)



# ------------------------------------------------------------
# Testes individuais
# ------------------------------------------------------------

print("\n================================")
print(" CAMPOS RELACIONAIS")
print("================================")


resultado_profundidade={}


for nome,x in campos.items():

    r2=r2_score(
        tau_n,
        x
    )

    resultado_profundidade[nome]=r2

    print(
        nome,
        "| R²:",
        r2
    )



# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------

X=np.column_stack(
    [
        campos["profundidade"],
        campos["conectividade"],
        campos["entropia"],
        campos["horizonte"]
    ]
)


corte=int(
    0.7*N
)


modelo=Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)


modelo.fit(
    X[:corte],
    tau_n[:corte]
)


pred=modelo.predict(
    X[corte:]
)


r2_comb=r2_score(
    tau_n[corte:],
    pred
)



print("\n================================")
print(" MODELO HORIZONTE INFORMACIONAL")
print("================================")

print(
    "R² teste:",
    r2_comb
)



resultado_horizonte_informacional={

    "individual":
    resultado_profundidade,

    "R2_combinado":
    r2_comb

}


print("\nVariável criada:")
print("- resultado_horizonte_informacional")


print("\n=== FIM S24-F.4 ===")

 IA-1 — S24-F.4
 PROFUNDIDADE RELACIONAL E HORIZONTE DE INFORMAÇÃO

 CAMPOS RELACIONAIS
profundidade | R²: -7.664779983902042
conectividade | R²: -10.586512502904165
entropia | R²: -10.586512502904165
horizonte | R²: -7.664779983902033

 MODELO HORIZONTE INFORMACIONAL
R² teste: -342026.1738236199

Variável criada:
- resultado_horizonte_informacional

=== FIM S24-F.4 ===


In [ ]:
# ============================================================
# IA-1 — S24-G.1
# VELOCIDADE DO FECHAMENTO RELACIONAL
# ============================================================

import numpy as np
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


print("="*60)
print(" IA-1 — S24-G.1")
print(" VELOCIDADE DO FECHAMENTO RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

necessarias=[
    "eta_hist",
    "v_hist",
    "K_rel_star",
    "resultado_tempo_relacional"
]


faltantes=[]

for x in necessarias:
    if x not in globals():
        faltantes.append(x)


if faltantes:
    print("\nVARIÁVEIS AUSENTES:")
    for x in faltantes:
        print("-",x)

    raise RuntimeError(
        "Estado S24 incompleto."
    )


# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

eta=np.array(eta_hist)
v=np.array(v_hist)

K=np.array(K_rel_star)


tau=np.array(
    resultado_tempo_relacional[
        "tau_depth_velocity"
    ]
)


N=min(
    len(eta),
    len(v),
    len(tau)
)


eta=eta[:N]
v=v[:N]
tau=tau[:N]


# ------------------------------------------------------------
# Reconstrução de fechamento Phi(k)
# ------------------------------------------------------------

# Operador relacional local:
# Phi_k = projeção do estado no operador K

Phi=[]

for i in range(N):

    estado=np.outer(
        eta[i],
        eta[i]
    )

    fechamento=K @ estado

    Phi.append(
        fechamento
    )


Phi=np.array(Phi)



# ------------------------------------------------------------
# Velocidade de fechamento
# ------------------------------------------------------------

delta_Phi=np.diff(
    Phi,
    axis=0
)


vel_fechamento=np.linalg.norm(
    delta_Phi.reshape(
        len(delta_Phi),
        -1
    ),
    axis=1
)


# alinhar

tau_f=tau[1:]



# ------------------------------------------------------------
# Medidas derivadas
# ------------------------------------------------------------

energia_fechamento=(
    vel_fechamento**2
)


acumulado=np.cumsum(
    vel_fechamento
)



def norm(x):

    return (
        x-x.min()
    )/(
        x.max()-x.min()+1e-12
    )



campos={

    "velocidade_fechamento":
    norm(vel_fechamento),

    "energia_fechamento":
    norm(energia_fechamento),

    "acumulacao_relacional":
    norm(acumulado)

}


tau_n=norm(tau_f)



# ------------------------------------------------------------
# Testes individuais
# ------------------------------------------------------------

print("\n================================")
print(" TESTES DE FECHAMENTO")
print("================================")


resultado_fechamento={}


for nome,x in campos.items():

    r2=r2_score(
        tau_n,
        x
    )

    resultado_fechamento[nome]=r2

    print(
        nome,
        "| R²:",
        r2
    )


# ------------------------------------------------------------
# Modelo combinado
# ------------------------------------------------------------

X=np.column_stack(
    [
        campos["velocidade_fechamento"],
        campos["energia_fechamento"],
        campos["acumulacao_relacional"]
    ]
)


corte=int(
    0.7*len(X)
)


modelo=Pipeline(
    [
        (
            "escala",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1e-3)
        )
    ]
)


modelo.fit(
    X[:corte],
    tau_n[:corte]
)


pred=modelo.predict(
    X[corte:]
)


r2=r2_score(
    tau_n[corte:],
    pred
)


print("\n================================")
print(" MODELO FECHAMENTO RELACIONAL")
print("================================")

print(
    "R² teste:",
    r2
)



resultado_fechamento_relacional={

    "individual":
    resultado_fechamento,

    "R2_combinado":
    r2

}


print("\nVariável criada:")
print("- resultado_fechamento_relacional")


print("\n=== FIM S24-G.1 ===")

 IA-1 — S24-G.1
 VELOCIDADE DO FECHAMENTO RELACIONAL

 TESTES DE FECHAMENTO
velocidade_fechamento | R²: -9.105815126328531
energia_fechamento | R²: -9.837446090240377
acumulacao_relacional | R²: -9.09980491988293

 MODELO FECHAMENTO RELACIONAL
R² teste: -2059504346729.1409

Variável criada:
- resultado_fechamento_relacional

=== FIM S24-G.1 ===


In [ ]:
# ============================================================
# IA-1 — S24-F.54
# CARACTERIZAÇÃO DO ATRATOR RELACIONAL
# ============================================================

import numpy as np

from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge


print("="*60)
print(" IA-1 — S24-F.54")
print(" CARACTERIZAÇÃO DO ATRATOR RELACIONAL")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]

necessarias=[
    "resultado_detector_transicao_temporal",
    "resultado_eventos_transicao",
    "eta_hist",
    "v_hist",
    "W_hist"
]


for x in necessarias:
    if x not in globals():
        faltando.append(x)


if len(faltando)>0:

    print("Variáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Reconstrução S24 necessária."
    )


# ------------------------------------------------------------
# Reconstrução do estado pós-transição
# ------------------------------------------------------------


eventos = resultado_eventos_transicao["eventos"]

fim = int(
    eventos[0]["fim"]
)


eta=np.array(
    eta_hist
)


v=np.array(
    v_hist
)


W=np.array(
    W_hist
)


eta=eta[fim:]
v=v[fim:]
W=W[fim:]



n=min(
    len(eta),
    len(v),
    len(W)
)


eta=eta[:n]
v=v[:n]
W=W[:n]


# ------------------------------------------------------------
# Estado relacional composto
# ------------------------------------------------------------


estado=np.column_stack(
    [
        np.linalg.norm(eta,axis=1),
        np.linalg.norm(v,axis=1),
        W
    ]
)



estado=(

    estado-np.mean(estado,axis=0)

)/(

    np.std(estado,axis=0)+1e-12

)



# ------------------------------------------------------------
# Dimensão efetiva do atrator
# ------------------------------------------------------------


pca=PCA()

Z=pca.fit_transform(
    estado
)


variancia=np.cumsum(
    pca.explained_variance_ratio_
)



dim=np.searchsorted(
    variancia,
    0.95
)+1



print("\n================================")
print(" DIMENSÃO DO ATRATOR")
print("================================")


print(
    "Dimensão efetiva:",
    dim
)


print(
    "Variância acumulada:",
    variancia[:5]
)



# ------------------------------------------------------------
# Memória interna
# ------------------------------------------------------------


print("\n================================")
print(" MEMÓRIA DO ATRATOR")
print("================================")


memorias=[
    1,
    5,
    10,
    20,
    50
]


resultado_memoria={}


for m in memorias:

    X=Z[:-m]

    Y=Z[m:]


    modelo=Ridge(
        alpha=1.0
    )


    modelo.fit(
        X,
        Y
    )


    pred=modelo.predict(
        X
    )


    r2=r2_score(
        Y,
        pred
    )


    resultado_memoria[m]=r2


    print(
        "Memória",
        m,
        "| R²:",
        r2
    )



# ------------------------------------------------------------
# Teste de órbita
# ------------------------------------------------------------


dif=np.linalg.norm(
    Z[1:]-Z[:-1],
    axis=1
)


mobilidade_media=np.mean(
    dif
)


mobilidade_final=np.mean(
    dif[-100:]
)



print("\n================================")
print(" MOBILIDADE")
print("================================")


print(
    "Mobilidade média:",
    mobilidade_media
)


print(
    "Mobilidade final:",
    mobilidade_final
)



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------


resultado_atrator_caracterizado={

    "dimensao_efetiva":
    dim,

    "variancia_pca":
    variancia,

    "memoria":
    resultado_memoria,

    "mobilidade_media":
    mobilidade_media,

    "mobilidade_final":
    mobilidade_final,

    "estado_latente":
    Z

}


print("\nVariável criada:")
print("- resultado_atrator_caracterizado")


print("\n=== FIM S24-F.54 ===")

 IA-1 — S24-F.54
 CARACTERIZAÇÃO DO ATRATOR RELACIONAL
Variáveis ausentes:
- resultado_detector_transicao_temporal
- resultado_eventos_transicao
- eta_hist
- v_hist
- W_hist


RuntimeError: Reconstrução S24 necessária.

In [ ]:
print("K:", "K_rel_star" in globals())
print("eta:", "eta_hist" in globals())
print("v:", "v_hist" in globals())
print("W:", "W_hist" in globals())
print("tempo:", "resultado_tempo_relacional" in globals())
print("F42:", "resultado_detector_transicao_temporal" in globals())
print("F47:", "resultado_eventos_transicao" in globals())

K: True
eta: True
v: True
W: True
tempo: False
F42: False
F47: False


In [ ]:
# ============================================================
# IA-1 — S24-C.1-R
# RECONSTRUÇÃO DA AUDITORIA DE INVARIANTES DO FLUXO RELACIONAL
# ============================================================

import numpy as np


print("="*60)
print(" IA-1 — S24-C.1-R")
print(" RECONSTRUÇÃO DOS INVARIANTES RELACIONAIS")
print("="*60)


# ------------------------------------------------------------
# Verificação
# ------------------------------------------------------------

faltando=[]

necessarias=[
    "K_rel_star",
    "eta_hist",
    "v_hist",
    "W_hist"
]


for x in necessarias:

    if x not in globals():

        faltando.append(x)


if len(faltando)>0:

    print("\nVariáveis ausentes:")

    for x in faltando:
        print("-",x)

    raise RuntimeError(
        "Execute S24-A.2.0 antes."
    )



# ------------------------------------------------------------
# Dados
# ------------------------------------------------------------

K=np.asarray(
    K_rel_star
)


eta=np.asarray(
    eta_hist
)


v=np.asarray(
    v_hist
)


W=np.asarray(
    W_hist
)



n=min(
    len(eta),
    len(v),
    len(W)
)


eta=eta[:n]
v=v[:n]
W=W[:n]



# ------------------------------------------------------------
# Invariante energético relacional
# ------------------------------------------------------------

energia=np.einsum(
    "ij,jk,ik->i",
    eta,
    K,
    eta
)



# ------------------------------------------------------------
# Momento relacional
# ------------------------------------------------------------

momento=np.sum(
    eta*v,
    axis=1
)



# ------------------------------------------------------------
# Escala relacional
# ------------------------------------------------------------

norma_eta=np.linalg.norm(
    eta,
    axis=1
)


razao_escala=(

    norma_eta[0]

    /

    (norma_eta+1e-12)

)



# ------------------------------------------------------------
# Curvatura efetiva
# ------------------------------------------------------------

dv=np.diff(
    v,
    axis=0
)


aceleracao=np.linalg.norm(
    dv,
    axis=1
)


aceleracao=np.pad(
    aceleracao,
    (1,0),
    mode="edge"
)



curvatura=(

    aceleracao

    /

    (norma_eta+1e-12)

)



# ------------------------------------------------------------
# Correlações
# ------------------------------------------------------------

def corr(a,b):

    if np.std(a)==0 or np.std(b)==0:

        return np.nan

    return np.corrcoef(a,b)[0,1]



print("\n================================")
print(" INVARIANTES RECONSTRUÍDOS")
print("================================")


for nome,x in {

    "Energia relacional":energia,

    "Momento relacional":momento,

    "Razão de escala":razao_escala,

    "Curvatura efetiva":curvatura

}.items():

    print("\n",nome)

    print(
        "Inicial:",
        x[0]
    )

    print(
        "Final:",
        x[-1]
    )

    print(
        "Média:",
        np.mean(x)
    )



print("\n================================")
print(" CORRELAÇÕES COM W")
print("================================")


for nome,x in {

    "Energia":energia,

    "Momento":momento,

    "Escala":razao_escala,

    "Curvatura":curvatura

}.items():

    print(
        nome,
        ":",
        corr(x,W)
    )



# ------------------------------------------------------------
# Saída
# ------------------------------------------------------------

resultado_invariantes={

    "energia":energia,

    "momento":momento,

    "escala":razao_escala,

    "curvatura":curvatura,

    "W":W

}



print("\nVariável criada:")
print("- resultado_invariantes")


print("\n=== FIM S24-C.1-R ===")

 IA-1 — S24-C.1-R
 RECONSTRUÇÃO DOS INVARIANTES RELACIONAIS

 INVARIANTES RECONSTRUÍDOS

 Energia relacional
Inicial: -0.2593407552878069
Final: -220299.45369320302
Média: -14496.90144001829

 Momento relacional
Inicial: 0.0036192717155654323
Final: 86650.06473684318
Média: 5726.435293179761

 Razão de escala
Inicial: 0.9999999999982748
Final: 0.0031329628062978783
Média: 0.3188021106903675

 Curvatura efetiva
Inicial: 0.002783253838773909
Final: 0.0064694352384549805
Média: 0.005353113009632315

 CORRELAÇÕES COM W
Energia : 0.9999982452133083
Momento : -0.9999921746239983
Escala : 0.3382631928188874
Curvatura : -0.3502047894854257

Variável criada:
- resultado_invariantes

=== FIM S24-C.1-R ===
